## portion 2 

In [1]:
# ============================================================
# CELL 0B — OFFLINE WHEEL INSTALL
# Run only if packages are missing in the offline GPU notebook.
# ============================================================

from pathlib import Path
import os

candidate_dirs = [
    Path("/kaggle/input/notebooks/nabidnur/wheel-for-slm"),
    Path("/kaggle/input/bleach_wheelhouse/bleach_wheelhouse"),
    Path("/kaggle/input/bleach-wheelhouse"),
]

wheel_dir = None
for p in candidate_dirs:
    if p.exists():
        wheel_dir = p
        break

if wheel_dir is None:
    print("[INFO] No wheelhouse dataset found. Skipping offline install.")
else:
    print("[INFO] Installing from:", wheel_dir)
    os.system(
        f"python -m pip install --no-index --find-links {wheel_dir} "
        "tokenizers safetensors einops pyyaml tqdm pandas pyarrow matplotlib scikit-learn"
    )

[INFO] Installing from: /kaggle/input/notebooks/nabidnur/wheel-for-slm
Looking in links: /kaggle/input/notebooks/nabidnur/wheel-for-slm


In [2]:
# ============================================================
# CELL 1 — IMPORTS, REPRODUCIBILITY, DEVICE
# ============================================================

from __future__ import annotations

import os
import gc
import re
import json
import math
import time
import random
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

SEED = 42


def set_seed(seed: int = 42, deterministic: bool = True) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    if deterministic:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")

print("=" * 100)
print("BLEACH SLM/LLM PORTION 2 — DENSE ARCHITECTURE VALIDATION")
print("=" * 100)
print(f"Seed   : {SEED}")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
print(f"Torch  : {torch.__version__}")
print("=" * 100)

BLEACH SLM/LLM PORTION 2 — DENSE ARCHITECTURE VALIDATION
Seed   : 42
Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 94.97 GB
Torch  : 2.10.0+cu128


In [3]:
# ============================================================
# CELL 2 — CONFIG AND PATH RESOLVER
# ============================================================

@dataclass(frozen=True)
class P2Config:
    output_dir: str = "/kaggle/working/artifacts/bleach_p2_dense_arch"

    # Expected model/data values from Portion 1.
    vocab_size: int = 32000
    block_size: int = 512
    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3

    # Dense model config.
    n_layers: int = 12
    d_model: int = 384
    n_heads: int = 6
    n_kv_heads: int = 2
    ffn_dim: int = 1024
    rope_base: float = 10000.0
    dropout: float = 0.12
    qk_norm: bool = True
    tie_embeddings: bool = True

    # Auxiliary future-token objective used in architecture validation.
    # In Portion 3: start with 0.0 during warmup, then 0.05.
    aux_future_loss_weight: float = 0.05

    # Architecture test.
    test_batch_size: int = 2
    expected_min_params: int = 30_500_000
    expected_max_params: int = 32_500_000

    # Dtypes.
    use_bf16_if_available: bool = True


CFG = P2Config()

OUT_DIR = Path(CFG.output_dir)
REPORT_DIR = OUT_DIR / "reports"
SOURCE_DIR = OUT_DIR / "source"
CONFIG_DIR = OUT_DIR / "config"

for p in [OUT_DIR, REPORT_DIR, SOURCE_DIR, CONFIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def find_first_existing(paths: List[Path]) -> Optional[Path]:
    for p in paths:
        if p.exists():
            return p
    return None


def resolve_slm_p1_root() -> Path:
    """
    Resolve the SLM Portion-1 output root from either:
      - current working directory,
      - Kaggle input dataset,
      - copied notebook output dataset.
    """
    candidates = [
        Path("/kaggle/working/artifacts/slm_p1_data"),
        Path("artifacts/slm_p1_data"),
    ]

    # Search under /kaggle/input.
    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = sorted(input_root.rglob("artifacts/slm_p1_data/packed/packing_manifest.json"))
        for m in matches:
            candidates.append(m.parents[2])  # .../artifacts/slm_p1_data

        matches = sorted(input_root.rglob("slm_p1_data/packed/packing_manifest.json"))
        for m in matches:
            candidates.append(m.parents[1])  # .../slm_p1_data

    for c in candidates:
        if (c / "packed" / "packing_manifest.json").exists():
            return c

    raise FileNotFoundError(
        "Could not resolve SLM Portion-1 root. Expected packed/packing_manifest.json."
    )


def resolve_tokenizer_frozen_root() -> Optional[Path]:
    """
    Resolve tokenizer frozen folder if available.
    Portion 2 can run without loading tokenizer.json, but we use its config if found.
    """
    candidates = [
        Path("/kaggle/working/artifacts/tokenizer_p5_freeze/final_tokenizer_frozen"),
        Path("artifacts/tokenizer_p5_freeze/final_tokenizer_frozen"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        matches = sorted(input_root.rglob("tokenizer_p5_freeze/final_tokenizer_frozen/tokenizer.json"))
        for m in matches:
            candidates.append(m.parent)

    for c in candidates:
        if (c / "tokenizer.json").exists():
            return c

    return None


SLM_P1_ROOT = resolve_slm_p1_root()
TOKENIZER_FROZEN_ROOT = resolve_tokenizer_frozen_root()

print("Resolved SLM P1 root       :", SLM_P1_ROOT)
print("Resolved tokenizer root    :", TOKENIZER_FROZEN_ROOT)

write_config = {
    "portion": "bleach_p2_dense_architecture",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "config": asdict(CFG),
    "slm_p1_root": str(SLM_P1_ROOT),
    "tokenizer_frozen_root": str(TOKENIZER_FROZEN_ROOT) if TOKENIZER_FROZEN_ROOT else None,
}

with open(CONFIG_DIR / "p2_resolved_paths.json", "w", encoding="utf-8") as f:
    json.dump(write_config, f, ensure_ascii=False, indent=2)

print("Saved:", CONFIG_DIR / "p2_resolved_paths.json")

Resolved SLM P1 root       : /kaggle/input/notebooks/nabidnur/bleach-slm/artifacts/slm_p1_data
Resolved tokenizer root    : /kaggle/input/datasets/nabidnur/tokenizer-best/artifacts/tokenizer_p5_freeze/final_tokenizer_frozen
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/config/p2_resolved_paths.json


In [4]:
# ============================================================
# CELL 3 — LOAD SLM P1 MANIFESTS AND ONE PACKED SHARD
# ============================================================

def read_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


packing_manifest_path = SLM_P1_ROOT / "packed" / "packing_manifest.json"
packing_summary_path = SLM_P1_ROOT / "packed" / "packing_summary.csv"
tokenizer_integrity_path = SLM_P1_ROOT / "config" / "tokenizer_integrity_report.json"
model_io_config_path = SLM_P1_ROOT / "config" / "model_io_config_used.json"
split_manifest_path = SLM_P1_ROOT / "splits" / "split_manifest.json"
tokenized_summary_path = SLM_P1_ROOT / "tokenized" / "tokenized_split_summary.json"

required_paths = [
    packing_manifest_path,
    tokenizer_integrity_path,
    model_io_config_path,
    split_manifest_path,
    tokenized_summary_path,
]

missing = [str(p) for p in required_paths if not p.exists()]
assert not missing, "Missing required Portion-1 files:\n" + "\n".join(missing)

packing_manifest = read_json(packing_manifest_path)
tokenizer_integrity = read_json(tokenizer_integrity_path)
model_io_config = read_json(model_io_config_path)
split_manifest = read_json(split_manifest_path)
tokenized_summary = read_json(tokenized_summary_path)

packing_summary = pd.read_csv(packing_summary_path) if packing_summary_path.exists() else pd.DataFrame()

print("=" * 100)
print("P1 MANIFEST SUMMARY")
print("=" * 100)
print("Vocab size from P1:", tokenizer_integrity.get("vocab_size"))
print("Special IDs       :", tokenizer_integrity.get("special_token_ids"))
print("Train rows        :", split_manifest.get("train_rows"))
print("Val rows          :", split_manifest.get("val_rows"))
print("External eval rows:", split_manifest.get("external_eval_rows"))
print("Packing manifest keys:", list(packing_manifest.keys()))

if len(packing_summary):
    print("\nPacking summary:")
    print(packing_summary.to_string(index=False))

# Validate tokenizer/model IO assumptions.
assert int(tokenizer_integrity["vocab_size"]) == CFG.vocab_size
assert int(model_io_config["vocab_size"]) == CFG.vocab_size

special_ids = tokenizer_integrity["special_token_ids"]
assert special_ids["pad_token_id"] == CFG.pad_token_id
assert special_ids["unk_token_id"] == CFG.unk_token_id
assert special_ids["bos_token_id"] == CFG.bos_token_id
assert special_ids["eos_token_id"] == CFG.eos_token_id

train_shard_dir = SLM_P1_ROOT / "packed" / "train_block512_shards"
val_shard_dir = SLM_P1_ROOT / "packed" / "val_block512_shards"
eval_shard_dir = SLM_P1_ROOT / "packed" / "eval_block512_shards"

train_shards = sorted(train_shard_dir.glob("*.npz"))
val_shards = sorted(val_shard_dir.glob("*.npz"))
eval_shards = sorted(eval_shard_dir.glob("*.npz"))

assert len(train_shards) > 0, f"No train shards found in {train_shard_dir}"
assert len(val_shards) > 0, f"No val shards found in {val_shard_dir}"
assert len(eval_shards) > 0, f"No eval shards found in {eval_shard_dir}"

print("\nShard counts:")
print("Train:", len(train_shards), train_shards[:2])
print("Val  :", len(val_shards), val_shards[:2])
print("Eval :", len(eval_shards), eval_shards[:2])


def load_npz_batch(shard_path: Path, batch_size: int) -> Dict[str, torch.Tensor]:
    arr = np.load(shard_path)

    input_ids = arr["input_ids"][:batch_size]              # [B, T]
    attention_mask = arr["attention_mask"][:batch_size]    # [B, T]
    labels = arr["labels"][:batch_size]                    # [B, T]
    dialect_id = arr["dialect_id"][:batch_size]            # [B]

    batch = {
        "input_ids": torch.as_tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.as_tensor(attention_mask, dtype=torch.long),
        "labels": torch.as_tensor(labels, dtype=torch.long),
        "dialect_id": torch.as_tensor(dialect_id, dtype=torch.long),
    }
    return batch


batch_cpu = load_npz_batch(train_shards[0], CFG.test_batch_size)

print("\nLoaded architecture-test batch:")
for k, v in batch_cpu.items():
    print(f"{k:16s}: shape={tuple(v.shape)} dtype={v.dtype}")

assert batch_cpu["input_ids"].shape == (CFG.test_batch_size, CFG.block_size)
assert batch_cpu["labels"].shape == (CFG.test_batch_size, CFG.block_size)
assert int(batch_cpu["input_ids"].max()) < CFG.vocab_size
assert int(batch_cpu["input_ids"].min()) >= 0
assert int((batch_cpu["input_ids"] == CFG.unk_token_id).sum()) == 0

print("\n[OK] P1 artifacts and packed batch validated.")

P1 MANIFEST SUMMARY
Vocab size from P1: 32000
Special IDs       : {'pad_token_id': 0, 'unk_token_id': 1, 'bos_token_id': 2, 'eos_token_id': 3}
Train rows        : 113594
Val rows          : 6500
External eval rows: 2875
Packing manifest keys: ['created_at', 'block_size', 'homogeneous_dialect_packing', 'allow_cross_dialect_packing', 'train', 'val', 'eval', 'packing_summary_csv', 'packing_block_metadata_csv']

Packing summary:
split dialect  blocks  total_nonpad_tokens  mean_nonpad_tokens  mean_samples_per_block  mean_synthetic_ratio  mean_quality
 eval     BAR       8                 3858          482.250000               46.875000              0.000000      1.000000
 eval     CHI      12                 6039          503.250000               52.083333              0.000000      1.000000
 eval     MYM      12                 6037          503.083333               52.083333              0.000000      1.000000
 eval     NOA      12                 6038          503.166667               52

In [5]:
# ============================================================
# CELL 4 — MODEL CONFIG AND PARAMETER ESTIMATOR
# ============================================================

@dataclass
class BLEACHDenseConfig:
    vocab_size: int = 32000
    block_size: int = 512

    n_layers: int = 12
    d_model: int = 384
    n_heads: int = 6
    n_kv_heads: int = 2
    ffn_dim: int = 1024

    rope_base: float = 10000.0
    dropout: float = 0.12
    qk_norm: bool = True
    tie_embeddings: bool = True

    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3

    aux_future_loss_weight: float = 0.05


MODEL_CFG = BLEACHDenseConfig(
    vocab_size=CFG.vocab_size,
    block_size=CFG.block_size,
    n_layers=CFG.n_layers,
    d_model=CFG.d_model,
    n_heads=CFG.n_heads,
    n_kv_heads=CFG.n_kv_heads,
    ffn_dim=CFG.ffn_dim,
    rope_base=CFG.rope_base,
    dropout=CFG.dropout,
    qk_norm=CFG.qk_norm,
    tie_embeddings=CFG.tie_embeddings,
    pad_token_id=CFG.pad_token_id,
    unk_token_id=CFG.unk_token_id,
    bos_token_id=CFG.bos_token_id,
    eos_token_id=CFG.eos_token_id,
    aux_future_loss_weight=CFG.aux_future_loss_weight,
)


def estimate_dense_params(cfg: BLEACHDenseConfig) -> Dict[str, int]:
    d = cfg.d_model
    h = cfg.n_heads
    kvh = cfg.n_kv_heads
    hd = d // h
    f = cfg.ffn_dim

    emb = cfg.vocab_size * d

    # Bias-free projections.
    q = d * (h * hd)
    k = d * (kvh * hd)
    v = d * (kvh * hd)
    o = d * d
    attn = q + k + v + o

    # QK norm params per layer, if enabled.
    qk_norm = (2 * hd) if cfg.qk_norm else 0

    # SwiGLU: gate, up, down.
    ffn = d * f + d * f + f * d

    # Two RMSNorms per block.
    norms = 2 * d

    block = attn + qk_norm + ffn + norms
    all_blocks = cfg.n_layers * block

    final_norm = d

    # LM head tied to embedding, so no extra output projection.
    lm_head = 0 if cfg.tie_embeddings else cfg.vocab_size * d

    total = emb + all_blocks + final_norm + lm_head

    return {
        "embedding_params": emb,
        "attention_params_per_layer": attn,
        "qk_norm_params_per_layer": qk_norm,
        "ffn_params_per_layer": ffn,
        "norm_params_per_layer": norms,
        "block_params_per_layer": block,
        "all_block_params": all_blocks,
        "final_norm_params": final_norm,
        "lm_head_params": lm_head,
        "estimated_total_params": total,
    }


estimate = estimate_dense_params(MODEL_CFG)

print("=" * 100)
print("BLEACH-DENSE-32M CONFIG")
print("=" * 100)
print(json.dumps(asdict(MODEL_CFG), indent=2))
print("\nParameter estimate:")
print(json.dumps(estimate, indent=2))
print(f"\nEstimated total params: {estimate['estimated_total_params']:,}")

assert CFG.expected_min_params <= estimate["estimated_total_params"] <= CFG.expected_max_params

with open(CONFIG_DIR / "dense_32m_config.json", "w", encoding="utf-8") as f:
    json.dump(asdict(MODEL_CFG), f, ensure_ascii=False, indent=2)

with open(REPORT_DIR / "parameter_estimate.json", "w", encoding="utf-8") as f:
    json.dump(estimate, f, ensure_ascii=False, indent=2)

print("\n[OK] Dense config and parameter estimate validated.")

BLEACH-DENSE-32M CONFIG
{
  "vocab_size": 32000,
  "block_size": 512,
  "n_layers": 12,
  "d_model": 384,
  "n_heads": 6,
  "n_kv_heads": 2,
  "ffn_dim": 1024,
  "rope_base": 10000.0,
  "dropout": 0.12,
  "qk_norm": true,
  "tie_embeddings": true,
  "pad_token_id": 0,
  "unk_token_id": 1,
  "bos_token_id": 2,
  "eos_token_id": 3,
  "aux_future_loss_weight": 0.05
}

Parameter estimate:
{
  "embedding_params": 12288000,
  "attention_params_per_layer": 393216,
  "qk_norm_params_per_layer": 128,
  "ffn_params_per_layer": 1179648,
  "norm_params_per_layer": 768,
  "block_params_per_layer": 1573760,
  "all_block_params": 18885120,
  "final_norm_params": 384,
  "lm_head_params": 0,
  "estimated_total_params": 31173504
}

Estimated total params: 31,173,504

[OK] Dense config and parameter estimate validated.


In [6]:
# ============================================================
# CELL 5 — ARCHITECTURE MODULES
# RMSNorm, RoPE, GQA Attention, SwiGLU FFN
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [..., dim]
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        y = x_float * rms
        return (y.to(dtype) * self.weight)


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int, base: float = 10000.0):
        super().__init__()
        assert dim % 2 == 0, "RoPE dim must be even."

        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)  # [T, dim/2]

        cos = torch.cos(freqs)
        sin = torch.sin(freqs)

        self.register_buffer("cos_cached", cos, persistent=False)
        self.register_buffer("sin_cached", sin, persistent=False)

    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
        # cos/sin: [T, dim/2]
        cos = self.cos_cached[:seq_len].to(device=device, dtype=dtype)
        sin = self.sin_cached[:seq_len].to(device=device, dtype=dtype)
        return cos, sin


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """
    x   : [B, H, T, D]
    cos : [T, D/2]
    sin : [T, D/2]
    """
    b, h, t, d = x.shape
    x_even = x[..., 0::2]  # [B, H, T, D/2]
    x_odd = x[..., 1::2]   # [B, H, T, D/2]

    cos = cos.view(1, 1, t, d // 2)
    sin = sin.view(1, 1, t, d // 2)

    out_even = x_even * cos - x_odd * sin
    out_odd = x_even * sin + x_odd * cos

    out = torch.empty_like(x)
    out[..., 0::2] = out_even
    out[..., 1::2] = out_odd
    return out


class GQASelfAttention(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()

        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0

        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.kv_repeat = cfg.n_heads // cfg.n_kv_heads
        self.dropout_p = cfg.dropout
        self.qk_norm_enabled = cfg.qk_norm

        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

        if self.qk_norm_enabled:
            self.q_norm = RMSNorm(self.head_dim)
            self.k_norm = RMSNorm(self.head_dim)

        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(
        self,
        x: torch.Tensor,
        cos: torch.Tensor,
        sin: torch.Tensor,
    ) -> torch.Tensor:
        # x: [B, T, d_model]
        b, t, d = x.shape

        q = self.q_proj(x)  # [B, T, H*Dh]
        k = self.k_proj(x)  # [B, T, Hkv*Dh]
        v = self.v_proj(x)  # [B, T, Hkv*Dh]

        q = q.view(b, t, self.n_heads, self.head_dim).transpose(1, 2)      # [B, H, T, Dh]
        k = k.view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)   # [B, Hkv, T, Dh]
        v = v.view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)   # [B, Hkv, T, Dh]

        if self.qk_norm_enabled:
            q = self.q_norm(q)  # [B, H, T, Dh]
            k = self.k_norm(k)  # [B, Hkv, T, Dh]

        q = apply_rope(q, cos, sin)  # [B, H, T, Dh]
        k = apply_rope(k, cos, sin)  # [B, Hkv, T, Dh]

        if self.kv_repeat > 1:
            k = k.repeat_interleave(self.kv_repeat, dim=1)  # [B, H, T, Dh]
            v = v.repeat_interleave(self.kv_repeat, dim=1)  # [B, H, T, Dh]

        # PyTorch SDPA uses memory-efficient kernels when available.
        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=None,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=True,
        )  # [B, H, T, Dh]

        y = y.transpose(1, 2).contiguous().view(b, t, d)  # [B, T, d_model]
        y = self.o_proj(y)                               # [B, T, d_model]
        y = self.resid_dropout(y)
        return y


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()

        self.gate_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.down_proj = nn.Linear(cfg.ffn_dim, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, d_model]
        gate = self.gate_proj(x)              # [B, T, ffn_dim]
        up = self.up_proj(x)                  # [B, T, ffn_dim]
        y = F.silu(gate) * up                 # [B, T, ffn_dim]
        y = self.down_proj(y)                 # [B, T, d_model]
        y = self.dropout(y)
        return y


class DecoderBlock(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model)
        self.attn = GQASelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model)
        self.ffn = SwiGLUFFN(cfg)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        # x: [B, T, d_model]
        x = x + self.attn(self.attn_norm(x), cos, sin)  # [B, T, d_model]
        x = x + self.ffn(self.ffn_norm(x))              # [B, T, d_model]
        return x


print("[OK] Architecture modules defined.")

[OK] Architecture modules defined.


In [7]:
# ============================================================
# CELL 6 — BLEACH-DENSE-32M MODEL
# ============================================================

class BLEACHDenseForCausalLM(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        self.cfg = cfg

        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

        head_dim = cfg.d_model // cfg.n_heads
        self.rope = RotaryEmbedding(
            dim=head_dim,
            max_seq_len=cfg.block_size,
            base=cfg.rope_base,
        )

        self.layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)

        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def count_parameters(self) -> Dict[str, int]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)

        emb = self.token_embedding.weight.numel()
        non_emb = total - emb

        return {
            "total_params": int(total),
            "trainable_params": int(trainable),
            "embedding_params": int(emb),
            "non_embedding_params": int(non_emb),
        }

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        aux_future_loss_weight: Optional[float] = None,
        return_hidden_states: bool = False,
    ) -> Dict[str, torch.Tensor]:
        """
        input_ids: [B, T]
        labels   : [B, T], with -100 for ignored positions.
        """
        b, t = input_ids.shape
        assert t <= self.cfg.block_size, f"Sequence length {t} > block size {self.cfg.block_size}"

        x = self.token_embedding(input_ids)  # [B, T, d_model]
        x = self.dropout(x)

        cos, sin = self.rope(seq_len=t, device=input_ids.device, dtype=x.dtype)

        hidden_states = []
        for layer in self.layers:
            x = layer(x, cos, sin)  # [B, T, d_model]
            if return_hidden_states:
                hidden_states.append(x)

        x = self.final_norm(x)      # [B, T, d_model]
        logits = self.lm_head(x)    # [B, T, vocab_size]

        out = {
            "logits": logits,
            "last_hidden_state": x,
        }

        if return_hidden_states:
            out["hidden_states"] = hidden_states

        if labels is not None:
            # Standard next-token prediction.
            shift_logits = logits[:, :-1, :].contiguous()    # [B, T-1, V]
            shift_labels = labels[:, 1:].contiguous()        # [B, T-1]

            loss_ntp = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

            if aux_future_loss_weight is None:
                aux_future_loss_weight = self.cfg.aux_future_loss_weight

            # Light future-token auxiliary objective:
            # logits at position t also weakly predict token t+2.
            if aux_future_loss_weight > 0 and logits.size(1) > 2:
                future_logits = logits[:, :-2, :].contiguous()   # [B, T-2, V]
                future_labels = labels[:, 2:].contiguous()       # [B, T-2]

                loss_future = F.cross_entropy(
                    future_logits.view(-1, future_logits.size(-1)),
                    future_labels.view(-1),
                    ignore_index=-100,
                )
            else:
                loss_future = torch.zeros_like(loss_ntp)

            loss = loss_ntp + float(aux_future_loss_weight) * loss_future

            out["loss"] = loss
            out["loss_ntp"] = loss_ntp.detach()
            out["loss_future"] = loss_future.detach()

        return out


model = BLEACHDenseForCausalLM(MODEL_CFG)
param_report = model.count_parameters()

print("=" * 100)
print("INSTANTIATED BLEACH-DENSE-32M")
print("=" * 100)
print(json.dumps(param_report, indent=2))
print(f"Total params: {param_report['total_params']:,}")

assert CFG.expected_min_params <= param_report["total_params"] <= CFG.expected_max_params

# Tied embedding validation.
if MODEL_CFG.tie_embeddings:
    assert model.lm_head.weight.data_ptr() == model.token_embedding.weight.data_ptr()

print("[OK] Model instantiated and parameter range validated.")

INSTANTIATED BLEACH-DENSE-32M
{
  "total_params": 31173504,
  "trainable_params": 31173504,
  "embedding_params": 12288000,
  "non_embedding_params": 18885504
}
Total params: 31,173,504
[OK] Model instantiated and parameter range validated.


In [8]:
# ============================================================
# CELL 7 — BATCH VALIDATION FROM PACKED SHARD
# ============================================================

def validate_batch(batch: Dict[str, torch.Tensor], cfg: P2Config) -> Dict[str, Any]:
    input_ids = batch["input_ids"]
    labels = batch["labels"]
    attention_mask = batch["attention_mask"]
    dialect_id = batch["dialect_id"]

    report = {
        "input_ids_shape": list(input_ids.shape),
        "labels_shape": list(labels.shape),
        "attention_mask_shape": list(attention_mask.shape),
        "dialect_id_shape": list(dialect_id.shape),
        "input_ids_min": int(input_ids.min()),
        "input_ids_max": int(input_ids.max()),
        "labels_min": int(labels[labels != -100].min()) if (labels != -100).any() else None,
        "labels_max": int(labels[labels != -100].max()) if (labels != -100).any() else None,
        "unk_count": int((input_ids == cfg.unk_token_id).sum()),
        "pad_count": int((input_ids == cfg.pad_token_id).sum()),
        "nonpad_count": int(attention_mask.sum()),
        "valid_shape": tuple(input_ids.shape) == (cfg.test_batch_size, cfg.block_size),
        "valid_token_ids": bool(input_ids.min() >= 0 and input_ids.max() < cfg.vocab_size),
        "valid_labels": bool(
            ((labels == -100) | ((labels >= 0) & (labels < cfg.vocab_size))).all()
        ),
    }

    report["passed"] = bool(
        report["valid_shape"]
        and report["valid_token_ids"]
        and report["valid_labels"]
        and report["unk_count"] == 0
    )
    return report


batch_report = validate_batch(batch_cpu, CFG)

print(json.dumps(batch_report, indent=2))
assert batch_report["passed"], "Packed batch validation failed."

with open(REPORT_DIR / "batch_validation_report.json", "w", encoding="utf-8") as f:
    json.dump(batch_report, f, ensure_ascii=False, indent=2)

print("[OK] Packed batch validation passed.")

{
  "input_ids_shape": [
    2,
    512
  ],
  "labels_shape": [
    2,
    512
  ],
  "attention_mask_shape": [
    2,
    512
  ],
  "dialect_id_shape": [
    2
  ],
  "input_ids_min": 0,
  "input_ids_max": 27459,
  "labels_min": 2,
  "labels_max": 27459,
  "unk_count": 0,
  "pad_count": 6,
  "nonpad_count": 1018,
  "valid_shape": true,
  "valid_token_ids": true,
  "valid_labels": true,
  "passed": true
}
[OK] Packed batch validation passed.


In [9]:
# ============================================================
# CELL 8 — FORWARD/BACKWARD ARCHITECTURE TESTS
# ============================================================

def get_amp_dtype() -> Optional[torch.dtype]:
    if DEVICE.type != "cuda":
        return None
    if CFG.use_bf16_if_available and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


amp_dtype = get_amp_dtype()

model = model.to(DEVICE)
model.train()

batch = {k: v.to(DEVICE) for k, v in batch_cpu.items()}

test_report: Dict[str, Any] = {
    "device": str(DEVICE),
    "amp_dtype": str(amp_dtype),
    "config": asdict(MODEL_CFG),
    "parameter_report": param_report,
}

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.0)

try:
    optimizer.zero_grad(set_to_none=True)

    if amp_dtype is not None:
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            out = model(
                input_ids=batch["input_ids"],                 # [B, 512]
                labels=batch["labels"],                       # [B, 512]
                aux_future_loss_weight=CFG.aux_future_loss_weight,
                return_hidden_states=False,
            )
    else:
        out = model(
            input_ids=batch["input_ids"],
            labels=batch["labels"],
            aux_future_loss_weight=CFG.aux_future_loss_weight,
            return_hidden_states=False,
        )

    logits = out["logits"]
    loss = out["loss"]

    test_report.update({
        "logits_shape": list(logits.shape),
        "last_hidden_state_shape": list(out["last_hidden_state"].shape),
        "loss": float(loss.detach().cpu()),
        "loss_ntp": float(out["loss_ntp"].detach().cpu()),
        "loss_future": float(out["loss_future"].detach().cpu()),
        "logits_finite": bool(torch.isfinite(logits).all().item()),
        "loss_finite": bool(torch.isfinite(loss).item()),
    })

    loss.backward()

    grad_norm_sq = 0.0
    nonfinite_grad_tensors = 0
    grad_tensor_count = 0

    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        grad_tensor_count += 1
        if not torch.isfinite(p.grad).all():
            nonfinite_grad_tensors += 1
        grad_norm_sq += float(p.grad.detach().float().norm(2).cpu().item() ** 2)

    grad_norm = math.sqrt(grad_norm_sq)

    optimizer.step()

    test_report.update({
        "grad_tensor_count": int(grad_tensor_count),
        "nonfinite_grad_tensors": int(nonfinite_grad_tensors),
        "global_grad_norm": float(grad_norm),
        "global_grad_norm_finite": bool(math.isfinite(grad_norm)),
    })

except RuntimeError as e:
    test_report["runtime_error"] = str(e)
    raise

finally:
    optimizer.zero_grad(set_to_none=True)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Hard tests.
expected_logits_shape = [CFG.test_batch_size, CFG.block_size, CFG.vocab_size]
expected_hidden_shape = [CFG.test_batch_size, CFG.block_size, CFG.d_model]

hard_failures = []

if test_report["logits_shape"] != expected_logits_shape:
    hard_failures.append(f"Bad logits shape: {test_report['logits_shape']} expected {expected_logits_shape}")

if test_report["last_hidden_state_shape"] != expected_hidden_shape:
    hard_failures.append(
        f"Bad hidden shape: {test_report['last_hidden_state_shape']} expected {expected_hidden_shape}"
    )

if not test_report["logits_finite"]:
    hard_failures.append("Logits contain non-finite values.")

if not test_report["loss_finite"]:
    hard_failures.append("Loss is non-finite.")

if test_report["nonfinite_grad_tensors"] > 0:
    hard_failures.append(f"Non-finite gradients in {test_report['nonfinite_grad_tensors']} tensors.")

if not test_report["global_grad_norm_finite"]:
    hard_failures.append("Global grad norm is non-finite.")

if not (CFG.expected_min_params <= param_report["total_params"] <= CFG.expected_max_params):
    hard_failures.append(f"Parameter count out of expected range: {param_report['total_params']}")

test_report["hard_failures"] = hard_failures
test_report["passed"] = len(hard_failures) == 0

print("=" * 100)
print("ARCHITECTURE TEST REPORT")
print("=" * 100)
print(json.dumps(test_report, indent=2))

with open(REPORT_DIR / "dense_architecture_tests.json", "w", encoding="utf-8") as f:
    json.dump(test_report, f, ensure_ascii=False, indent=2)

assert test_report["passed"], "Dense architecture tests failed."

print("\n[OK] Forward/backward architecture tests passed.")

ARCHITECTURE TEST REPORT
{
  "device": "cuda",
  "amp_dtype": "torch.bfloat16",
  "config": {
    "vocab_size": 32000,
    "block_size": 512,
    "n_layers": 12,
    "d_model": 384,
    "n_heads": 6,
    "n_kv_heads": 2,
    "ffn_dim": 1024,
    "rope_base": 10000.0,
    "dropout": 0.12,
    "qk_norm": true,
    "tie_embeddings": true,
    "pad_token_id": 0,
    "unk_token_id": 1,
    "bos_token_id": 2,
    "eos_token_id": 3,
    "aux_future_loss_weight": 0.05
  },
  "parameter_report": {
    "total_params": 31173504,
    "trainable_params": 31173504,
    "embedding_params": 12288000,
    "non_embedding_params": 18885504
  },
  "logits_shape": [
    2,
    512,
    32000
  ],
  "last_hidden_state_shape": [
    2,
    512,
    384
  ],
  "loss": 11.07490062713623,
  "loss_ntp": 10.547675132751465,
  "loss_future": 10.544502258300781,
  "logits_finite": true,
  "loss_finite": true,
  "grad_tensor_count": 134,
  "nonfinite_grad_tensors": 0,
  "global_grad_norm": 14.40431964093209,
  "glob

In [10]:
# ============================================================
# CELL 9 — EXPORT REPORTS AND REUSABLE SOURCE FILE
# ============================================================

# Save model initial state for reproducibility/debugging only.
# Portion 3 should train from this architecture config, not treat this as trained.
initial_state_path = OUT_DIR / "bleach_dense_32m_initial_state.pt"
torch.save(
    {
        "model_config": asdict(MODEL_CFG),
        "state_dict": model.cpu().state_dict(),
        "parameter_report": param_report,
        "architecture_test_report": test_report,
    },
    initial_state_path,
)

# Move back to device only if needed later.
model = model.to(DEVICE)

parameter_report = {
    "portion": "bleach_p2_dense_architecture",
    "model_name": "BLEACH-Dense-32M",
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_config": asdict(MODEL_CFG),
    "estimate": estimate,
    "actual": param_report,
    "expected_param_range": {
        "min": CFG.expected_min_params,
        "max": CFG.expected_max_params,
    },
    "passed": bool(CFG.expected_min_params <= param_report["total_params"] <= CFG.expected_max_params),
}

with open(REPORT_DIR / "parameter_report.json", "w", encoding="utf-8") as f:
    json.dump(parameter_report, f, ensure_ascii=False, indent=2)

# YAML export if PyYAML is available.
try:
    import yaml
    with open(CONFIG_DIR / "dense_32m_config.yaml", "w", encoding="utf-8") as f:
        yaml.safe_dump(asdict(MODEL_CFG), f, sort_keys=False, allow_unicode=True)
    print("Saved:", CONFIG_DIR / "dense_32m_config.yaml")
except Exception as e:
    print("[INFO] PyYAML unavailable; JSON config already saved.", e)

# Save a compact reusable architecture source file.
source_code = r'''
from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class BLEACHDenseConfig:
    vocab_size: int = 32000
    block_size: int = 512
    n_layers: int = 12
    d_model: int = 384
    n_heads: int = 6
    n_kv_heads: int = 2
    ffn_dim: int = 1024
    rope_base: float = 10000.0
    dropout: float = 0.12
    qk_norm: bool = True
    tie_embeddings: bool = True
    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3
    aux_future_loss_weight: float = 0.05


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms).to(dtype) * self.weight


class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_seq_len: int, base: float = 10000.0):
        super().__init__()
        assert dim % 2 == 0
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        t = torch.arange(max_seq_len, dtype=torch.float32)
        freqs = torch.outer(t, inv_freq)
        self.register_buffer("cos_cached", torch.cos(freqs), persistent=False)
        self.register_buffer("sin_cached", torch.sin(freqs), persistent=False)

    def forward(self, seq_len: int, device: torch.device, dtype: torch.dtype) -> Tuple[torch.Tensor, torch.Tensor]:
        return (
            self.cos_cached[:seq_len].to(device=device, dtype=dtype),
            self.sin_cached[:seq_len].to(device=device, dtype=dtype),
        )


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    b, h, t, d = x.shape
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    cos = cos.view(1, 1, t, d // 2)
    sin = sin.view(1, 1, t, d // 2)

    out = torch.empty_like(x)
    out[..., 0::2] = x_even * cos - x_odd * sin
    out[..., 1::2] = x_even * sin + x_odd * cos
    return out


class GQASelfAttention(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0
        assert cfg.n_heads % cfg.n_kv_heads == 0

        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.n_kv_heads = cfg.n_kv_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.kv_repeat = cfg.n_heads // cfg.n_kv_heads
        self.dropout_p = cfg.dropout
        self.qk_norm_enabled = cfg.qk_norm

        self.q_proj = nn.Linear(cfg.d_model, cfg.n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)

        if cfg.qk_norm:
            self.q_norm = RMSNorm(self.head_dim)
            self.k_norm = RMSNorm(self.head_dim)

        self.resid_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape

        q = self.q_proj(x).view(b, t, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.n_kv_heads, self.head_dim).transpose(1, 2)

        if self.qk_norm_enabled:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)

        if self.kv_repeat > 1:
            k = k.repeat_interleave(self.kv_repeat, dim=1)
            v = v.repeat_interleave(self.kv_repeat, dim=1)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=True,
        )

        y = y.transpose(1, 2).contiguous().view(b, t, d)
        return self.resid_dropout(self.o_proj(y))


class SwiGLUFFN(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.up_proj = nn.Linear(cfg.d_model, cfg.ffn_dim, bias=False)
        self.down_proj = nn.Linear(cfg.ffn_dim, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class DecoderBlock(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model)
        self.attn = GQASelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model)
        self.ffn = SwiGLUFFN(cfg)

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.attn_norm(x), cos, sin)
        x = x + self.ffn(self.ffn_norm(x))
        return x


class BLEACHDenseForCausalLM(nn.Module):
    def __init__(self, cfg: BLEACHDenseConfig):
        super().__init__()
        self.cfg = cfg

        self.token_embedding = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.dropout = nn.Dropout(cfg.dropout)

        head_dim = cfg.d_model // cfg.n_heads
        self.rope = RotaryEmbedding(head_dim, cfg.block_size, cfg.rope_base)

        self.layers = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_embeddings:
            self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def count_parameters(self) -> Dict[str, int]:
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        emb = self.token_embedding.weight.numel()
        return {
            "total_params": int(total),
            "trainable_params": int(trainable),
            "embedding_params": int(emb),
            "non_embedding_params": int(total - emb),
        }

    def forward(
        self,
        input_ids: torch.Tensor,
        labels: Optional[torch.Tensor] = None,
        aux_future_loss_weight: Optional[float] = None,
        return_hidden_states: bool = False,
    ) -> Dict[str, torch.Tensor]:
        b, t = input_ids.shape
        assert t <= self.cfg.block_size

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        cos, sin = self.rope(t, input_ids.device, x.dtype)

        hidden_states = []
        for layer in self.layers:
            x = layer(x, cos, sin)
            if return_hidden_states:
                hidden_states.append(x)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        out = {
            "logits": logits,
            "last_hidden_state": x,
        }

        if return_hidden_states:
            out["hidden_states"] = hidden_states

        if labels is not None:
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss_ntp = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

            if aux_future_loss_weight is None:
                aux_future_loss_weight = self.cfg.aux_future_loss_weight

            if aux_future_loss_weight > 0 and logits.size(1) > 2:
                future_logits = logits[:, :-2, :].contiguous()
                future_labels = labels[:, 2:].contiguous()
                loss_future = F.cross_entropy(
                    future_logits.view(-1, future_logits.size(-1)),
                    future_labels.view(-1),
                    ignore_index=-100,
                )
            else:
                loss_future = torch.zeros_like(loss_ntp)

            loss = loss_ntp + float(aux_future_loss_weight) * loss_future

            out["loss"] = loss
            out["loss_ntp"] = loss_ntp.detach()
            out["loss_future"] = loss_future.detach()

        return out
'''

source_path = SOURCE_DIR / "bleach_dense_model.py"
with open(source_path, "w", encoding="utf-8") as f:
    f.write(source_code)

manifest = {
    "portion": "bleach_p2_dense_architecture",
    "status": "passed" if test_report["passed"] else "failed",
    "model_name": "BLEACH-Dense-32M",
    "outputs": {
        "dense_config_json": str(CONFIG_DIR / "dense_32m_config.json"),
        "dense_config_yaml": str(CONFIG_DIR / "dense_32m_config.yaml"),
        "parameter_report": str(REPORT_DIR / "parameter_report.json"),
        "batch_validation_report": str(REPORT_DIR / "batch_validation_report.json"),
        "architecture_tests": str(REPORT_DIR / "dense_architecture_tests.json"),
        "initial_state": str(initial_state_path),
        "source_file": str(source_path),
    },
}

with open(OUT_DIR / "p2_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Saved:", REPORT_DIR / "parameter_report.json")
print("Saved:", REPORT_DIR / "dense_architecture_tests.json")
print("Saved:", source_path)
print("Saved:", OUT_DIR / "p2_manifest.json")
print("Saved:", initial_state_path)

Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/config/dense_32m_config.yaml
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/reports/parameter_report.json
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/reports/dense_architecture_tests.json
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/source/bleach_dense_model.py
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/p2_manifest.json
Saved: /kaggle/working/artifacts/bleach_p2_dense_arch/bleach_dense_32m_initial_state.pt


In [11]:
# ============================================================
# CELL 10 — FINAL CHECKPOINT
# ============================================================

print("=" * 100)
print("PORTION 2 FINAL CHECKPOINT")
print("=" * 100)

print("Model name       : BLEACH-Dense-32M")
print("Status           :", "PASSED" if test_report["passed"] else "FAILED")
print("Total params     :", f"{param_report['total_params']:,}")
print("Embedding params :", f"{param_report['embedding_params']:,}")
print("Non-embed params :", f"{param_report['non_embedding_params']:,}")
print("Logits shape     :", test_report["logits_shape"])
print("Loss             :", test_report["loss"])
print("NTP loss         :", test_report["loss_ntp"])
print("Future loss      :", test_report["loss_future"])
print("Grad norm        :", test_report["global_grad_norm"])
print("Output dir       :", OUT_DIR)

print("\nArtifacts:")
for k, v in manifest["outputs"].items():
    print(f"  {k:28s}: {v}")

assert test_report["passed"]
assert parameter_report["passed"]

print("\n✅ BLEACH SLM/LLM PORTION 2 COMPLETE")
print("Next: Portion 3 — Dense CLM training with Muon+AdamW, source-aware loss, per-dialect PPL, and checkpointing.")

PORTION 2 FINAL CHECKPOINT
Model name       : BLEACH-Dense-32M
Status           : PASSED
Total params     : 31,173,504
Embedding params : 12,288,000
Non-embed params : 18,885,504
Logits shape     : [2, 512, 32000]
Loss             : 11.07490062713623
NTP loss         : 10.547675132751465
Future loss      : 10.544502258300781
Grad norm        : 14.40431964093209
Output dir       : /kaggle/working/artifacts/bleach_p2_dense_arch

Artifacts:
  dense_config_json           : /kaggle/working/artifacts/bleach_p2_dense_arch/config/dense_32m_config.json
  dense_config_yaml           : /kaggle/working/artifacts/bleach_p2_dense_arch/config/dense_32m_config.yaml
  parameter_report            : /kaggle/working/artifacts/bleach_p2_dense_arch/reports/parameter_report.json
  batch_validation_report     : /kaggle/working/artifacts/bleach_p2_dense_arch/reports/batch_validation_report.json
  architecture_tests          : /kaggle/working/artifacts/bleach_p2_dense_arch/reports/dense_architecture_tests.json


## portion 3


In [12]:
# ============================================================
# CELL 1 — IMPORTS, REPRODUCIBILITY, DEVICE
# BLEACH SLM/LLM PORTION 3 — DENSE CLM TRAINING
# ============================================================

from __future__ import annotations

import os
import gc
import sys
import json
import math
import time
import random
import shutil
import warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple, Iterable
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

SEED = 42


def set_seed(seed: int = 42, deterministic: bool = False) -> None:
    """
    For training speed on RTX PRO 6000, deterministic=False is recommended.
    Set deterministic=True only for exact reproducibility debugging.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.benchmark = not deterministic
    torch.backends.cudnn.deterministic = deterministic


set_seed(SEED, deterministic=False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

print("=" * 100)
print("BLEACH SLM/LLM PORTION 3 — DENSE CLM TRAINING")
print("=" * 100)
print(f"Seed   : {SEED}")
print(f"Device : {DEVICE}")
print(f"Torch  : {torch.__version__}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {props.total_memory / 1024**3:.2f} GB")
    print(f"bf16   : {torch.cuda.is_bf16_supported()}")

print("=" * 100)

BLEACH SLM/LLM PORTION 3 — DENSE CLM TRAINING
Seed   : 42
Device : cuda
Torch  : 2.10.0+cu128
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 94.97 GB
bf16   : True


In [13]:
# ============================================================
# CELL 2 — CONFIG AND ARTIFACT RESOLVER
# ============================================================

@dataclass(frozen=True)
class P3TrainConfig:
    # Output.
    output_dir: str = "/kaggle/working/artifacts/bleach_p3_dense_clm"

    # Model/data.
    model_name: str = "BLEACH-Dense-32M"
    vocab_size: int = 32000
    block_size: int = 512
    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3

    # Training.
    epochs: int = 48
    micro_batch_size: int = 24          # RTX PRO 6000 96GB: 24 is safe. Try 32 if memory allows.
    grad_accum_steps: int = 4           # effective batch = 96 sequences
    eval_micro_batch_size: int = 32
    num_workers: int = 2
    pin_memory: bool = True

    # Optimizer.
    use_muon: bool = True
    muon_lr: float = 0.015
    adamw_lr: float = 3e-4
    min_lr_ratio: float = 0.10
    weight_decay: float = 0.10
    adamw_betas: Tuple[float, float] = (0.9, 0.95)
    adamw_eps: float = 1e-8
    muon_momentum: float = 0.95

    # Schedule.
    warmup_ratio: float = 0.05
    grad_clip_norm: float = 1.0

    # Future-token auxiliary loss.
    aux_future_final_weight: float = 0.05
    aux_future_start_after_warmup: bool = True
    aux_future_ramp_ratio: float = 0.10

    # Source-aware loss weighting.
    real_block_weight: float = 1.00
    synthetic_block_weight: float = 0.65
    synthetic_block_weight_low_real: float = 0.45
    low_real_synthetic_heavy_dialects: Tuple[str, ...] = ("KHU", "RAJ")
    min_quality_weight: float = 0.50
    max_quality_weight: float = 1.10
    min_block_weight: float = 0.25
    max_block_weight: float = 1.50

    # Weighted sampler.
    use_weighted_sampler: bool = True
    dialect_sampling_alpha: float = 0.30
    sampler_replacement: bool = True

    # Evaluation/checkpointing.
    eval_every_epochs: int = 1
    save_every_epochs: int = 8
    keep_epoch_checkpoints: bool = True
    resume_from_last: bool = True
    save_optimizer_state: bool = True

    # Early stop: keep False to honor requested 48 epochs.
    use_early_stopping: bool = False
    early_stopping_patience: int = 10

    # Divergence guard.
    abort_on_nonfinite_loss: bool = True
    max_reasonable_loss: float = 30.0

    # Rare-token evaluation.
    rare_token_max_count: int = 5
    rare_token_min_id: int = 17    # skip special + dialect tags if dialect IDs are 4–16

    # Precision.
    prefer_bf16: bool = True

    # Logging.
    log_every_steps: int = 10


CFG = P3TrainConfig()

OUT_DIR = Path(CFG.output_dir)
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
REPORT_DIR = OUT_DIR / "reports"
PLOT_DIR = OUT_DIR / "plots"
CONFIG_DIR = OUT_DIR / "config"
CACHE_DIR = OUT_DIR / "cache"

for p in [OUT_DIR, CHECKPOINT_DIR, REPORT_DIR, PLOT_DIR, CONFIG_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)


DIALECTS = ["BAR", "CHI", "KHU", "KIS", "MYM", "NAR", "NOA", "NSD", "RAJ", "RAN", "STD", "SYL", "TAN"]
DIALECT_TO_ID = {d: i for i, d in enumerate(DIALECTS)}
ID_TO_DIALECT = {i: d for d, i in DIALECT_TO_ID.items()}


def read_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def resolve_slm_p1_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/slm_p1_data"),
        Path("artifacts/slm_p1_data"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("slm_p1_data/packed/packing_manifest.json")):
            candidates.append(m.parents[1])
        for m in sorted(input_root.rglob("artifacts/slm_p1_data/packed/packing_manifest.json")):
            candidates.append(m.parents[2])

    for c in candidates:
        if (c / "packed" / "packing_manifest.json").exists():
            return c

    raise FileNotFoundError("Could not resolve SLM Portion-1 root.")


def resolve_p2_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/bleach_p2_dense_arch"),
        Path("artifacts/bleach_p2_dense_arch"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("bleach_p2_dense_arch/p2_manifest.json")):
            candidates.append(m.parent)
        for m in sorted(input_root.rglob("artifacts/bleach_p2_dense_arch/p2_manifest.json")):
            candidates.append(m.parent)

    for c in candidates:
        if (c / "source" / "bleach_dense_model.py").exists():
            return c

    raise FileNotFoundError("Could not resolve BLEACH Portion-2 architecture root.")


SLM_P1_ROOT = resolve_slm_p1_root()
P2_ROOT = resolve_p2_root()

print("Resolved SLM P1 root:", SLM_P1_ROOT)
print("Resolved P2 root    :", P2_ROOT)

write_json(CONFIG_DIR / "p3_train_config.json", asdict(CFG))
write_json(CONFIG_DIR / "p3_resolved_paths.json", {
    "slm_p1_root": str(SLM_P1_ROOT),
    "p2_root": str(P2_ROOT),
    "output_dir": str(OUT_DIR),
})

print("Saved:", CONFIG_DIR / "p3_train_config.json")

Resolved SLM P1 root: /kaggle/input/notebooks/nabidnur/bleach-slm/artifacts/slm_p1_data
Resolved P2 root    : /kaggle/working/artifacts/bleach_p2_dense_arch
Saved: /kaggle/working/artifacts/bleach_p3_dense_clm/config/p3_train_config.json


In [14]:
# ============================================================
# CELL 3 — IMPORT BLEACH-DENSE MODEL FROM PORTION 2
# ============================================================

P2_SOURCE_DIR = P2_ROOT / "source"
sys.path.insert(0, str(P2_SOURCE_DIR))

from bleach_dense_model import BLEACHDenseConfig, BLEACHDenseForCausalLM

dense_config_path_json = P2_ROOT / "config" / "dense_32m_config.json"
assert dense_config_path_json.exists(), f"Missing P2 dense config: {dense_config_path_json}"

dense_cfg_dict = read_json(dense_config_path_json)
MODEL_CFG = BLEACHDenseConfig(**dense_cfg_dict)

model = BLEACHDenseForCausalLM(MODEL_CFG)
param_report = model.count_parameters()

print("=" * 100)
print("LOADED BLEACH-DENSE MODEL")
print("=" * 100)
print("Model config:")
print(json.dumps(dense_cfg_dict, indent=2))
print("\nParameter report:")
print(json.dumps(param_report, indent=2))

assert param_report["total_params"] == 31_173_504, (
    f"Unexpected parameter count: {param_report['total_params']}. "
    "Expected 31,173,504 from Portion 2."
)

# Initialize from Portion-2 initial architecture checkpoint if present.
p2_initial_state = P2_ROOT / "bleach_dense_32m_initial_state.pt"
if p2_initial_state.exists():
    ckpt = torch.load(p2_initial_state, map_location="cpu")
    missing, unexpected = model.load_state_dict(ckpt["state_dict"], strict=False)
    print("\nLoaded P2 initial state.")
    print("Missing keys   :", missing)
    print("Unexpected keys:", unexpected)
else:
    print("\n[WARN] P2 initial state not found. Using freshly initialized model.")

model = model.to(DEVICE)

print("\n[OK] Model ready on device.")

LOADED BLEACH-DENSE MODEL
Model config:
{
  "vocab_size": 32000,
  "block_size": 512,
  "n_layers": 12,
  "d_model": 384,
  "n_heads": 6,
  "n_kv_heads": 2,
  "ffn_dim": 1024,
  "rope_base": 10000.0,
  "dropout": 0.12,
  "qk_norm": true,
  "tie_embeddings": true,
  "pad_token_id": 0,
  "unk_token_id": 1,
  "bos_token_id": 2,
  "eos_token_id": 3,
  "aux_future_loss_weight": 0.05
}

Parameter report:
{
  "total_params": 31173504,
  "trainable_params": 31173504,
  "embedding_params": 12288000,
  "non_embedding_params": 18885504
}

Loaded P2 initial state.
Missing keys   : []
Unexpected keys: []

[OK] Model ready on device.


In [15]:
# ============================================================
# CELL 4 — LOAD PACKED DATASETS AND BUILD METADATA
# ============================================================

class PackedBlockDataset(Dataset):
    def __init__(
        self,
        shard_dir: Path,
        split_name: str,
        cfg: P3TrainConfig,
        load_to_ram: bool = True,
    ):
        self.shard_dir = Path(shard_dir)
        self.split_name = split_name
        self.cfg = cfg
        self.shards = sorted(self.shard_dir.glob("*.npz"))

        assert len(self.shards) > 0, f"No shards found in {self.shard_dir}"

        self.load_to_ram = load_to_ram

        if load_to_ram:
            arrays = defaultdict(list)
            for sp in self.shards:
                z = np.load(sp)
                for k in z.files:
                    arrays[k].append(z[k])

            self.arrays = {}
            for k, parts in arrays.items():
                self.arrays[k] = np.concatenate(parts, axis=0)

            self.num_blocks = int(self.arrays["input_ids"].shape[0])
        else:
            raise NotImplementedError("For this dataset size, load_to_ram=True is recommended.")

        self._validate()

    def _validate(self) -> None:
        a = self.arrays

        required = [
            "input_ids",
            "attention_mask",
            "labels",
            "dialect_id",
            "dialect_tag_id",
            "synthetic_ratio",
            "mean_quality",
        ]
        missing = [k for k in required if k not in a]
        assert not missing, f"{self.split_name} missing arrays: {missing}"

        assert a["input_ids"].shape[1] == self.cfg.block_size
        assert a["labels"].shape == a["input_ids"].shape
        assert a["attention_mask"].shape == a["input_ids"].shape

        assert int(a["input_ids"].min()) >= 0
        assert int(a["input_ids"].max()) < self.cfg.vocab_size
        assert int((a["input_ids"] == self.cfg.unk_token_id).sum()) == 0

    def __len__(self) -> int:
        return self.num_blocks

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        a = self.arrays

        item = {
            "input_ids": torch.as_tensor(a["input_ids"][idx], dtype=torch.long),                # [T]
            "attention_mask": torch.as_tensor(a["attention_mask"][idx], dtype=torch.long),      # [T]
            "labels": torch.as_tensor(a["labels"][idx], dtype=torch.long),                      # [T]
            "dialect_id": torch.as_tensor(a["dialect_id"][idx], dtype=torch.long),              # []
            "dialect_tag_id": torch.as_tensor(a["dialect_tag_id"][idx], dtype=torch.long),      # []
            "synthetic_ratio": torch.as_tensor(a["synthetic_ratio"][idx], dtype=torch.float32), # []
            "mean_quality": torch.as_tensor(a["mean_quality"][idx], dtype=torch.float32),       # []
        }

        return item

    def metadata_frame(self) -> pd.DataFrame:
        a = self.arrays
        df = pd.DataFrame({
            "idx": np.arange(self.num_blocks),
            "split": self.split_name,
            "dialect_id": a["dialect_id"].astype(int),
            "dialect": [ID_TO_DIALECT.get(int(x), f"UNK_{int(x)}") for x in a["dialect_id"]],
            "dialect_tag_id": a["dialect_tag_id"].astype(int),
            "synthetic_ratio": a["synthetic_ratio"].astype(float),
            "mean_quality": a["mean_quality"].astype(float),
            "nonpad_tokens": a["attention_mask"].sum(axis=1).astype(int),
        })
        return df


train_dir = SLM_P1_ROOT / "packed" / "train_block512_shards"
val_dir = SLM_P1_ROOT / "packed" / "val_block512_shards"
eval_dir = SLM_P1_ROOT / "packed" / "eval_block512_shards"

train_ds = PackedBlockDataset(train_dir, "train", CFG, load_to_ram=True)
val_ds = PackedBlockDataset(val_dir, "val", CFG, load_to_ram=True)
eval_ds = PackedBlockDataset(eval_dir, "external_eval", CFG, load_to_ram=True)

train_meta = train_ds.metadata_frame()
val_meta = val_ds.metadata_frame()
eval_meta = eval_ds.metadata_frame()

train_meta.to_csv(REPORT_DIR / "train_block_metadata_p3.csv", index=False)
val_meta.to_csv(REPORT_DIR / "val_block_metadata_p3.csv", index=False)
eval_meta.to_csv(REPORT_DIR / "eval_block_metadata_p3.csv", index=False)

print("=" * 100)
print("DATASET SUMMARY")
print("=" * 100)
print(f"Train blocks: {len(train_ds):,}")
print(f"Val blocks  : {len(val_ds):,}")
print(f"Eval blocks : {len(eval_ds):,}")

print("\nTrain blocks by dialect:")
print(train_meta["dialect"].value_counts().sort_index().to_string())

print("\nVal blocks by dialect:")
print(val_meta["dialect"].value_counts().sort_index().to_string())

print("\nExternal eval blocks by dialect:")
print(eval_meta["dialect"].value_counts().sort_index().to_string())

print("\nTrain synthetic ratio by dialect:")
print(train_meta.groupby("dialect")["synthetic_ratio"].mean().round(4).to_string())

print("\n[OK] Packed datasets loaded.")

DATASET SUMMARY
Train blocks: 2,221
Val blocks  : 132
Eval blocks : 56

Train blocks by dialect:
dialect
BAR    145
CHI    299
KHU    130
KIS    185
MYM    158
NAR    172
NOA    155
NSD    186
RAJ    161
RAN    162
STD    142
SYL    169
TAN    157

Val blocks by dialect:
dialect
BAR    10
CHI    11
KHU     8
KIS    10
MYM    10
NAR    11
NOA    10
NSD    12
RAJ     9
RAN    11
STD     9
SYL    10
TAN    11

External eval blocks by dialect:
dialect
BAR     8
CHI    12
MYM    12
NOA    12
SYL    12

Train synthetic ratio by dialect:
dialect
BAR    0.1861
CHI    0.0000
KHU    0.9461
KIS    0.1273
MYM    0.4491
NAR    0.0000
NOA    0.5065
NSD    0.4041
RAJ    0.9541
RAN    0.1434
STD    0.5882
SYL    0.3006
TAN    0.3639

[OK] Packed datasets loaded.


In [16]:
# ============================================================
# CELL 5 — RARE-TOKEN STATISTICS AND SAMPLING WEIGHTS
# ============================================================

def compute_train_token_counts(dataset: PackedBlockDataset, cfg: P3TrainConfig) -> np.ndarray:
    ids = dataset.arrays["input_ids"].reshape(-1)
    labels = dataset.arrays["labels"].reshape(-1)

    # Count only valid non-pad label tokens.
    valid = labels[(labels >= 0) & (labels < cfg.vocab_size)]
    counts = np.bincount(valid, minlength=cfg.vocab_size).astype(np.int64)
    return counts


token_counts = compute_train_token_counts(train_ds, CFG)

rare_token_mask_np = np.zeros(CFG.vocab_size, dtype=np.bool_)
rare_token_mask_np[CFG.rare_token_min_id:] = token_counts[CFG.rare_token_min_id:] <= CFG.rare_token_max_count
rare_token_mask_np[:CFG.rare_token_min_id] = False

rare_token_ids = np.where(rare_token_mask_np)[0]

rare_report = {
    "rare_token_max_count": CFG.rare_token_max_count,
    "rare_token_min_id": CFG.rare_token_min_id,
    "num_rare_tokens": int(len(rare_token_ids)),
    "vocab_size": CFG.vocab_size,
    "rare_token_fraction": float(len(rare_token_ids) / CFG.vocab_size),
}

write_json(REPORT_DIR / "rare_token_report.json", rare_report)

print("Rare-token report:")
print(json.dumps(rare_report, indent=2))


def compute_block_loss_weight(meta: pd.DataFrame, cfg: P3TrainConfig) -> np.ndarray:
    weights = []

    for _, r in meta.iterrows():
        dialect = str(r["dialect"])
        syn = float(r["synthetic_ratio"])
        quality = float(r["mean_quality"])

        if dialect in cfg.low_real_synthetic_heavy_dialects:
            syn_w = cfg.synthetic_block_weight_low_real
        else:
            syn_w = cfg.synthetic_block_weight

        source_w = (1.0 - syn) * cfg.real_block_weight + syn * syn_w
        quality_w = float(np.clip(quality, cfg.min_quality_weight, cfg.max_quality_weight))

        w = source_w * quality_w
        w = float(np.clip(w, cfg.min_block_weight, cfg.max_block_weight))
        weights.append(w)

    return np.asarray(weights, dtype=np.float32)


def compute_sampler_weights(meta: pd.DataFrame, cfg: P3TrainConfig) -> np.ndarray:
    dialect_counts = meta["dialect"].value_counts().to_dict()

    raw_mass = {
        d: float(max(1, dialect_counts.get(d, 0)) ** cfg.dialect_sampling_alpha)
        for d in DIALECTS
    }
    mass_sum = sum(raw_mass.values())
    dialect_probs = {d: raw_mass[d] / mass_sum for d in DIALECTS}

    weights = []
    for _, r in meta.iterrows():
        d = str(r["dialect"])
        n_d = max(1, dialect_counts.get(d, 1))
        dialect_factor = dialect_probs[d] / n_d
        weights.append(dialect_factor)

    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.mean()

    return weights


train_block_loss_weight = compute_block_loss_weight(train_meta, CFG)
train_sampler_weight = compute_sampler_weights(train_meta, CFG)

train_meta["loss_weight"] = train_block_loss_weight
train_meta["sampler_weight"] = train_sampler_weight
train_meta.to_csv(REPORT_DIR / "train_block_weights_p3.csv", index=False)

print("\nLoss weight by dialect:")
print(train_meta.groupby("dialect")["loss_weight"].describe().round(4).to_string())

print("\nSampler weight by dialect:")
print(train_meta.groupby("dialect")["sampler_weight"].describe().round(4).to_string())

# Attach loss weights to dataset arrays for fast access.
train_ds.arrays["loss_weight"] = train_block_loss_weight.astype(np.float32)
val_ds.arrays["loss_weight"] = np.ones(len(val_ds), dtype=np.float32)
eval_ds.arrays["loss_weight"] = np.ones(len(eval_ds), dtype=np.float32)

print("\n[OK] Rare-token mask and block weights ready.")

Rare-token report:
{
  "rare_token_max_count": 5,
  "rare_token_min_id": 17,
  "num_rare_tokens": 19593,
  "vocab_size": 32000,
  "rare_token_fraction": 0.61228125
}

Loss weight by dialect:
         count    mean     std     min     25%     50%     75%     max
dialect                                                               
BAR      145.0  0.9151  0.0206  0.8660  0.8999  0.9170  0.9290  0.9773
CHI      299.0  0.9970  0.0027  0.9854  0.9956  0.9978  1.0000  1.0000
KHU      130.0  0.4716  0.0168  0.4392  0.4587  0.4702  0.4825  0.5124
KIS      185.0  0.9297  0.0227  0.8644  0.9162  0.9322  0.9439  0.9822
MYM      158.0  0.8428  0.0235  0.7620  0.8284  0.8425  0.8574  0.9011
NAR      172.0  0.9950  0.0038  0.9733  0.9933  0.9956  0.9979  1.0000
NOA      155.0  0.8227  0.0261  0.7433  0.8078  0.8250  0.8418  0.8714
NSD      186.0  0.8136  0.0317  0.7222  0.7933  0.8158  0.8316  0.9108
RAJ      161.0  0.4627  0.0169  0.4304  0.4495  0.4602  0.4750  0.5126
RAN      162.0  0.9246  0.02

In [17]:
# ============================================================
# CELL 6 — MUON OPTIMIZER + ADAMW HYBRID PARAMETER GROUPING
# ============================================================

@torch.no_grad()
def zeropower_via_newtonschulz5(g: torch.Tensor, steps: int = 5, eps: float = 1e-7) -> torch.Tensor:
    """
    Newton-Schulz orthogonalization used by Muon-style updates.
    Input must be 2D.
    """
    assert g.ndim == 2

    orig_dtype = g.dtype
    x = g.float()

    if x.size(0) > x.size(1):
        x = x.T
        transposed = True
    else:
        transposed = False

    x = x / (x.norm() + eps)

    # Coefficients from common Muon implementations.
    a, b, c = 3.4445, -4.7750, 2.0315

    for _ in range(steps):
        xx_t = x @ x.T
        x = a * x + b * (xx_t @ x) + c * (xx_t @ xx_t @ x)

    if transposed:
        x = x.T

    return x.to(orig_dtype)


class Muon(torch.optim.Optimizer):
    """
    Minimal Muon optimizer for 2D hidden matrices.
    Use AdamW for embeddings, norms, biases, output head, router-like small tensors.
    """
    def __init__(
        self,
        params,
        lr: float = 0.015,
        momentum: float = 0.95,
        weight_decay: float = 0.10,
        ns_steps: int = 5,
    ):
        defaults = dict(
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay,
            ns_steps=ns_steps,
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]
            weight_decay = group["weight_decay"]
            ns_steps = group["ns_steps"]

            for p in group["params"]:
                if p.grad is None:
                    continue

                g = p.grad
                if g.ndim != 2:
                    raise ValueError("Muon should only receive 2D parameters.")

                state = self.state[p]
                if len(state) == 0:
                    state["momentum_buffer"] = torch.zeros_like(p)

                buf = state["momentum_buffer"]
                buf.mul_(momentum).add_(g, alpha=1.0 - momentum)

                update = zeropower_via_newtonschulz5(buf, steps=ns_steps)

                # Decoupled weight decay.
                if weight_decay > 0:
                    p.mul_(1.0 - lr * weight_decay)

                # Width/height scale used in many Muon recipes.
                scale = math.sqrt(max(1.0, p.size(0) / max(1, p.size(1))))
                p.add_(update, alpha=-lr * scale)

        return loss


def build_optimizer_groups(model: nn.Module, cfg: P3TrainConfig):
    muon_params = []
    adamw_params = []

    muon_names = []
    adamw_names = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        # Avoid double-optimizing tied embedding/lm_head pointer.
        if "lm_head.weight" in name:
            continue

        use_muon_for_param = (
            cfg.use_muon
            and p.ndim == 2
            and "token_embedding" not in name
            and "embedding" not in name
            and "norm" not in name
        )

        if use_muon_for_param:
            muon_params.append(p)
            muon_names.append(name)
        else:
            adamw_params.append(p)
            adamw_names.append(name)

    return muon_params, adamw_params, muon_names, adamw_names


muon_params, adamw_params, muon_names, adamw_names = build_optimizer_groups(model, CFG)

print("=" * 100)
print("OPTIMIZER PARAMETER GROUPS")
print("=" * 100)
print(f"Muon params tensors : {len(muon_params)}")
print(f"AdamW params tensors: {len(adamw_params)}")
print(f"Muon param count    : {sum(p.numel() for p in muon_params):,}")
print(f"AdamW param count   : {sum(p.numel() for p in adamw_params):,}")

group_report = {
    "muon_names": muon_names,
    "adamw_names": adamw_names,
    "muon_param_count": int(sum(p.numel() for p in muon_params)),
    "adamw_param_count": int(sum(p.numel() for p in adamw_params)),
}

write_json(REPORT_DIR / "optimizer_parameter_groups.json", group_report)

muon_optimizer = Muon(
    muon_params,
    lr=CFG.muon_lr,
    momentum=CFG.muon_momentum,
    weight_decay=CFG.weight_decay,
) if len(muon_params) else None

try:
    adamw_optimizer = torch.optim.AdamW(
        adamw_params,
        lr=CFG.adamw_lr,
        betas=CFG.adamw_betas,
        eps=CFG.adamw_eps,
        weight_decay=CFG.weight_decay,
        fused=True if DEVICE.type == "cuda" else False,
    )
except TypeError:
    adamw_optimizer = torch.optim.AdamW(
        adamw_params,
        lr=CFG.adamw_lr,
        betas=CFG.adamw_betas,
        eps=CFG.adamw_eps,
        weight_decay=CFG.weight_decay,
    )

print("\n[OK] Optimizers created.")

OPTIMIZER PARAMETER GROUPS
Muon params tensors : 84
AdamW params tensors: 50
Muon param count    : 18,874,368
AdamW param count   : 12,299,136

[OK] Optimizers created.


In [18]:
# ============================================================
# CELL 7 — LOSS FUNCTIONS AND METRIC UTILITIES
# ============================================================

def get_amp_dtype(cfg: P3TrainConfig) -> Optional[torch.dtype]:
    if DEVICE.type != "cuda":
        return None
    if cfg.prefer_bf16 and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


AMP_DTYPE = get_amp_dtype(CFG)
USE_SCALER = AMP_DTYPE == torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=USE_SCALER)

print("AMP dtype :", AMP_DTYPE)
print("Use scaler:", USE_SCALER)


def cosine_lr(
    step: int,
    total_steps: int,
    warmup_steps: int,
    peak_lr: float,
    min_lr_ratio: float,
) -> float:
    if step < warmup_steps:
        return peak_lr * float(step + 1) / float(max(1, warmup_steps))

    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    min_lr = peak_lr * min_lr_ratio
    return min_lr + (peak_lr - min_lr) * cosine


def aux_future_weight_schedule(step: int, total_steps: int, warmup_steps: int, cfg: P3TrainConfig) -> float:
    if cfg.aux_future_final_weight <= 0:
        return 0.0

    if cfg.aux_future_start_after_warmup and step < warmup_steps:
        return 0.0

    ramp_steps = int(total_steps * cfg.aux_future_ramp_ratio)
    ramp_start = warmup_steps if cfg.aux_future_start_after_warmup else 0

    if step <= ramp_start:
        return 0.0

    progress = min(1.0, float(step - ramp_start) / float(max(1, ramp_steps)))
    return cfg.aux_future_final_weight * progress


def set_optimizer_lrs(
    step: int,
    total_steps: int,
    warmup_steps: int,
    cfg: P3TrainConfig,
) -> Dict[str, float]:
    muon_lr = cosine_lr(step, total_steps, warmup_steps, cfg.muon_lr, cfg.min_lr_ratio)
    adamw_lr = cosine_lr(step, total_steps, warmup_steps, cfg.adamw_lr, cfg.min_lr_ratio)

    if muon_optimizer is not None:
        for group in muon_optimizer.param_groups:
            group["lr"] = muon_lr

    for group in adamw_optimizer.param_groups:
        group["lr"] = adamw_lr

    return {"muon_lr": muon_lr, "adamw_lr": adamw_lr}


def move_batch_to_device(batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}


def block_source_loss_weight(batch: Dict[str, torch.Tensor], cfg: P3TrainConfig) -> torch.Tensor:
    """
    batch synthetic_ratio: [B]
    mean_quality: [B]
    dialect_id: [B]
    """
    syn = batch["synthetic_ratio"].float()       # [B]
    quality = batch["mean_quality"].float()      # [B]
    dialect_id = batch["dialect_id"].long()      # [B]

    weights = []

    for i in range(syn.numel()):
        d = ID_TO_DIALECT[int(dialect_id[i].detach().cpu())]
        if d in cfg.low_real_synthetic_heavy_dialects:
            syn_w = cfg.synthetic_block_weight_low_real
        else:
            syn_w = cfg.synthetic_block_weight

        source_w = (1.0 - syn[i]) * cfg.real_block_weight + syn[i] * syn_w
        quality_w = torch.clamp(quality[i], cfg.min_quality_weight, cfg.max_quality_weight)
        w = source_w * quality_w
        weights.append(w)

    w = torch.stack(weights).to(DEVICE)
    w = torch.clamp(w, cfg.min_block_weight, cfg.max_block_weight)
    return w


def compute_weighted_clm_loss(
    model: nn.Module,
    batch: Dict[str, torch.Tensor],
    aux_future_weight: float,
    cfg: P3TrainConfig,
    training: bool,
) -> Dict[str, torch.Tensor]:
    input_ids = batch["input_ids"]       # [B, T]
    labels = batch["labels"]             # [B, T]

    out = model(input_ids=input_ids, labels=None)
    logits = out["logits"]               # [B, T, V]

    block_weight = block_source_loss_weight(batch, cfg) if training else torch.ones(
        input_ids.size(0), device=DEVICE, dtype=torch.float32
    )                                    # [B]

    # Next-token loss.
    shift_logits = logits[:, :-1, :].contiguous()       # [B, T-1, V]
    shift_labels = labels[:, 1:].contiguous()           # [B, T-1]
    mask = shift_labels != -100                         # [B, T-1]

    ce = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
        reduction="none",
    ).view(shift_labels.shape)                           # [B, T-1]

    token_count = mask.sum(dim=1).clamp(min=1)           # [B]
    seq_loss = (ce * mask).sum(dim=1) / token_count      # [B]

    loss_ntp = (seq_loss * block_weight).sum() / block_weight.sum().clamp(min=1e-8)

    # Future-token auxiliary loss.
    if aux_future_weight > 0 and logits.size(1) > 2:
        future_logits = logits[:, :-2, :].contiguous()  # [B, T-2, V]
        future_labels = labels[:, 2:].contiguous()      # [B, T-2]
        future_mask = future_labels != -100             # [B, T-2]

        ce_future = F.cross_entropy(
            future_logits.view(-1, future_logits.size(-1)),
            future_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(future_labels.shape)

        future_token_count = future_mask.sum(dim=1).clamp(min=1)
        seq_future_loss = (ce_future * future_mask).sum(dim=1) / future_token_count
        loss_future = (seq_future_loss * block_weight).sum() / block_weight.sum().clamp(min=1e-8)
    else:
        loss_future = torch.zeros_like(loss_ntp)

    loss = loss_ntp + float(aux_future_weight) * loss_future

    with torch.no_grad():
        preds = shift_logits.argmax(dim=-1)              # [B, T-1]
        correct = ((preds == shift_labels) & mask).sum()
        total = mask.sum().clamp(min=1)
        token_acc = correct.float() / total.float()

        # Per-token NLL for eval aggregation.
        total_nll = (ce * mask).sum()
        total_tokens = mask.sum()

    return {
        "loss": loss,
        "loss_ntp": loss_ntp.detach(),
        "loss_future": loss_future.detach(),
        "token_acc": token_acc.detach(),
        "total_nll": total_nll.detach(),
        "total_tokens": total_tokens.detach(),
        "logits": logits,
    }


def compute_grad_norm(parameters: Iterable[torch.nn.Parameter]) -> float:
    total_sq = 0.0
    for p in parameters:
        if p.grad is None:
            continue
        grad = p.grad.detach()
        if not torch.isfinite(grad).all():
            return float("nan")
        total_sq += float(grad.float().norm(2).cpu().item() ** 2)
    return math.sqrt(total_sq)


def safe_ppl(loss_value: float) -> float:
    if loss_value > 30:
        return float("inf")
    return float(math.exp(loss_value))


print("[OK] Loss and metric utilities ready.")

AMP dtype : torch.bfloat16
Use scaler: False
[OK] Loss and metric utilities ready.


In [19]:
# ============================================================
# CELL 8 — EVALUATION FUNCTION
# ============================================================

rare_token_mask = torch.as_tensor(rare_token_mask_np, dtype=torch.bool, device=DEVICE)


def make_eval_loader(dataset: PackedBlockDataset, cfg: P3TrainConfig) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=cfg.eval_micro_batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=False,
    )


@torch.no_grad()
def evaluate_model(
    model: nn.Module,
    dataset: PackedBlockDataset,
    split_name: str,
    cfg: P3TrainConfig,
) -> Dict[str, Any]:
    model.eval()
    loader = make_eval_loader(dataset, cfg)

    global_nll = 0.0
    global_tokens = 0
    global_correct = 0
    global_pred_tokens = 0

    by_dialect = defaultdict(lambda: {"nll": 0.0, "tokens": 0, "correct": 0, "pred_tokens": 0})
    by_source_bin = defaultdict(lambda: {"nll": 0.0, "tokens": 0})

    rare_by_dialect = defaultdict(lambda: {"nll": 0.0, "tokens": 0})

    entropy_sum = 0.0
    entropy_tokens = 0

    start = time.time()

    for batch_cpu in loader:
        batch = move_batch_to_device(batch_cpu)

        if AMP_DTYPE is not None:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = model(input_ids=batch["input_ids"], labels=None)
                logits = out["logits"]  # [B, T, V]
        else:
            out = model(input_ids=batch["input_ids"], labels=None)
            logits = out["logits"]

        labels = batch["labels"]

        shift_logits = logits[:, :-1, :].contiguous()     # [B, T-1, V]
        shift_labels = labels[:, 1:].contiguous()         # [B, T-1]
        mask = shift_labels != -100                       # [B, T-1]

        ce = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
            reduction="none",
        ).view(shift_labels.shape)                        # [B, T-1]

        preds = shift_logits.argmax(dim=-1)               # [B, T-1]
        correct_mask = (preds == shift_labels) & mask

        batch_nll = float((ce * mask).sum().detach().cpu())
        batch_tokens = int(mask.sum().detach().cpu())

        global_nll += batch_nll
        global_tokens += batch_tokens
        global_correct += int(correct_mask.sum().detach().cpu())
        global_pred_tokens += batch_tokens

        # Entropy diagnostic.
        probs_log = F.log_softmax(shift_logits.float(), dim=-1)
        probs = probs_log.exp()
        entropy = -(probs * probs_log).sum(dim=-1)        # [B, T-1]
        entropy_sum += float((entropy * mask).sum().detach().cpu())
        entropy_tokens += batch_tokens

        # Rare-token loss.
        rare_mask_batch = rare_token_mask[shift_labels.clamp(min=0)] & mask

        for i in range(batch["input_ids"].size(0)):
            d_id = int(batch["dialect_id"][i].detach().cpu())
            dialect = ID_TO_DIALECT.get(d_id, f"UNK_{d_id}")

            row_mask = mask[i]
            row_nll = float((ce[i] * row_mask).sum().detach().cpu())
            row_tokens = int(row_mask.sum().detach().cpu())
            row_correct = int(correct_mask[i].sum().detach().cpu())

            by_dialect[dialect]["nll"] += row_nll
            by_dialect[dialect]["tokens"] += row_tokens
            by_dialect[dialect]["correct"] += row_correct
            by_dialect[dialect]["pred_tokens"] += row_tokens

            syn_ratio = float(batch["synthetic_ratio"][i].detach().cpu())
            if syn_ratio < 0.25:
                source_bin = "real_dominant"
            elif syn_ratio >= 0.75:
                source_bin = "synthetic_dominant"
            else:
                source_bin = "mixed"

            by_source_bin[source_bin]["nll"] += row_nll
            by_source_bin[source_bin]["tokens"] += row_tokens

            rare_row_mask = rare_mask_batch[i]
            rare_nll = float((ce[i] * rare_row_mask).sum().detach().cpu())
            rare_tokens = int(rare_row_mask.sum().detach().cpu())

            rare_by_dialect[dialect]["nll"] += rare_nll
            rare_by_dialect[dialect]["tokens"] += rare_tokens

    elapsed = max(1e-8, time.time() - start)

    global_loss = global_nll / max(1, global_tokens)
    global_ppl = safe_ppl(global_loss)
    global_acc = global_correct / max(1, global_pred_tokens)

    dialect_rows = []
    for d, m in sorted(by_dialect.items()):
        loss = m["nll"] / max(1, m["tokens"])
        dialect_rows.append({
            "split": split_name,
            "dialect": d,
            "tokens": int(m["tokens"]),
            "loss": float(loss),
            "ppl": safe_ppl(loss),
            "token_accuracy": float(m["correct"] / max(1, m["pred_tokens"])),
        })

    source_rows = []
    for s, m in sorted(by_source_bin.items()):
        loss = m["nll"] / max(1, m["tokens"])
        source_rows.append({
            "split": split_name,
            "source_bin": s,
            "tokens": int(m["tokens"]),
            "loss": float(loss),
            "ppl": safe_ppl(loss),
        })

    rare_rows = []
    for d, m in sorted(rare_by_dialect.items()):
        loss = m["nll"] / max(1, m["tokens"]) if m["tokens"] > 0 else float("nan")
        rare_rows.append({
            "split": split_name,
            "dialect": d,
            "rare_tokens": int(m["tokens"]),
            "rare_token_loss": float(loss) if m["tokens"] > 0 else float("nan"),
            "rare_token_ppl": safe_ppl(loss) if m["tokens"] > 0 else float("nan"),
        })

    result = {
        "split": split_name,
        "loss": float(global_loss),
        "ppl": float(global_ppl),
        "token_accuracy": float(global_acc),
        "tokens": int(global_tokens),
        "mean_entropy": float(entropy_sum / max(1, entropy_tokens)),
        "elapsed_sec": float(elapsed),
        "tokens_per_sec": float(global_tokens / elapsed),
        "by_dialect": dialect_rows,
        "by_source_bin": source_rows,
        "rare_by_dialect": rare_rows,
    }

    model.train()
    return result


print("[OK] Evaluation function ready.")

[OK] Evaluation function ready.


In [20]:
# ============================================================
# CELL 9 — TRAINING LOOP, CHECKPOINTING, RESUME SUPPORT
# ============================================================

def make_train_loader(dataset: PackedBlockDataset, meta: pd.DataFrame, cfg: P3TrainConfig) -> DataLoader:
    if cfg.use_weighted_sampler:
        weights = torch.as_tensor(meta["sampler_weight"].values, dtype=torch.double)
        sampler = WeightedRandomSampler(
            weights=weights,
            num_samples=len(dataset),
            replacement=cfg.sampler_replacement,
        )
        shuffle = False
    else:
        sampler = None
        shuffle = True

    return DataLoader(
        dataset,
        batch_size=cfg.micro_batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
        drop_last=True,
    )


train_loader = make_train_loader(train_ds, train_meta, CFG)

steps_per_epoch = math.ceil(len(train_loader) / CFG.grad_accum_steps)
total_steps = steps_per_epoch * CFG.epochs
warmup_steps = int(total_steps * CFG.warmup_ratio)

print("=" * 100)
print("TRAINING SCHEDULE")
print("=" * 100)
print(f"Train loader batches/epoch : {len(train_loader)}")
print(f"Gradient accumulation      : {CFG.grad_accum_steps}")
print(f"Optimizer steps/epoch      : {steps_per_epoch}")
print(f"Epochs                     : {CFG.epochs}")
print(f"Total optimizer steps      : {total_steps}")
print(f"Warmup steps               : {warmup_steps}")
print(f"Effective batch sequences  : {CFG.micro_batch_size * CFG.grad_accum_steps}")
print(f"Effective tokens/step      : {CFG.micro_batch_size * CFG.grad_accum_steps * CFG.block_size:,}")


def save_training_checkpoint(
    epoch: int,
    global_step: int,
    best_val_ppl: float,
    model: nn.Module,
    path: Path,
    is_best: bool = False,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    ckpt = {
        "epoch": int(epoch),
        "global_step": int(global_step),
        "best_val_ppl": float(best_val_ppl),
        "model_config": asdict(MODEL_CFG),
        "train_config": asdict(CFG),
        "model_state_dict": model.state_dict(),
    }

    if CFG.save_optimizer_state:
        ckpt["adamw_optimizer_state_dict"] = adamw_optimizer.state_dict()
        if muon_optimizer is not None:
            ckpt["muon_optimizer_state_dict"] = muon_optimizer.state_dict()
        ckpt["scaler_state_dict"] = scaler.state_dict() if scaler is not None else None

    torch.save(ckpt, path)

    if is_best:
        # Save model-only safetensors when available.
        try:
            from safetensors.torch import save_file
            state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            save_file(state, CHECKPOINT_DIR / "best_dense_32m.safetensors")
        except Exception as e:
            print("[WARN] Could not save safetensors:", e)


def load_last_checkpoint_if_available(model: nn.Module) -> Tuple[int, int, float]:
    last_path = CHECKPOINT_DIR / "last_checkpoint.pt"
    if not CFG.resume_from_last or not last_path.exists():
        return 0, 0, float("inf")

    ckpt = torch.load(last_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"], strict=True)

    if CFG.save_optimizer_state:
        if "adamw_optimizer_state_dict" in ckpt:
            adamw_optimizer.load_state_dict(ckpt["adamw_optimizer_state_dict"])
        if muon_optimizer is not None and "muon_optimizer_state_dict" in ckpt:
            muon_optimizer.load_state_dict(ckpt["muon_optimizer_state_dict"])
        if USE_SCALER and ckpt.get("scaler_state_dict") is not None:
            scaler.load_state_dict(ckpt["scaler_state_dict"])

    start_epoch = int(ckpt["epoch"])
    global_step = int(ckpt["global_step"])
    best_val_ppl = float(ckpt["best_val_ppl"])

    model.to(DEVICE)

    print(f"[RESUME] Loaded {last_path}")
    print(f"[RESUME] start_epoch={start_epoch}, global_step={global_step}, best_val_ppl={best_val_ppl:.4f}")

    return start_epoch, global_step, best_val_ppl


def train_one_epoch(
    model: nn.Module,
    epoch: int,
    global_step: int,
    total_steps: int,
    warmup_steps: int,
) -> Tuple[int, Dict[str, Any]]:
    model.train()

    epoch_loss_sum = 0.0
    epoch_ntp_sum = 0.0
    epoch_future_sum = 0.0
    epoch_acc_sum = 0.0
    epoch_batches = 0
    epoch_tokens = 0

    step_start_time = time.time()
    epoch_start_time = time.time()

    adamw_optimizer.zero_grad(set_to_none=True)
    if muon_optimizer is not None:
        muon_optimizer.zero_grad(set_to_none=True)

    for batch_idx, batch_cpu in enumerate(train_loader):
        batch = move_batch_to_device(batch_cpu)

        lr_info = set_optimizer_lrs(global_step, total_steps, warmup_steps, CFG)
        aux_w = aux_future_weight_schedule(global_step, total_steps, warmup_steps, CFG)

        if AMP_DTYPE is not None:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                loss_dict = compute_weighted_clm_loss(
                    model=model,
                    batch=batch,
                    aux_future_weight=aux_w,
                    cfg=CFG,
                    training=True,
                )
                loss = loss_dict["loss"] / CFG.grad_accum_steps
        else:
            loss_dict = compute_weighted_clm_loss(
                model=model,
                batch=batch,
                aux_future_weight=aux_w,
                cfg=CFG,
                training=True,
            )
            loss = loss_dict["loss"] / CFG.grad_accum_steps

        if not torch.isfinite(loss):
            msg = f"Non-finite loss at epoch={epoch}, batch={batch_idx}, global_step={global_step}"
            if CFG.abort_on_nonfinite_loss:
                raise FloatingPointError(msg)
            print("[WARN]", msg)
            continue

        if float(loss.detach().cpu()) * CFG.grad_accum_steps > CFG.max_reasonable_loss:
            print(
                f"[WARN] High loss at epoch={epoch}, batch={batch_idx}: "
                f"{float(loss.detach().cpu()) * CFG.grad_accum_steps:.4f}"
            )

        if USE_SCALER:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        epoch_loss_sum += float(loss_dict["loss"].detach().cpu())
        epoch_ntp_sum += float(loss_dict["loss_ntp"].detach().cpu())
        epoch_future_sum += float(loss_dict["loss_future"].detach().cpu())
        epoch_acc_sum += float(loss_dict["token_acc"].detach().cpu())
        epoch_batches += 1
        epoch_tokens += int(loss_dict["total_tokens"].detach().cpu())

        should_step = ((batch_idx + 1) % CFG.grad_accum_steps == 0)

        if should_step:
            if USE_SCALER:
                scaler.unscale_(adamw_optimizer)
                if muon_optimizer is not None:
                    scaler.unscale_(muon_optimizer)

            grad_norm = compute_grad_norm(model.parameters())
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip_norm)

            if USE_SCALER:
                scaler.step(adamw_optimizer)
                if muon_optimizer is not None:
                    scaler.step(muon_optimizer)
                scaler.update()
            else:
                adamw_optimizer.step()
                if muon_optimizer is not None:
                    muon_optimizer.step()

            adamw_optimizer.zero_grad(set_to_none=True)
            if muon_optimizer is not None:
                muon_optimizer.zero_grad(set_to_none=True)

            global_step += 1

            if DEVICE.type == "cuda":
                mem_alloc = torch.cuda.memory_allocated() / 1024**3
                mem_reserved = torch.cuda.memory_reserved() / 1024**3
                max_mem = torch.cuda.max_memory_allocated() / 1024**3
            else:
                mem_alloc = mem_reserved = max_mem = 0.0

            if global_step % CFG.log_every_steps == 0:
                elapsed = max(1e-8, time.time() - step_start_time)
                recent_tokens_per_sec = (CFG.micro_batch_size * CFG.grad_accum_steps * CFG.block_size) / elapsed
                step_start_time = time.time()

                print(
                    f"[E{epoch:03d} | step {global_step:05d}/{total_steps}] "
                    f"loss={float(loss_dict['loss'].detach().cpu()):.4f} "
                    f"ntp={float(loss_dict['loss_ntp'].detach().cpu()):.4f} "
                    f"future={float(loss_dict['loss_future'].detach().cpu()):.4f} "
                    f"acc={float(loss_dict['token_acc'].detach().cpu()):.4f} "
                    f"aux={aux_w:.4f} "
                    f"adamw_lr={lr_info['adamw_lr']:.2e} "
                    f"muon_lr={lr_info['muon_lr']:.2e} "
                    f"grad={grad_norm:.3f} "
                    f"tok/s={recent_tokens_per_sec:,.0f} "
                    f"mem={mem_alloc:.1f}/{mem_reserved:.1f}GB max={max_mem:.1f}GB"
                )

        del batch, loss, loss_dict

    epoch_elapsed = max(1e-8, time.time() - epoch_start_time)

    train_summary = {
        "epoch": int(epoch),
        "global_step": int(global_step),
        "train_loss": epoch_loss_sum / max(1, epoch_batches),
        "train_ntp_loss": epoch_ntp_sum / max(1, epoch_batches),
        "train_future_loss": epoch_future_sum / max(1, epoch_batches),
        "train_ppl": safe_ppl(epoch_loss_sum / max(1, epoch_batches)),
        "train_token_accuracy": epoch_acc_sum / max(1, epoch_batches),
        "train_batches": int(epoch_batches),
        "train_tokens": int(epoch_tokens),
        "epoch_elapsed_sec": float(epoch_elapsed),
        "train_tokens_per_sec": float(epoch_tokens / epoch_elapsed),
    }

    return global_step, train_summary


print("[OK] Training loop utilities ready.")

TRAINING SCHEDULE
Train loader batches/epoch : 92
Gradient accumulation      : 4
Optimizer steps/epoch      : 23
Epochs                     : 48
Total optimizer steps      : 1104
Warmup steps               : 55
Effective batch sequences  : 96
Effective tokens/step      : 49,152
[OK] Training loop utilities ready.


In [21]:
# ============================================================
# CELL 10 — RUN 48-EPOCH DENSE CLM TRAINING
# ============================================================

start_epoch, global_step, best_val_ppl = load_last_checkpoint_if_available(model)

history_rows = []
val_dialect_rows_all = []
eval_dialect_rows_all = []
source_rows_all = []
rare_rows_all = []

epochs_without_improvement = 0

training_start = time.time()

for epoch in range(start_epoch + 1, CFG.epochs + 1):
    print("\n" + "=" * 100)
    print(f"EPOCH {epoch}/{CFG.epochs}")
    print("=" * 100)

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    global_step, train_summary = train_one_epoch(
        model=model,
        epoch=epoch,
        global_step=global_step,
        total_steps=total_steps,
        warmup_steps=warmup_steps,
    )

    row = dict(train_summary)

    do_eval = (epoch % CFG.eval_every_epochs == 0) or (epoch == CFG.epochs)

    if do_eval:
        val_result = evaluate_model(model, val_ds, "val", CFG)
        eval_result = evaluate_model(model, eval_ds, "external_eval", CFG)

        row.update({
            "val_loss": val_result["loss"],
            "val_ppl": val_result["ppl"],
            "val_token_accuracy": val_result["token_accuracy"],
            "val_mean_entropy": val_result["mean_entropy"],
            "val_tokens_per_sec": val_result["tokens_per_sec"],
            "external_eval_loss": eval_result["loss"],
            "external_eval_ppl": eval_result["ppl"],
            "external_eval_token_accuracy": eval_result["token_accuracy"],
            "external_eval_mean_entropy": eval_result["mean_entropy"],
        })

        for r in val_result["by_dialect"]:
            rr = dict(r)
            rr["epoch"] = epoch
            val_dialect_rows_all.append(rr)

        for r in eval_result["by_dialect"]:
            rr = dict(r)
            rr["epoch"] = epoch
            eval_dialect_rows_all.append(rr)

        for r in val_result["by_source_bin"]:
            rr = dict(r)
            rr["epoch"] = epoch
            source_rows_all.append(rr)

        for r in eval_result["by_source_bin"]:
            rr = dict(r)
            rr["epoch"] = epoch
            source_rows_all.append(rr)

        for r in val_result["rare_by_dialect"]:
            rr = dict(r)
            rr["epoch"] = epoch
            rare_rows_all.append(rr)

        print("\nValidation:")
        print(
            f"val_loss={val_result['loss']:.4f} | "
            f"val_ppl={val_result['ppl']:.4f} | "
            f"val_acc={val_result['token_accuracy']:.4f} | "
            f"entropy={val_result['mean_entropy']:.4f}"
        )

        print("External holdout:")
        print(
            f"eval_loss={eval_result['loss']:.4f} | "
            f"eval_ppl={eval_result['ppl']:.4f} | "
            f"eval_acc={eval_result['token_accuracy']:.4f} | "
            f"entropy={eval_result['mean_entropy']:.4f}"
        )

        val_ppl = float(val_result["ppl"])

        improved = val_ppl < best_val_ppl
        if improved:
            best_val_ppl = val_ppl
            epochs_without_improvement = 0
            print(f"[BEST] New best val PPL: {best_val_ppl:.4f}")

            save_training_checkpoint(
                epoch=epoch,
                global_step=global_step,
                best_val_ppl=best_val_ppl,
                model=model,
                path=CHECKPOINT_DIR / "best_checkpoint.pt",
                is_best=True,
            )
        else:
            epochs_without_improvement += 1
            print(f"[NO IMPROVEMENT] epochs_without_improvement={epochs_without_improvement}")

    # Always save last checkpoint.
    save_training_checkpoint(
        epoch=epoch,
        global_step=global_step,
        best_val_ppl=best_val_ppl,
        model=model,
        path=CHECKPOINT_DIR / "last_checkpoint.pt",
        is_best=False,
    )

    if CFG.keep_epoch_checkpoints and (epoch % CFG.save_every_epochs == 0 or epoch == CFG.epochs):
        save_training_checkpoint(
            epoch=epoch,
            global_step=global_step,
            best_val_ppl=best_val_ppl,
            model=model,
            path=CHECKPOINT_DIR / f"epoch_{epoch:03d}_checkpoint.pt",
            is_best=False,
        )

    history_rows.append(row)

    # Save progressive logs every epoch.
    pd.DataFrame(history_rows).to_csv(REPORT_DIR / "train_log.csv", index=False)
    pd.DataFrame(val_dialect_rows_all).to_csv(REPORT_DIR / "val_perplexity_by_dialect_all_epochs.csv", index=False)
    pd.DataFrame(eval_dialect_rows_all).to_csv(REPORT_DIR / "external_eval_perplexity_by_dialect_all_epochs.csv", index=False)
    pd.DataFrame(source_rows_all).to_csv(REPORT_DIR / "perplexity_by_source_bin_all_epochs.csv", index=False)
    pd.DataFrame(rare_rows_all).to_csv(REPORT_DIR / "rare_token_loss_by_dialect_all_epochs.csv", index=False)

    if CFG.use_early_stopping and epochs_without_improvement >= CFG.early_stopping_patience:
        print(f"[EARLY STOP] No improvement for {CFG.early_stopping_patience} eval epochs.")
        break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

training_elapsed = time.time() - training_start

print("\n" + "=" * 100)
print("TRAINING COMPLETE")
print("=" * 100)
print(f"Epochs completed : {len(history_rows)}")
print(f"Global steps     : {global_step}")
print(f"Best val PPL     : {best_val_ppl:.4f}")
print(f"Elapsed          : {training_elapsed / 3600:.2f} hours")
print("Best checkpoint  :", CHECKPOINT_DIR / "best_checkpoint.pt")
print("Best safetensors :", CHECKPOINT_DIR / "best_dense_32m.safetensors")


EPOCH 1/48
[E001 | step 00010/1104] loss=9.7694 ntp=9.7694 future=0.0000 acc=0.1126 aux=0.0000 adamw_lr=5.45e-05 muon_lr=2.73e-03 grad=3.750 tok/s=20,276 mem=1.4/10.9GB max=8.8GB
[E001 | step 00020/1104] loss=8.8997 ntp=8.8997 future=0.0000 acc=0.1879 aux=0.0000 adamw_lr=1.09e-04 muon_lr=5.45e-03 grad=2.742 tok/s=22,572 mem=1.4/10.9GB max=8.8GB

Validation:
val_loss=8.3156 | val_ppl=4086.9705 | val_acc=0.2870 | entropy=10.2436
External holdout:
eval_loss=8.3334 | eval_ppl=4160.5100 | eval_acc=0.2962 | entropy=10.2361
[BEST] New best val PPL: 4086.9705

EPOCH 2/48
[E002 | step 00030/1104] loss=7.6437 ntp=7.6437 future=0.0000 acc=0.2992 aux=0.0000 adamw_lr=1.64e-04 muon_lr=8.18e-03 grad=2.653 tok/s=31,438 mem=1.4/11.1GB max=8.8GB
[E002 | step 00040/1104] loss=6.6708 ntp=6.6708 future=0.0000 acc=0.3288 aux=0.0000 adamw_lr=2.18e-04 muon_lr=1.09e-02 grad=2.428 tok/s=22,583 mem=1.4/11.1GB max=8.8GB

Validation:
val_loss=6.3832 | val_ppl=591.8088 | val_acc=0.3207 | entropy=9.0642
External ho

In [22]:
# ============================================================
# CELL 11 — EXPORT FINAL REPORTS AND PLOTS
# ============================================================

train_log = pd.DataFrame(history_rows)
val_dialect_df = pd.DataFrame(val_dialect_rows_all)
eval_dialect_df = pd.DataFrame(eval_dialect_rows_all)
source_bin_df = pd.DataFrame(source_rows_all)
rare_df = pd.DataFrame(rare_rows_all)

train_log.to_csv(REPORT_DIR / "train_log.csv", index=False)
val_dialect_df.to_csv(REPORT_DIR / "val_perplexity_by_dialect_all_epochs.csv", index=False)
eval_dialect_df.to_csv(REPORT_DIR / "external_eval_perplexity_by_dialect_all_epochs.csv", index=False)
source_bin_df.to_csv(REPORT_DIR / "perplexity_by_source_bin_all_epochs.csv", index=False)
rare_df.to_csv(REPORT_DIR / "rare_token_loss_by_dialect_all_epochs.csv", index=False)

# Final epoch slices.
if len(val_dialect_df):
    final_val_epoch = int(val_dialect_df["epoch"].max())
    final_val = val_dialect_df[val_dialect_df["epoch"] == final_val_epoch].copy()
    final_val.to_csv(REPORT_DIR / "val_perplexity_by_dialect_final.csv", index=False)

if len(eval_dialect_df):
    final_eval_epoch = int(eval_dialect_df["epoch"].max())
    final_eval = eval_dialect_df[eval_dialect_df["epoch"] == final_eval_epoch].copy()
    final_eval.to_csv(REPORT_DIR / "external_eval_perplexity_by_dialect_final.csv", index=False)

if len(source_bin_df):
    final_source = source_bin_df[source_bin_df["epoch"] == source_bin_df["epoch"].max()].copy()
    final_source.to_csv(REPORT_DIR / "perplexity_by_source_bin_final.csv", index=False)

if len(rare_df):
    final_rare = rare_df[rare_df["epoch"] == rare_df["epoch"].max()].copy()
    final_rare.to_csv(REPORT_DIR / "rare_token_loss_by_dialect_final.csv", index=False)


# -------------------------
# Plot 1: loss curves
# -------------------------
plt.figure(figsize=(10, 6))
plt.plot(train_log["epoch"], train_log["train_loss"], label="train_loss")
if "val_loss" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["val_loss"], label="val_loss")
if "external_eval_loss" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["external_eval_loss"], label="external_eval_loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("BLEACH-Dense-32M CLM Loss")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "dense_training_loss_curves.png", dpi=200)
plt.close()


# -------------------------
# Plot 2: PPL curves
# -------------------------
plt.figure(figsize=(10, 6))
plt.plot(train_log["epoch"], train_log["train_ppl"], label="train_ppl")
if "val_ppl" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["val_ppl"], label="val_ppl")
if "external_eval_ppl" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["external_eval_ppl"], label="external_eval_ppl")
plt.xlabel("Epoch")
plt.ylabel("Perplexity")
plt.title("BLEACH-Dense-32M Perplexity")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "dense_ppl_curves.png", dpi=200)
plt.close()


# -------------------------
# Plot 3: token accuracy
# -------------------------
plt.figure(figsize=(10, 6))
plt.plot(train_log["epoch"], train_log["train_token_accuracy"], label="train_token_accuracy")
if "val_token_accuracy" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["val_token_accuracy"], label="val_token_accuracy")
if "external_eval_token_accuracy" in train_log.columns:
    plt.plot(train_log["epoch"], train_log["external_eval_token_accuracy"], label="external_eval_token_accuracy")
plt.xlabel("Epoch")
plt.ylabel("Token Accuracy")
plt.title("BLEACH-Dense-32M Token Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "dense_token_accuracy_curves.png", dpi=200)
plt.close()


# -------------------------
# Plot 4: final val PPL by dialect
# -------------------------
if len(val_dialect_df):
    final_val = val_dialect_df[val_dialect_df["epoch"] == val_dialect_df["epoch"].max()].copy()
    final_val = final_val.sort_values("dialect")

    plt.figure(figsize=(12, 6))
    plt.bar(final_val["dialect"], final_val["ppl"])
    plt.xlabel("Dialect")
    plt.ylabel("Validation PPL")
    plt.title("Final Validation PPL by Dialect")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_val_ppl_by_dialect.png", dpi=200)
    plt.close()


# -------------------------
# Plot 5: external eval PPL by dialect
# -------------------------
if len(eval_dialect_df):
    final_eval = eval_dialect_df[eval_dialect_df["epoch"] == eval_dialect_df["epoch"].max()].copy()
    final_eval = final_eval.sort_values("dialect")

    plt.figure(figsize=(12, 6))
    plt.bar(final_eval["dialect"], final_eval["ppl"])
    plt.xlabel("Dialect")
    plt.ylabel("External Holdout PPL")
    plt.title("Final External Holdout PPL by Dialect")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_external_eval_ppl_by_dialect.png", dpi=200)
    plt.close()


# -------------------------
# Plot 6: real/synthetic source-bin PPL
# -------------------------
if len(source_bin_df):
    final_source = source_bin_df[source_bin_df["epoch"] == source_bin_df["epoch"].max()].copy()

    plt.figure(figsize=(8, 5))
    plt.bar(final_source["split"] + "::" + final_source["source_bin"], final_source["ppl"])
    plt.xlabel("Split / Source Bin")
    plt.ylabel("PPL")
    plt.title("Final PPL by Source Composition")
    plt.xticks(rotation=35, ha="right")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_ppl_by_source_bin.png", dpi=200)
    plt.close()


# -------------------------
# Plot 7: rare-token PPL by dialect
# -------------------------
if len(rare_df):
    final_rare = rare_df[rare_df["epoch"] == rare_df["epoch"].max()].copy()
    final_rare = final_rare.sort_values("dialect")

    plt.figure(figsize=(12, 6))
    plt.bar(final_rare["dialect"], final_rare["rare_token_ppl"])
    plt.xlabel("Dialect")
    plt.ylabel("Rare-token PPL")
    plt.title("Final Rare-token PPL by Dialect")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "final_rare_token_ppl_by_dialect.png", dpi=200)
    plt.close()


summary = {
    "portion": "bleach_p3_dense_clm_training",
    "status": "complete",
    "model_name": CFG.model_name,
    "epochs_requested": CFG.epochs,
    "epochs_completed": int(len(train_log)),
    "global_steps": int(global_step),
    "best_val_ppl": float(best_val_ppl),
    "final_train_loss": float(train_log["train_loss"].iloc[-1]),
    "final_train_ppl": float(train_log["train_ppl"].iloc[-1]),
    "final_val_ppl": float(train_log["val_ppl"].dropna().iloc[-1]) if "val_ppl" in train_log.columns and train_log["val_ppl"].notna().any() else None,
    "final_external_eval_ppl": float(train_log["external_eval_ppl"].dropna().iloc[-1]) if "external_eval_ppl" in train_log.columns and train_log["external_eval_ppl"].notna().any() else None,
    "training_elapsed_hours": float(training_elapsed / 3600),
    "best_checkpoint": str(CHECKPOINT_DIR / "best_checkpoint.pt"),
    "best_safetensors": str(CHECKPOINT_DIR / "best_dense_32m.safetensors"),
    "reports": {
        "train_log": str(REPORT_DIR / "train_log.csv"),
        "val_perplexity_by_dialect_final": str(REPORT_DIR / "val_perplexity_by_dialect_final.csv"),
        "external_eval_perplexity_by_dialect_final": str(REPORT_DIR / "external_eval_perplexity_by_dialect_final.csv"),
        "source_bin_final": str(REPORT_DIR / "perplexity_by_source_bin_final.csv"),
        "rare_token_final": str(REPORT_DIR / "rare_token_loss_by_dialect_final.csv"),
    },
    "plots": {
        "loss_curves": str(PLOT_DIR / "dense_training_loss_curves.png"),
        "ppl_curves": str(PLOT_DIR / "dense_ppl_curves.png"),
        "token_accuracy": str(PLOT_DIR / "dense_token_accuracy_curves.png"),
        "val_ppl_by_dialect": str(PLOT_DIR / "final_val_ppl_by_dialect.png"),
        "external_ppl_by_dialect": str(PLOT_DIR / "final_external_eval_ppl_by_dialect.png"),
        "source_bin_ppl": str(PLOT_DIR / "final_ppl_by_source_bin.png"),
        "rare_token_ppl": str(PLOT_DIR / "final_rare_token_ppl_by_dialect.png"),
    },
}

write_json(REPORT_DIR / "p3_training_summary.json", summary)

print("=" * 100)
print("EXPORTED REPORTS AND PLOTS")
print("=" * 100)
print(json.dumps(summary, indent=2))

EXPORTED REPORTS AND PLOTS
{
  "portion": "bleach_p3_dense_clm_training",
  "status": "complete",
  "model_name": "BLEACH-Dense-32M",
  "epochs_requested": 48,
  "epochs_completed": 48,
  "global_steps": 1104,
  "best_val_ppl": 46.740541908799166,
  "final_train_loss": 2.109935928945956,
  "final_train_ppl": 8.24771282804311,
  "final_val_ppl": 56.88155348885377,
  "final_external_eval_ppl": 13.785883640005368,
  "training_elapsed_hours": 0.09171940591600206,
  "best_checkpoint": "/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_checkpoint.pt",
  "best_safetensors": "/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_dense_32m.safetensors",
  "reports": {
    "train_log": "/kaggle/working/artifacts/bleach_p3_dense_clm/reports/train_log.csv",
    "val_perplexity_by_dialect_final": "/kaggle/working/artifacts/bleach_p3_dense_clm/reports/val_perplexity_by_dialect_final.csv",
    "external_eval_perplexity_by_dialect_final": "/kaggle/working/artifacts/bleach_p3_dens

In [23]:
# ============================================================
# CELL 12 — FINAL CHECKPOINT SUMMARY
# ============================================================

print("=" * 100)
print("BLEACH PORTION 3 FINAL CHECKPOINT")
print("=" * 100)

print(f"Model                 : {CFG.model_name}")
print(f"Epochs completed      : {len(train_log)} / {CFG.epochs}")
print(f"Global optimizer steps: {global_step}")
print(f"Best validation PPL   : {best_val_ppl:.4f}")
print(f"Final train loss      : {train_log['train_loss'].iloc[-1]:.4f}")
print(f"Final train PPL       : {train_log['train_ppl'].iloc[-1]:.4f}")

if "val_ppl" in train_log.columns and train_log["val_ppl"].notna().any():
    print(f"Final val PPL         : {train_log['val_ppl'].dropna().iloc[-1]:.4f}")

if "external_eval_ppl" in train_log.columns and train_log["external_eval_ppl"].notna().any():
    print(f"Final external PPL    : {train_log['external_eval_ppl'].dropna().iloc[-1]:.4f}")

print("\nKey outputs:")
print("Best checkpoint       :", CHECKPOINT_DIR / "best_checkpoint.pt")
print("Best safetensors      :", CHECKPOINT_DIR / "best_dense_32m.safetensors")
print("Last checkpoint       :", CHECKPOINT_DIR / "last_checkpoint.pt")
print("Train log             :", REPORT_DIR / "train_log.csv")
print("P3 summary            :", REPORT_DIR / "p3_training_summary.json")
print("Plots                 :", PLOT_DIR)

print("\nPaste these after training:")
print("1. reports/p3_training_summary.json")
print("2. reports/train_log.csv final 5 rows")
print("3. reports/val_perplexity_by_dialect_final.csv")
print("4. reports/external_eval_perplexity_by_dialect_final.csv")

print("\n✅ BLEACH SLM/LLM PORTION 3 COMPLETE")
print("Next: Portion 4 — Dense downstream tasks: dialect classification ")

BLEACH PORTION 3 FINAL CHECKPOINT
Model                 : BLEACH-Dense-32M
Epochs completed      : 48 / 48
Global optimizer steps: 1104
Best validation PPL   : 46.7405
Final train loss      : 2.1099
Final train PPL       : 8.2477
Final val PPL         : 56.8816
Final external PPL    : 13.7859

Key outputs:
Best checkpoint       : /kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_checkpoint.pt
Best safetensors      : /kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_dense_32m.safetensors
Last checkpoint       : /kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/last_checkpoint.pt
Train log             : /kaggle/working/artifacts/bleach_p3_dense_clm/reports/train_log.csv
P3 summary            : /kaggle/working/artifacts/bleach_p3_dense_clm/reports/p3_training_summary.json
Plots                 : /kaggle/working/artifacts/bleach_p3_dense_clm/plots

Paste these after training:
1. reports/p3_training_summary.json
2. reports/train_log.csv final 5 rows
3. repo

##  Portion 4 — Dense downstream tasks: dialect classification

In [24]:
# ============================================================
# PORTION 4.0 — SETUP + ARTIFACT RESOLVER
# Classification-only final dense downstream module
# ============================================================

from __future__ import annotations

import os
import re
import gc
import sys
import json
import math
import time
import random
import hashlib
import warnings
import unicodedata
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Optional, Tuple
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

warnings.filterwarnings("ignore")

try:
    from tokenizers import Tokenizer
except Exception as e:
    raise ImportError(
        "Missing `tokenizers`. Install it from your offline wheelhouse before running Portion 4."
    ) from e

try:
    from safetensors.torch import load_file as safe_load_file
    from safetensors.torch import save_file as safe_save_file
    SAFETENSORS_AVAILABLE = True
except Exception:
    SAFETENSORS_AVAILABLE = False

try:
    from sklearn.metrics import (
        accuracy_score,
        f1_score,
        balanced_accuracy_score,
        matthews_corrcoef,
        classification_report,
        confusion_matrix,
    )
    from sklearn.pipeline import Pipeline, FeatureUnion
    from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.svm import LinearSVC
    from sklearn.preprocessing import StandardScaler
    from sklearn.decomposition import PCA
    SKLEARN_AVAILABLE = True
except Exception as e:
    SKLEARN_AVAILABLE = False
    raise ImportError(
        "Missing scikit-learn. Install it from your offline wheelhouse before running Portion 4."
    ) from e


SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


DIALECTS = ["BAR", "CHI", "KHU", "KIS", "MYM", "NAR", "NOA", "NSD", "RAJ", "RAN", "STD", "SYL", "TAN"]
DIALECT_TO_ID = {d: i for i, d in enumerate(DIALECTS)}
ID_TO_DIALECT = {i: d for d, i in DIALECT_TO_ID.items()}


@dataclass(frozen=True)
class P4Config:
    output_dir: str = "/kaggle/working/artifacts/bleach_p4_final_classification"

    # Token/model.
    vocab_size: int = 32000
    max_length: int = 256
    pad_token_id: int = 0
    unk_token_id: int = 1
    bos_token_id: int = 2
    eos_token_id: int = 3

    # Classical baselines.
    tfidf_char_max_features: int = 250000
    tfidf_word_max_features: int = 100000
    tfidf_min_df: int = 2

    # Neural classifier.
    epochs: int = 24
    batch_size: int = 64
    eval_batch_size: int = 128
    num_workers: int = 2

    # Staged SFT.
    head_only_epochs: int = 3
    top4_epochs: int = 9
    top6_epochs: int = 12

    # LRs.
    lr_head: float = 1.0e-3
    lr_pooler: float = 7.5e-4
    lr_top_layers: float = 1.5e-5
    lr_lower_top_layers: float = 7.5e-6

    # Regularization.
    weight_decay: float = 0.05
    dropout: float = 0.25
    grad_clip_norm: float = 1.0
    label_smoothing: float = 0.03
    use_rdrop: bool = True
    rdrop_start_epoch: int = 5
    rdrop_alpha: float = 0.10

    # Sampler/loss balancing.
    use_balanced_sampler: bool = True
    sampler_alpha: float = 0.35
    class_weight_power: float = 0.50

    # Schedule.
    warmup_ratio: float = 0.06
    min_lr_ratio: float = 0.10
    patience: int = 6

    # Precision.
    prefer_bf16: bool = True

    # Logging.
    log_every_steps: int = 150

    # Acceptance targets.
    target_val_acc_min: float = 0.72
    target_val_macro_f1_min: float = 0.72
    target_external5_acc_min: float = 0.35
    target_external5_macro_f1_min: float = 0.30


CFG = P4Config()

OUT_DIR = Path(CFG.output_dir)
CONFIG_DIR = OUT_DIR / "config"
REPORT_DIR = OUT_DIR / "reports"
PLOT_DIR = OUT_DIR / "plots"
CKPT_DIR = OUT_DIR / "checkpoints"
PRED_DIR = OUT_DIR / "predictions"

for p in [OUT_DIR, CONFIG_DIR, REPORT_DIR, PLOT_DIR, CKPT_DIR, PRED_DIR]:
    p.mkdir(parents=True, exist_ok=True)


def read_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def resolve_slm_p1_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/slm_p1_data"),
        Path("artifacts/slm_p1_data"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("slm_p1_data/splits/model_train_rows.parquet")):
            candidates.append(m.parents[1])
        for m in sorted(input_root.rglob("artifacts/slm_p1_data/splits/model_train_rows.parquet")):
            candidates.append(m.parents[2])

    for c in candidates:
        if (c / "splits" / "model_train_rows.parquet").exists():
            return c

    raise FileNotFoundError("Could not resolve SLM Portion-1 root.")


def resolve_p2_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/bleach_p2_dense_arch"),
        Path("artifacts/bleach_p2_dense_arch"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("bleach_p2_dense_arch/source/bleach_dense_model.py")):
            candidates.append(m.parents[1])
        for m in sorted(input_root.rglob("artifacts/bleach_p2_dense_arch/source/bleach_dense_model.py")):
            candidates.append(m.parents[1])

    for c in candidates:
        if (c / "source" / "bleach_dense_model.py").exists():
            return c

    raise FileNotFoundError("Could not resolve Portion-2 dense architecture root.")


def resolve_tokenizer_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/tokenizer_p5_freeze/final_tokenizer_frozen"),
        Path("artifacts/tokenizer_p5_freeze/final_tokenizer_frozen"),
        Path("/kaggle/working/artifacts/tokenizer_p4_wordpiece/final_tokenizer"),
        Path("artifacts/tokenizer_p4_wordpiece/final_tokenizer"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("final_tokenizer_frozen/tokenizer.json")):
            candidates.append(m.parent)
        for m in sorted(input_root.rglob("final_tokenizer/tokenizer.json")):
            candidates.append(m.parent)

    for c in candidates:
        if (c / "tokenizer.json").exists():
            return c

    raise FileNotFoundError("Could not resolve frozen/final tokenizer root.")


def resolve_selected_dense_checkpoint() -> Dict[str, Any]:
    """
    This resolves only dense LM checkpoints from P3B/P3.
    It does not assume or use any previous classification checkpoint.
    """
    p3b_safe = Path("/kaggle/working/artifacts/bleach_p3b_dense_refinement/checkpoints/best_dense_32m_p3b.safetensors")
    p3b_pt = Path("/kaggle/working/artifacts/bleach_p3b_dense_refinement/checkpoints/best_checkpoint_p3b.pt")
    p3b_summary = Path("/kaggle/working/artifacts/bleach_p3b_dense_refinement/reports/p3b_refinement_summary.json")

    p3_safe = Path("/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_dense_32m.safetensors")
    p3_pt = Path("/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_checkpoint.pt")

    if p3b_safe.exists() or p3b_pt.exists():
        return {
            "stage": "p3b",
            "safetensors": str(p3b_safe) if p3b_safe.exists() else None,
            "checkpoint_pt": str(p3b_pt) if p3b_pt.exists() else None,
            "summary": str(p3b_summary) if p3b_summary.exists() else None,
        }

    if p3_safe.exists() or p3_pt.exists():
        return {
            "stage": "p3",
            "safetensors": str(p3_safe) if p3_safe.exists() else None,
            "checkpoint_pt": str(p3_pt) if p3_pt.exists() else None,
            "summary": None,
        }

    raise FileNotFoundError(
        "Could not find a selected dense LM checkpoint. Expected P3B or P3 best checkpoint."
    )


SLM_P1_ROOT = resolve_slm_p1_root()
P2_ROOT = resolve_p2_root()
TOKENIZER_ROOT = resolve_tokenizer_root()
SELECTED_DENSE = resolve_selected_dense_checkpoint()

write_json(CONFIG_DIR / "p4_config.json", asdict(CFG))
write_json(CONFIG_DIR / "p4_resolved_paths.json", {
    "slm_p1_root": str(SLM_P1_ROOT),
    "p2_root": str(P2_ROOT),
    "tokenizer_root": str(TOKENIZER_ROOT),
    "selected_dense_checkpoint": SELECTED_DENSE,
})

print("=" * 100)
print("BLEACH PORTION 4 — FINAL CLASSIFICATION ONLY")
print("=" * 100)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16:", torch.cuda.is_bf16_supported())
print("SLM P1 root:", SLM_P1_ROOT)
print("P2 root:", P2_ROOT)
print("Tokenizer root:", TOKENIZER_ROOT)
print("Selected dense checkpoint:", json.dumps(SELECTED_DENSE, indent=2))
print("=" * 100)

BLEACH PORTION 4 — FINAL CLASSIFICATION ONLY
Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
bf16: True
SLM P1 root: /kaggle/input/notebooks/nabidnur/bleach-slm/artifacts/slm_p1_data
P2 root: /kaggle/working/artifacts/bleach_p2_dense_arch
Tokenizer root: /kaggle/input/datasets/nabidnur/tokenizer-best/artifacts/tokenizer_p5_freeze/final_tokenizer_frozen
Selected dense checkpoint: {
  "stage": "p3",
  "safetensors": "/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_dense_32m.safetensors",
  "checkpoint_pt": "/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_checkpoint.pt",
  "summary": null
}


In [25]:
# ============================================================
# PORTION 4.1 — LOAD ROW-LEVEL DATA + PROTOCOL AUDIT
# ============================================================

DIALECT_TAG_RE = re.compile(r"<dial:[A-Za-z0-9_\-]+>")
BENGALI_RE = re.compile(r"[\u0980-\u09FF]")


def normalize_text_for_cls(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x)
    s = unicodedata.normalize("NFC", s)
    s = DIALECT_TAG_RE.sub(" ", s)
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", s)
    s = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def has_bangla(s: str) -> bool:
    return bool(BENGALI_RE.search(str(s)))


def select_text_column(df: pd.DataFrame) -> str:
    preferred = [
        "text",
        "body_text",
        "clean_text",
        "normalized_text",
        "sentence",
        "utterance",
        "content",
    ]
    for c in preferred:
        if c in df.columns:
            return c

    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        raise ValueError("No text-like column found.")
    return max(obj_cols, key=lambda c: df[c].dropna().astype(str).map(len).mean())


def infer_is_synthetic(df: pd.DataFrame) -> pd.Series:
    if "is_synthetic" in df.columns:
        return df["is_synthetic"].astype(str).str.lower().isin(["true", "1", "yes", "synthetic"])

    for c in ["source_type", "source", "source_file", "dataset_source"]:
        if c in df.columns:
            return df[c].astype(str).str.lower().str.contains("synthetic")

    return pd.Series([False] * len(df), index=df.index)


def load_split_rows(path: Path, split_name: str) -> pd.DataFrame:
    assert path.exists(), f"Missing split parquet: {path}"

    df = pd.read_parquet(path)
    assert "dialect" in df.columns, f"{split_name} missing dialect column."

    text_col = select_text_column(df)

    df = df.copy()
    df["p4_text_raw"] = df[text_col].astype(str)
    df["p4_text"] = df["p4_text_raw"].map(normalize_text_for_cls)
    df["p4_len"] = df["p4_text"].str.len()
    df["p4_has_bangla"] = df["p4_text"].map(has_bangla)
    df["p4_split"] = split_name
    df["is_synthetic_inferred"] = infer_is_synthetic(df)

    if "quality_score_cleaned" in df.columns:
        df["quality_score_p4"] = pd.to_numeric(df["quality_score_cleaned"], errors="coerce").fillna(1.0)
    elif "quality_score" in df.columns:
        df["quality_score_p4"] = pd.to_numeric(df["quality_score"], errors="coerce").fillna(1.0)
    else:
        df["quality_score_p4"] = 1.0

    leaked = int(df["p4_text"].str.contains(r"<dial:", regex=True).sum())
    assert leaked == 0, f"{split_name} still contains dialect tags after cleaning: {leaked}"

    before = len(df)
    df = df[df["dialect"].isin(DIALECTS)].copy()
    df = df[df["p4_len"] > 0].copy()
    df = df[df["p4_has_bangla"]].copy()
    df = df.drop_duplicates(subset=["p4_text", "dialect"]).reset_index(drop=True)

    print(f"{split_name}: text_col={text_col} | rows {before:,} -> {len(df):,}")
    return df


train_df = load_split_rows(SLM_P1_ROOT / "splits" / "model_train_rows.parquet", "train")
val_df = load_split_rows(SLM_P1_ROOT / "splits" / "model_val_balanced_rows.parquet", "val")
external_df = load_split_rows(SLM_P1_ROOT / "splits" / "model_eval_holdout_rows.parquet", "external")

external_present_dialects = sorted(external_df["dialect"].unique().tolist())
external_present_ids = [DIALECT_TO_ID[d] for d in external_present_dialects]

protocol_audit = {
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "external_rows": int(len(external_df)),
    "train_dialects": sorted(train_df["dialect"].unique().tolist()),
    "val_dialects": sorted(val_df["dialect"].unique().tolist()),
    "external_present_dialects": external_present_dialects,
    "external_present_ids": external_present_ids,
    "train_counts": train_df["dialect"].value_counts().sort_index().to_dict(),
    "val_counts": val_df["dialect"].value_counts().sort_index().to_dict(),
    "external_counts": external_df["dialect"].value_counts().sort_index().to_dict(),
    "synthetic_ratio_train_by_dialect": train_df.groupby("dialect")["is_synthetic_inferred"].mean().round(4).to_dict(),
    "synthetic_ratio_val_by_dialect": val_df.groupby("dialect")["is_synthetic_inferred"].mean().round(4).to_dict(),
    "synthetic_ratio_external_by_dialect": external_df.groupby("dialect")["is_synthetic_inferred"].mean().round(4).to_dict(),
    "evaluation_protocol": {
        "validation": "13-way",
        "external": "report both 13-way and restricted present-dialect evaluation",
        "classification_input_removes_dialect_tags": True,
    },
}

write_json(REPORT_DIR / "p4_protocol_audit.json", protocol_audit)

pd.concat([
    train_df.assign(split="train"),
    val_df.assign(split="val"),
    external_df.assign(split="external"),
], axis=0).groupby(["split", "dialect", "is_synthetic_inferred"]).size().reset_index(name="rows").to_csv(
    REPORT_DIR / "p4_split_dialect_source_counts.csv",
    index=False,
)

print(json.dumps(protocol_audit, indent=2, ensure_ascii=False))

train: text_col=text | rows 113,594 -> 113,594
val: text_col=text | rows 6,500 -> 6,500
external: text_col=text | rows 2,875 -> 2,875
{
  "train_rows": 113594,
  "val_rows": 6500,
  "external_rows": 2875,
  "train_dialects": [
    "BAR",
    "CHI",
    "KHU",
    "KIS",
    "MYM",
    "NAR",
    "NOA",
    "NSD",
    "RAJ",
    "RAN",
    "STD",
    "SYL",
    "TAN"
  ],
  "val_dialects": [
    "BAR",
    "CHI",
    "KHU",
    "KIS",
    "MYM",
    "NAR",
    "NOA",
    "NSD",
    "RAJ",
    "RAN",
    "STD",
    "SYL",
    "TAN"
  ],
  "external_present_dialects": [
    "BAR",
    "CHI",
    "MYM",
    "NOA",
    "SYL"
  ],
  "external_present_ids": [
    0,
    1,
    4,
    6,
    11
  ],
  "train_counts": {
    "BAR": 7960,
    "CHI": 14090,
    "KHU": 7626,
    "KIS": 9399,
    "MYM": 8115,
    "NAR": 8452,
    "NOA": 7777,
    "NSD": 8985,
    "RAJ": 8455,
    "RAN": 7947,
    "STD": 7886,
    "SYL": 9049,
    "TAN": 7853
  },
  "val_counts": {
    "BAR": 500,
    "CHI": 500,
   

In [26]:
# ============================================================
# PORTION 4.2 — METRIC UTILITIES
# ============================================================

def expected_calibration_error(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = pred == y_true

    ece = 0.0
    edges = np.linspace(0.0, 1.0, n_bins + 1)

    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi)
        if not np.any(mask):
            continue
        ece += abs(correct[mask].mean() - conf[mask].mean()) * mask.mean()

    return float(ece)


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: List[int],
    label_names: List[str],
    probs: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_true)) > 1 else 0.0,
        "ece": expected_calibration_error(probs, y_true) if probs is not None else None,
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels),
    }


def restrict_probs(probs_13: np.ndarray, allowed_ids: List[int]) -> Tuple[np.ndarray, np.ndarray]:
    restricted = probs_13[:, allowed_ids]
    restricted = restricted / np.maximum(restricted.sum(axis=1, keepdims=True), 1e-12)
    pred_local = restricted.argmax(axis=1)
    pred_global = np.asarray([allowed_ids[i] for i in pred_local], dtype=np.int64)
    return restricted, pred_global


def evaluate_probs(
    y_true: np.ndarray,
    probs_13: np.ndarray,
    split: str,
    restricted_ids: Optional[List[int]] = None,
) -> Dict[str, Any]:
    pred_13 = probs_13.argmax(axis=1)

    out = {
        f"{split}_13way": compute_metrics(
            y_true=y_true,
            y_pred=pred_13,
            labels=list(range(len(DIALECTS))),
            label_names=DIALECTS,
            probs=probs_13,
        )
    }

    if restricted_ids is not None:
        restricted_probs, restricted_pred = restrict_probs(probs_13, restricted_ids)
        restricted_names = [ID_TO_DIALECT[i] for i in restricted_ids]
        out[f"{split}_restricted"] = compute_metrics(
            y_true=y_true,
            y_pred=restricted_pred,
            labels=restricted_ids,
            label_names=restricted_names,
            probs=restricted_probs,
        )

    return out


def onehot_probs(pred: np.ndarray, n_classes: int = len(DIALECTS), confidence: float = 0.999) -> np.ndarray:
    probs = np.full((len(pred), n_classes), (1.0 - confidence) / max(1, n_classes - 1), dtype=np.float32)
    probs[np.arange(len(pred)), pred] = confidence
    return probs


def serializable_metrics(metrics: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in metrics.items():
        out[k] = {}
        for kk, vv in v.items():
            if kk == "confusion_matrix":
                out[k][kk] = vv.tolist()
            else:
                out[k][kk] = vv
    return out


def report_to_df(report_dict: Dict[str, Any], split: str, label_names: List[str]) -> pd.DataFrame:
    rows = []
    for d in label_names:
        r = report_dict.get(d, {})
        rows.append({
            "split": split,
            "dialect": d,
            "precision": float(r.get("precision", 0.0)),
            "recall": float(r.get("recall", 0.0)),
            "f1": float(r.get("f1-score", 0.0)),
            "support": int(r.get("support", 0)),
        })
    return pd.DataFrame(rows)


print("[OK] Metric utilities ready.")

[OK] Metric utilities ready.


In [27]:
# ============================================================
# PORTION 4.3 — MAJORITY + TF-IDF BASELINES
# ============================================================

X_train = train_df["p4_text"].tolist()
y_train = train_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

X_val = val_df["p4_text"].tolist()
y_val = val_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

X_ext = external_df["p4_text"].tolist()
y_ext = external_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

baseline_rows = []
baseline_metric_dump = {}


def add_baseline_result(name: str, val_probs: np.ndarray, ext_probs: np.ndarray, extra: Optional[Dict[str, Any]] = None):
    val_metrics = evaluate_probs(y_val, val_probs, "val", restricted_ids=None)
    ext_metrics = evaluate_probs(y_ext, ext_probs, "external", restricted_ids=external_present_ids)

    row = {
        "model": name,
        "val_13_acc": val_metrics["val_13way"]["accuracy"],
        "val_13_macro_f1": val_metrics["val_13way"]["macro_f1"],
        "val_13_weighted_f1": val_metrics["val_13way"]["weighted_f1"],
        "external_13_acc": ext_metrics["external_13way"]["accuracy"],
        "external_13_macro_f1": ext_metrics["external_13way"]["macro_f1"],
        "external_present_acc": ext_metrics["external_restricted"]["accuracy"],
        "external_present_macro_f1": ext_metrics["external_restricted"]["macro_f1"],
        "external_present_weighted_f1": ext_metrics["external_restricted"]["weighted_f1"],
        "external_present_ece": ext_metrics["external_restricted"]["ece"],
    }

    if extra:
        row.update(extra)

    baseline_rows.append(row)
    baseline_metric_dump[name] = {
        "val": serializable_metrics(val_metrics),
        "external": serializable_metrics(ext_metrics),
    }

    print(f"\n{name}")
    print(pd.DataFrame([row]).to_string(index=False))


# Majority baseline.
majority_label = Counter(y_train.tolist()).most_common(1)[0][0]
val_majority = np.full_like(y_val, majority_label)
ext_majority = np.full_like(y_ext, majority_label)

add_baseline_result(
    "majority_train_13way",
    onehot_probs(val_majority),
    onehot_probs(ext_majority),
)

# Restricted external majority baseline.
train_present = train_df[train_df["dialect"].isin(external_present_dialects)].copy()
present_majority_label = Counter(train_present["dialect"].map(DIALECT_TO_ID).values.tolist()).most_common(1)[0][0]

val_present_majority = np.full_like(y_val, present_majority_label)
ext_present_majority = np.full_like(y_ext, present_majority_label)

add_baseline_result(
    "majority_external_present",
    onehot_probs(val_present_majority),
    onehot_probs(ext_present_majority),
)


def aligned_probs_from_model(model, X: List[str], n_classes: int = len(DIALECTS)) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        local_probs = model.predict_proba(X)
        classes = list(model.classes_) if hasattr(model, "classes_") else list(model.named_steps["clf"].classes_)
        full_probs = np.zeros((len(X), n_classes), dtype=np.float32)
        for j, c in enumerate(classes):
            full_probs[:, int(c)] = local_probs[:, j]
        return full_probs

    pred = model.predict(X)
    return onehot_probs(np.asarray(pred, dtype=np.int64), n_classes=n_classes)


baseline_models = {
    "char_tfidf_logreg": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 6),
            max_features=CFG.tfidf_char_max_features,
            min_df=CFG.tfidf_min_df,
        )),
        ("clf", LogisticRegression(
            max_iter=2500,
            class_weight="balanced",
            C=4.0,
            n_jobs=-1,
            random_state=SEED,
        )),
    ]),
    "char_tfidf_linear_svm": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 6),
            max_features=CFG.tfidf_char_max_features,
            min_df=CFG.tfidf_min_df,
        )),
        ("clf", LinearSVC(
            class_weight="balanced",
            C=1.0,
            random_state=SEED,
        )),
    ]),
    "word_tfidf_logreg": Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            max_features=CFG.tfidf_word_max_features,
            min_df=CFG.tfidf_min_df,
        )),
        ("clf", LogisticRegression(
            max_iter=2500,
            class_weight="balanced",
            C=4.0,
            n_jobs=-1,
            random_state=SEED,
        )),
    ]),
    "combined_char_word_tfidf_logreg": Pipeline([
        ("features", FeatureUnion([
            ("char", TfidfVectorizer(
                analyzer="char",
                ngram_range=(2, 6),
                max_features=CFG.tfidf_char_max_features,
                min_df=CFG.tfidf_min_df,
            )),
            ("word", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 3),
                max_features=CFG.tfidf_word_max_features,
                min_df=CFG.tfidf_min_df,
            )),
        ])),
        ("clf", LogisticRegression(
            max_iter=2500,
            class_weight="balanced",
            C=4.0,
            n_jobs=-1,
            random_state=SEED,
        )),
    ]),
    "char_hashing_sgd_logloss": Pipeline([
        ("hash", HashingVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 6),
            n_features=2**20,
            alternate_sign=False,
        )),
        ("clf", SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=30,
            class_weight="balanced",
            random_state=SEED,
        )),
    ]),
}

for name, model in baseline_models.items():
    print("=" * 100)
    print("Training baseline:", name)
    print("=" * 100)

    start = time.time()
    model.fit(X_train, y_train)

    val_probs = aligned_probs_from_model(model, X_val)
    ext_probs = aligned_probs_from_model(model, X_ext)

    add_baseline_result(
        name,
        val_probs,
        ext_probs,
        extra={"train_sec": time.time() - start},
    )

# Train a separate restricted-present classifier for external protocol sanity.
train_present = train_df[train_df["dialect"].isin(external_present_dialects)].copy()
val_present = val_df[val_df["dialect"].isin(external_present_dialects)].copy()

X_train_present = train_present["p4_text"].tolist()
y_train_present = train_present["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

present_model = Pipeline([
    ("features", FeatureUnion([
        ("char", TfidfVectorizer(
            analyzer="char",
            ngram_range=(2, 6),
            max_features=CFG.tfidf_char_max_features,
            min_df=CFG.tfidf_min_df,
        )),
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            max_features=CFG.tfidf_word_max_features,
            min_df=CFG.tfidf_min_df,
        )),
    ])),
    ("clf", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        C=4.0,
        n_jobs=-1,
        random_state=SEED,
    )),
])

present_model.fit(X_train_present, y_train_present)

val_present_probs = aligned_probs_from_model(present_model, val_present["p4_text"].tolist())
ext_present_probs = aligned_probs_from_model(present_model, X_ext)

# This row uses val restricted subset for val numbers.
val_present_y = val_present["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

val_present_metrics = evaluate_probs(
    val_present_y,
    val_present_probs,
    "val_present",
    restricted_ids=external_present_ids,
)
ext_present_metrics = evaluate_probs(
    y_ext,
    ext_present_probs,
    "external",
    restricted_ids=external_present_ids,
)

baseline_rows.append({
    "model": "combined_tfidf_retrained_present_dialects",
    "val_13_acc": np.nan,
    "val_13_macro_f1": np.nan,
    "val_present_acc": val_present_metrics["val_present_restricted"]["accuracy"],
    "val_present_macro_f1": val_present_metrics["val_present_restricted"]["macro_f1"],
    "external_13_acc": ext_present_metrics["external_13way"]["accuracy"],
    "external_13_macro_f1": ext_present_metrics["external_13way"]["macro_f1"],
    "external_present_acc": ext_present_metrics["external_restricted"]["accuracy"],
    "external_present_macro_f1": ext_present_metrics["external_restricted"]["macro_f1"],
    "external_present_weighted_f1": ext_present_metrics["external_restricted"]["weighted_f1"],
    "external_present_ece": ext_present_metrics["external_restricted"]["ece"],
})

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(REPORT_DIR / "p4_baseline_results.csv", index=False)
write_json(REPORT_DIR / "p4_baseline_metric_dump.json", baseline_metric_dump)

print("=" * 100)
print("P4 BASELINE RESULTS")
print("=" * 100)
print(baseline_df.to_string(index=False))


majority_train_13way
               model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece
majority_train_13way    0.076923         0.010989            0.010989         0.217391              0.027473              0.217391                   0.071429                       0.07764              0.782275

majority_external_present
                    model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece
majority_external_present    0.076923         0.010989            0.010989         0.217391              0.027473              0.217391                   0.071429                       0.07764              0.782275
Training baseline: char_tfidf_logreg

char_tfidf_logreg
            model  val_13_acc  val_13_macro_f

In [28]:
# ============================================================
# PORTION 4.4 — DENSE MODEL + FROZEN BLEACH EMBEDDING BASELINE
# ============================================================

P2_SOURCE_DIR = P2_ROOT / "source"
sys.path.insert(0, str(P2_SOURCE_DIR))

from bleach_dense_model import BLEACHDenseConfig, BLEACHDenseForCausalLM

MODEL_CFG = BLEACHDenseConfig(**read_json(P2_ROOT / "config" / "dense_32m_config.json"))
tokenizer = Tokenizer.from_file(str(TOKENIZER_ROOT / "tokenizer.json"))

assert tokenizer.get_vocab_size() == CFG.vocab_size, tokenizer.get_vocab_size()


def load_dense_lm() -> BLEACHDenseForCausalLM:
    model = BLEACHDenseForCausalLM(MODEL_CFG)

    if SELECTED_DENSE.get("safetensors") and Path(SELECTED_DENSE["safetensors"]).exists():
        state = safe_load_file(SELECTED_DENSE["safetensors"])
        model.load_state_dict(state, strict=True)
        loaded_from = SELECTED_DENSE["safetensors"]
    else:
        ckpt_path = Path(SELECTED_DENSE["checkpoint_pt"])
        ckpt = torch.load(ckpt_path, map_location="cpu")
        if "model_state_dict" in ckpt:
            model.load_state_dict(ckpt["model_state_dict"], strict=True)
        elif "state_dict" in ckpt:
            model.load_state_dict(ckpt["state_dict"], strict=True)
        else:
            raise KeyError(f"Cannot find model state in {ckpt_path}")
        loaded_from = str(ckpt_path)

    model.loaded_from = loaded_from
    return model


def encode_for_model(text: str, max_length: int, random_crop: bool = False) -> Tuple[List[int], List[int]]:
    ids = list(map(int, tokenizer.encode(normalize_text_for_cls(text)).ids))
    max_inner = max_length - 2

    if len(ids) > max_inner:
        if random_crop:
            start = random.randint(0, len(ids) - max_inner)
            ids = ids[start:start + max_inner]
        else:
            ids = ids[:max_inner]

    ids = [CFG.bos_token_id] + ids + [CFG.eos_token_id]
    mask = [1] * len(ids)

    pad_len = max_length - len(ids)
    if pad_len > 0:
        ids += [CFG.pad_token_id] * pad_len
        mask += [0] * pad_len

    return ids, mask


@torch.no_grad()
def build_frozen_embedding_features(df: pd.DataFrame, batch_size: int = 512) -> np.ndarray:
    base = load_dense_lm().to(DEVICE)
    base.eval()

    feats = []

    for start in range(0, len(df), batch_size):
        sub = df.iloc[start:start + batch_size]

        ids_list, mask_list = [], []
        for text in sub["p4_text"].tolist():
            ids, mask = encode_for_model(text, CFG.max_length, random_crop=False)
            ids_list.append(ids)
            mask_list.append(mask)

        input_ids = torch.as_tensor(ids_list, dtype=torch.long, device=DEVICE)
        mask = torch.as_tensor(mask_list, dtype=torch.float32, device=DEVICE)

        emb = base.token_embedding(input_ids)  # [B, T, D]
        mean = (emb * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1.0)

        feats.append(mean.detach().float().cpu().numpy())

    del base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.concatenate(feats, axis=0)


print("Building frozen BLEACH embedding features...")
train_emb = build_frozen_embedding_features(train_df)
val_emb = build_frozen_embedding_features(val_df)
ext_emb = build_frozen_embedding_features(external_df)

emb_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2500,
        class_weight="balanced",
        C=4.0,
        n_jobs=-1,
        random_state=SEED,
    )),
])

emb_clf.fit(train_emb, y_train)

val_emb_probs_local = emb_clf.predict_proba(val_emb)
ext_emb_probs_local = emb_clf.predict_proba(ext_emb)

val_emb_probs = np.zeros((len(val_df), len(DIALECTS)), dtype=np.float32)
ext_emb_probs = np.zeros((len(external_df), len(DIALECTS)), dtype=np.float32)

for j, c in enumerate(emb_clf.named_steps["clf"].classes_):
    val_emb_probs[:, int(c)] = val_emb_probs_local[:, j]
    ext_emb_probs[:, int(c)] = ext_emb_probs_local[:, j]

add_baseline_result(
    "frozen_bleach_embedding_logreg",
    val_emb_probs,
    ext_emb_probs,
)

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(REPORT_DIR / "p4_baseline_results.csv", index=False)

print(baseline_df.to_string(index=False))

Building frozen BLEACH embedding features...

frozen_bleach_embedding_logreg
                         model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece
frozen_bleach_embedding_logreg    0.689538         0.689566            0.689566         0.168696               0.05188               0.20313                   0.151059                        0.1597              0.488312
                                    model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece  train_sec  val_present_acc  val_present_macro_f1
                     majority_train_13way    0.076923         0.010989            0.010989         0.217391              0.027473              0.217391                   0.071429                      0.

In [29]:
# ============================================================
# PORTION 4.5 — BLEACH CLASSIFIER DATASET + LOADERS
# ============================================================

class DialectClassificationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, max_length: int, random_crop: bool = False):
        self.df = df.reset_index(drop=True).copy()
        self.max_length = max_length
        self.random_crop = random_crop
        self.labels = self.df["dialect"].map(DIALECT_TO_ID).astype(int).values

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        ids, mask = encode_for_model(row["p4_text"], self.max_length, random_crop=self.random_crop)

        return {
            "input_ids": torch.as_tensor(ids, dtype=torch.long),
            "attention_mask": torch.as_tensor(mask, dtype=torch.long),
            "labels": torch.as_tensor(int(self.labels[idx]), dtype=torch.long),
            "dialect": row["dialect"],
            "text": row["p4_text"],
            "is_synthetic": torch.as_tensor(float(row.get("is_synthetic_inferred", False)), dtype=torch.float32),
            "quality_score": torch.as_tensor(float(row.get("quality_score_p4", 1.0)), dtype=torch.float32),
        }


def compute_sampler_weights(df: pd.DataFrame) -> np.ndarray:
    counts = df["dialect"].value_counts().to_dict()

    raw_mass = {
        d: max(1.0, float(counts.get(d, 0))) ** CFG.sampler_alpha
        for d in DIALECTS
    }
    total = sum(raw_mass.values())
    target_probs = {d: raw_mass[d] / total for d in DIALECTS}

    weights = []
    for _, row in df.iterrows():
        d = row["dialect"]
        n = max(1, counts.get(d, 1))
        weights.append(target_probs[d] / n)

    weights = np.asarray(weights, dtype=np.float64)
    weights = weights / weights.mean()
    return weights


def make_loader(ds: Dataset, df: pd.DataFrame, batch_size: int, train: bool) -> DataLoader:
    if train and CFG.use_balanced_sampler:
        weights = compute_sampler_weights(df)
        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(weights, dtype=torch.double),
            num_samples=len(ds),
            replacement=True,
        )
        shuffle = False
    else:
        sampler = None
        shuffle = train

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=CFG.num_workers,
        pin_memory=True,
        drop_last=False,
    )


train_ds = DialectClassificationDataset(train_df, CFG.max_length, random_crop=True)
val_ds = DialectClassificationDataset(val_df, CFG.max_length, random_crop=False)
external_ds = DialectClassificationDataset(external_df, CFG.max_length, random_crop=False)

train_loader = make_loader(train_ds, train_df, CFG.batch_size, train=True)
val_loader = make_loader(val_ds, val_df, CFG.eval_batch_size, train=False)
external_loader = make_loader(external_ds, external_df, CFG.eval_batch_size, train=False)

counts = train_df["dialect"].value_counts().reindex(DIALECTS).fillna(0).values.astype(np.float32)
class_w = counts.sum() / np.maximum(1.0, len(DIALECTS) * counts)
class_w = class_w ** CFG.class_weight_power
class_weights = torch.as_tensor(class_w, dtype=torch.float32, device=DEVICE)

loader_report = {
    "train_batches": len(train_loader),
    "val_batches": len(val_loader),
    "external_batches": len(external_loader),
    "class_weights": {d: float(class_weights[i].detach().cpu()) for i, d in enumerate(DIALECTS)},
    "effective_train_batch": CFG.batch_size,
    "max_length": CFG.max_length,
}

write_json(REPORT_DIR / "p4_neural_loader_report.json", loader_report)
print(json.dumps(loader_report, indent=2))

{
  "train_batches": 1775,
  "val_batches": 51,
  "external_batches": 23,
  "class_weights": {
    "BAR": 1.0477303266525269,
    "CHI": 0.7874999046325684,
    "KHU": 1.0704283714294434,
    "KIS": 0.964195728302002,
    "MYM": 1.0376759767532349,
    "NAR": 1.0167783498764038,
    "NOA": 1.059985637664795,
    "NSD": 0.986159086227417,
    "RAJ": 1.0165979862213135,
    "RAN": 1.0485868453979492,
    "STD": 1.052634596824646,
    "SYL": 0.9826655387878418,
    "TAN": 1.0548440217971802
  },
  "effective_train_batch": 64,
  "max_length": 256
}


In [30]:
# ============================================================
# PORTION 4.6 — CE-FIRST BLEACH CLASSIFIER ARCHITECTURE
# No previous classifier checkpoint is used.
# ============================================================

class BLEACHBackboneEncoder(nn.Module):
    def __init__(self, base: BLEACHDenseForCausalLM):
        super().__init__()
        self.base = base
        self.cfg = base.cfg

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # input_ids: [B, T]
        b, t = input_ids.shape

        x = self.base.token_embedding(input_ids)  # [B, T, D]
        x = self.base.dropout(x)

        cos, sin = self.base.rope(seq_len=t, device=input_ids.device, dtype=x.dtype)

        for layer in self.base.layers:
            x = layer(x, cos, sin)                # [B, T, D]

        x = self.base.final_norm(x)               # [B, T, D]
        return x


class AttentivePooler(nn.Module):
    def __init__(self, d_model: int, dropout: float):
        super().__init__()
        self.proj = nn.Linear(d_model, d_model)
        self.score = nn.Linear(d_model, 1, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden: torch.Tensor, attention_mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # hidden: [B, T, D], attention_mask: [B, T]
        h = torch.tanh(self.proj(self.dropout(hidden)))      # [B, T, D]
        scores = self.score(h).squeeze(-1)                   # [B, T]
        scores = scores.masked_fill(attention_mask == 0, -1e4)
        weights = F.softmax(scores.float(), dim=-1).to(hidden.dtype)
        pooled = (hidden * weights.unsqueeze(-1)).sum(dim=1) # [B, D]
        return pooled, weights


class BLEACHCEFirstClassifier(nn.Module):
    def __init__(self, base: BLEACHDenseForCausalLM, num_classes: int, dropout: float):
        super().__init__()
        self.encoder = BLEACHBackboneEncoder(base)
        d = base.cfg.d_model

        self.attn_pool = AttentivePooler(d, dropout=dropout)
        self.pool_norm = nn.LayerNorm(d * 4)

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d * 4, d * 2),
            nn.GELU(),
            nn.LayerNorm(d * 2),
            nn.Dropout(dropout),
            nn.Linear(d * 2, num_classes),
        )

    def last_pool(self, hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        lengths = attention_mask.long().sum(dim=1).clamp(min=1) - 1
        idx = torch.arange(hidden.size(0), device=hidden.device)
        return hidden[idx, lengths]                         # [B, D]

    def mean_pool(self, hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        mask = attention_mask.unsqueeze(-1).to(hidden.dtype) # [B, T, 1]
        return (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

    def max_pool(self, hidden: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        masked = hidden.masked_fill(attention_mask.unsqueeze(-1) == 0, -1e4)
        return masked.max(dim=1).values

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, return_features: bool = False) -> Dict[str, torch.Tensor]:
        hidden = self.encoder(input_ids)                      # [B, T, D]

        last = self.last_pool(hidden, attention_mask)          # [B, D]
        mean = self.mean_pool(hidden, attention_mask)          # [B, D]
        maxp = self.max_pool(hidden, attention_mask)           # [B, D]
        attn, attn_weights = self.attn_pool(hidden, attention_mask)

        features = torch.cat([last, mean, maxp, attn], dim=-1) # [B, 4D]
        features = self.pool_norm(features)

        logits = self.classifier(features)                     # [B, C]

        out = {
            "logits": logits,
            "features": features,
            "attention_weights": attn_weights,
        }

        if return_features:
            out["hidden"] = hidden

        return out


def load_fresh_classifier() -> BLEACHCEFirstClassifier:
    base = load_dense_lm()
    model = BLEACHCEFirstClassifier(
        base=base,
        num_classes=len(DIALECTS),
        dropout=CFG.dropout,
    )
    return model.to(DEVICE)


classifier = load_fresh_classifier()

param_report = {
    "total_params": int(sum(p.numel() for p in classifier.parameters())),
    "trainable_params_initial": int(sum(p.numel() for p in classifier.parameters() if p.requires_grad)),
    "dense_lm_loaded_from": classifier.encoder.base.loaded_from,
}

write_json(REPORT_DIR / "p4_classifier_parameter_report.json", param_report)
print(json.dumps(param_report, indent=2))

{
  "total_params": 32516749,
  "trainable_params_initial": 32516749,
  "dense_lm_loaded_from": "/kaggle/working/artifacts/bleach_p3_dense_clm/checkpoints/best_dense_32m.safetensors"
}


In [31]:
# ============================================================
# PORTION 4.7 — LOSS, TRAINABILITY, OPTIMIZER
# ============================================================

class WeightedLabelSmoothingCE(nn.Module):
    def __init__(self, num_classes: int, smoothing: float, class_weights: torch.Tensor):
        super().__init__()
        self.num_classes = num_classes
        self.smoothing = smoothing
        self.register_buffer("class_weights", class_weights)

    def forward(self, logits: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        log_probs = F.log_softmax(logits.float(), dim=-1)

        with torch.no_grad():
            target = torch.zeros_like(log_probs)
            target.fill_(self.smoothing / max(1, self.num_classes - 1))
            target.scatter_(1, labels.unsqueeze(1), 1.0 - self.smoothing)

        ce = -(target * log_probs).sum(dim=-1)
        w = self.class_weights[labels].to(logits.device)
        return (ce * w).mean()


def symmetric_kl(logits_a: torch.Tensor, logits_b: torch.Tensor) -> torch.Tensor:
    logp_a = F.log_softmax(logits_a.float(), dim=-1)
    logp_b = F.log_softmax(logits_b.float(), dim=-1)
    p_a = logp_a.exp()
    p_b = logp_b.exp()
    return 0.5 * (
        F.kl_div(logp_a, p_b, reduction="batchmean") +
        F.kl_div(logp_b, p_a, reduction="batchmean")
    )


criterion = WeightedLabelSmoothingCE(
    num_classes=len(DIALECTS),
    smoothing=CFG.label_smoothing,
    class_weights=class_weights,
).to(DEVICE)


def get_stage(epoch: int) -> str:
    if epoch <= CFG.head_only_epochs:
        return "head_pooler_only"
    if epoch <= CFG.head_only_epochs + CFG.top4_epochs:
        return "top4_layers"
    return "top6_layers"


def set_trainability(model: BLEACHCEFirstClassifier, epoch: int) -> str:
    stage = get_stage(epoch)

    for p in model.parameters():
        p.requires_grad = False

    # Always train pooler/head.
    for module in [model.attn_pool, model.pool_norm, model.classifier]:
        for p in module.parameters():
            p.requires_grad = True

    if stage == "head_pooler_only":
        pass

    elif stage == "top4_layers":
        for layer in model.encoder.base.layers[-4:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in model.encoder.base.final_norm.parameters():
            p.requires_grad = True

    elif stage == "top6_layers":
        for layer in model.encoder.base.layers[-6:]:
            for p in layer.parameters():
                p.requires_grad = True
        for p in model.encoder.base.final_norm.parameters():
            p.requires_grad = True

    return stage


def build_optimizer(model: BLEACHCEFirstClassifier):
    head_params = []
    pooler_params = []
    top_params = []
    lower_top_params = []

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if name.startswith("classifier"):
            head_params.append(p)
        elif name.startswith("attn_pool") or name.startswith("pool_norm"):
            pooler_params.append(p)
        elif "encoder.base.layers" in name:
            m = re.search(r"encoder\.base\.layers\.(\d+)\.", name)
            layer_idx = int(m.group(1)) if m else MODEL_CFG.n_layers - 1
            if layer_idx >= MODEL_CFG.n_layers - 4:
                top_params.append(p)
            else:
                lower_top_params.append(p)
        else:
            top_params.append(p)

    groups = []
    if head_params:
        groups.append({"params": head_params, "lr": CFG.lr_head, "weight_decay": CFG.weight_decay, "name": "head"})
    if pooler_params:
        groups.append({"params": pooler_params, "lr": CFG.lr_pooler, "weight_decay": CFG.weight_decay, "name": "pooler"})
    if top_params:
        groups.append({"params": top_params, "lr": CFG.lr_top_layers, "weight_decay": CFG.weight_decay, "name": "top_layers"})
    if lower_top_params:
        groups.append({"params": lower_top_params, "lr": CFG.lr_lower_top_layers, "weight_decay": CFG.weight_decay, "name": "lower_top_layers"})

    try:
        opt = torch.optim.AdamW(
            groups,
            betas=(0.9, 0.95),
            eps=1e-8,
            fused=True if DEVICE.type == "cuda" else False,
        )
    except TypeError:
        opt = torch.optim.AdamW(groups, betas=(0.9, 0.95), eps=1e-8)

    return opt


def trainable_count(model: nn.Module) -> Dict[str, int]:
    return {
        "trainable_params": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
        "total_params": int(sum(p.numel() for p in model.parameters())),
    }


def get_amp_dtype():
    if DEVICE.type != "cuda":
        return None
    if CFG.prefer_bf16 and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    return torch.float16


AMP_DTYPE = get_amp_dtype()

print("AMP dtype:", AMP_DTYPE)
print("[OK] Loss/trainability/optimizer utilities ready.")

AMP dtype: torch.bfloat16
[OK] Loss/trainability/optimizer utilities ready.


In [32]:
# ============================================================
# PORTION 4.8 — NEURAL EVALUATION FUNCTION
# ============================================================

@torch.no_grad()
def evaluate_classifier(
    model: BLEACHCEFirstClassifier,
    loader: DataLoader,
    split: str,
    collect_features: bool = False,
) -> Tuple[Dict[str, Any], pd.DataFrame, Optional[np.ndarray]]:
    model.eval()

    y_true_all = []
    logits_all = []
    text_all = []
    features_all = []

    loss_sum = 0.0
    batches = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        if AMP_DTYPE is not None:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out = model(input_ids, attention_mask, return_features=collect_features)
                logits = out["logits"]
                loss = criterion(logits, labels)
        else:
            out = model(input_ids, attention_mask, return_features=collect_features)
            logits = out["logits"]
            loss = criterion(logits, labels)

        loss_sum += float(loss.detach().cpu())
        batches += 1

        y_true_all.extend(labels.detach().cpu().numpy().tolist())
        logits_all.append(logits.detach().float().cpu().numpy())
        text_all.extend(batch["text"])

        if collect_features:
            features_all.append(out["features"].detach().float().cpu().numpy())

    y_true = np.asarray(y_true_all, dtype=np.int64)
    logits = np.concatenate(logits_all, axis=0)
    probs = torch.softmax(torch.as_tensor(logits), dim=-1).numpy()

    restricted_ids = external_present_ids if split == "external" else None
    metrics = evaluate_probs(y_true, probs, split, restricted_ids=restricted_ids)
    metrics[f"{split}_13way"]["loss"] = loss_sum / max(1, batches)

    pred_13 = probs.argmax(axis=1)

    pred_df = pd.DataFrame({
        "split": split,
        "gold_id": y_true,
        "pred_13_id": pred_13,
        "gold_dialect": [ID_TO_DIALECT[int(x)] for x in y_true],
        "pred_13_dialect": [ID_TO_DIALECT[int(x)] for x in pred_13],
        "correct_13": (y_true == pred_13).astype(int),
        "confidence_13": probs.max(axis=1),
        "text": text_all,
    })

    if split == "external":
        probs_restricted, pred_restricted = restrict_probs(probs, external_present_ids)
        pred_df["pred_restricted_id"] = pred_restricted
        pred_df["pred_restricted_dialect"] = [ID_TO_DIALECT[int(x)] for x in pred_restricted]
        pred_df["correct_restricted"] = (y_true == pred_restricted).astype(int)
        pred_df["confidence_restricted"] = probs_restricted.max(axis=1)

    for i, d in enumerate(DIALECTS):
        pred_df[f"prob_{d}"] = probs[:, i]

    features = np.concatenate(features_all, axis=0) if collect_features and features_all else None

    return metrics, pred_df, features


print("[OK] Evaluation function ready.")

[OK] Evaluation function ready.


In [33]:
# ============================================================
# PORTION 4.9 — TRAIN CE-FIRST BLEACH CLASSIFIER
# ============================================================

def cosine_scale(step: int, total_steps: int, warmup_steps: int) -> float:
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))

    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    progress = min(1.0, max(0.0, progress))
    return CFG.min_lr_ratio + (1.0 - CFG.min_lr_ratio) * 0.5 * (1.0 + math.cos(math.pi * progress))


def set_lr_scale(optimizer: torch.optim.Optimizer, scale: float) -> None:
    for group in optimizer.param_groups:
        if "base_lr" not in group:
            group["base_lr"] = group["lr"]
        group["lr"] = group["base_lr"] * scale


history = []
best_val_macro = -1.0
best_epoch = -1
epochs_no_improve = 0
global_step = 0

steps_per_epoch = len(train_loader)
total_steps = CFG.epochs * steps_per_epoch
warmup_steps = int(total_steps * CFG.warmup_ratio)

current_stage = None
optimizer = None

print("=" * 100)
print("P4 NEURAL TRAINING START")
print("=" * 100)
print("steps_per_epoch:", steps_per_epoch)
print("total_steps:", total_steps)
print("warmup_steps:", warmup_steps)
print("=" * 100)

for epoch in range(1, CFG.epochs + 1):
    stage = set_trainability(classifier, epoch)

    if stage != current_stage or optimizer is None:
        optimizer = build_optimizer(classifier)
        current_stage = stage
        print(f"[STAGE CHANGE] epoch={epoch}, stage={stage}, trainable={trainable_count(classifier)}")

    classifier.train()

    loss_sum = 0.0
    ce_sum = 0.0
    rdrop_sum = 0.0
    correct = 0
    total = 0
    batches = 0

    start_time = time.time()

    for step, batch in enumerate(train_loader, start=1):
        scale = cosine_scale(global_step, total_steps, warmup_steps)
        set_lr_scale(optimizer, scale)

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        use_rdrop_now = CFG.use_rdrop and epoch >= CFG.rdrop_start_epoch

        if AMP_DTYPE is not None:
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                out1 = classifier(input_ids, attention_mask)
                logits1 = out1["logits"]
                ce1 = criterion(logits1, labels)

                if use_rdrop_now:
                    out2 = classifier(input_ids, attention_mask)
                    logits2 = out2["logits"]
                    rdrop = symmetric_kl(logits1, logits2)
                    loss = ce1 + CFG.rdrop_alpha * rdrop
                else:
                    rdrop = torch.zeros_like(ce1)
                    loss = ce1
        else:
            out1 = classifier(input_ids, attention_mask)
            logits1 = out1["logits"]
            ce1 = criterion(logits1, labels)

            if use_rdrop_now:
                out2 = classifier(input_ids, attention_mask)
                logits2 = out2["logits"]
                rdrop = symmetric_kl(logits1, logits2)
                loss = ce1 + CFG.rdrop_alpha * rdrop
            else:
                rdrop = torch.zeros_like(ce1)
                loss = ce1

        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in classifier.parameters() if p.requires_grad],
            CFG.grad_clip_norm,
        )
        optimizer.step()

        with torch.no_grad():
            pred = logits1.float().argmax(dim=-1)
            correct += int((pred == labels).sum().detach().cpu())
            total += int(labels.numel())

        loss_sum += float(loss.detach().cpu())
        ce_sum += float(ce1.detach().cpu())
        rdrop_sum += float(rdrop.detach().cpu())
        batches += 1
        global_step += 1

        if step % CFG.log_every_steps == 0:
            lrs = {g.get("name", f"group_{i}"): g["lr"] for i, g in enumerate(optimizer.param_groups)}
            print(
                f"[E{epoch:02d} step {step:04d}/{len(train_loader)}] "
                f"loss={loss_sum/max(1,batches):.4f} "
                f"ce={ce_sum/max(1,batches):.4f} "
                f"rdrop={rdrop_sum/max(1,batches):.4f} "
                f"acc={correct/max(1,total):.4f} "
                f"scale={scale:.3f} "
                f"lrs={lrs}"
            )

    val_metrics, val_pred_tmp, _ = evaluate_classifier(classifier, val_loader, "val", collect_features=False)
    ext_metrics, ext_pred_tmp, _ = evaluate_classifier(classifier, external_loader, "external", collect_features=False)

    val13 = val_metrics["val_13way"]
    ext13 = ext_metrics["external_13way"]
    ext_present = ext_metrics["external_restricted"]

    row = {
        "epoch": epoch,
        "stage": stage,
        "global_step": global_step,
        "train_loss": loss_sum / max(1, batches),
        "train_ce": ce_sum / max(1, batches),
        "train_rdrop": rdrop_sum / max(1, batches),
        "train_accuracy": correct / max(1, total),
        "val_13_acc": val13["accuracy"],
        "val_13_macro_f1": val13["macro_f1"],
        "val_13_weighted_f1": val13["weighted_f1"],
        "val_13_mcc": val13["mcc"],
        "val_13_ece": val13["ece"],
        "external_13_acc": ext13["accuracy"],
        "external_13_macro_f1": ext13["macro_f1"],
        "external_13_weighted_f1": ext13["weighted_f1"],
        "external_13_mcc": ext13["mcc"],
        "external_13_ece": ext13["ece"],
        "external_present_acc": ext_present["accuracy"],
        "external_present_macro_f1": ext_present["macro_f1"],
        "external_present_weighted_f1": ext_present["weighted_f1"],
        "external_present_mcc": ext_present["mcc"],
        "external_present_ece": ext_present["ece"],
        "elapsed_sec": time.time() - start_time,
    }

    history.append(row)
    pd.DataFrame(history).to_csv(REPORT_DIR / "p4_neural_train_log.csv", index=False)

    print(
        f"[E{epoch:02d}] "
        f"train_acc={row['train_accuracy']:.4f} | "
        f"val_acc={row['val_13_acc']:.4f} val_macroF1={row['val_13_macro_f1']:.4f} | "
        f"external13_acc={row['external_13_acc']:.4f} | "
        f"external_present_acc={row['external_present_acc']:.4f} "
        f"external_present_macroF1={row['external_present_macro_f1']:.4f}"
    )

    if row["val_13_macro_f1"] > best_val_macro:
        best_val_macro = row["val_13_macro_f1"]
        best_epoch = epoch
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "best_val_macro_f1": best_val_macro,
                "model_state_dict": classifier.state_dict(),
                "model_config": asdict(MODEL_CFG),
                "p4_config": asdict(CFG),
                "dialects": DIALECTS,
                "external_present_dialects": external_present_dialects,
                "selected_dense": SELECTED_DENSE,
                "history": history,
            },
            CKPT_DIR / "best_p4_bleach_classifier.pt",
        )

        if SAFETENSORS_AVAILABLE:
            state = {k: v.detach().cpu() for k, v in classifier.state_dict().items()}
            safe_save_file(state, CKPT_DIR / "best_p4_bleach_classifier.safetensors")

        print(f"[BEST] epoch={epoch}, val_13_macro_f1={best_val_macro:.4f}")

    else:
        epochs_no_improve += 1
        print(f"[NO IMPROVEMENT] {epochs_no_improve}/{CFG.patience}")

    if epochs_no_improve >= CFG.patience:
        print("[EARLY STOP]")
        break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("=" * 100)
print("P4 NEURAL TRAINING COMPLETE")
print("Best epoch:", best_epoch)
print("Best val macro-F1:", best_val_macro)
print("=" * 100)

P4 NEURAL TRAINING START
steps_per_epoch: 1775
total_steps: 42600
warmup_steps: 2556
[STAGE CHANGE] epoch=1, stage=head_pooler_only, trainable={'trainable_params': 1343245, 'total_params': 32516749}
[E01 step 0150/1775] loss=2.5775 ce=2.5775 rdrop=0.0000 acc=0.1226 scale=0.059 lrs={'head': 5.868544600938967e-05, 'pooler': 4.4014084507042256e-05}
[E01 step 0300/1775] loss=2.3986 ce=2.3986 rdrop=0.0000 acc=0.1791 scale=0.117 lrs={'head': 0.00011737089201877934, 'pooler': 8.802816901408451e-05}
[E01 step 0450/1775] loss=2.2980 ce=2.2980 rdrop=0.0000 acc=0.2118 scale=0.176 lrs={'head': 0.000176056338028169, 'pooler': 0.00013204225352112675}
[E01 step 0600/1775] loss=2.2360 ce=2.2360 rdrop=0.0000 acc=0.2338 scale=0.235 lrs={'head': 0.00023474178403755868, 'pooler': 0.00017605633802816902}
[E01 step 0750/1775] loss=2.1920 ce=2.1920 rdrop=0.0000 acc=0.2490 scale=0.293 lrs={'head': 0.00029342723004694836, 'pooler': 0.00022007042253521127}
[E01 step 0900/1775] loss=2.1561 ce=2.1561 rdrop=0.0000

In [34]:
# ============================================================
# PORTION 4.10 — RELOAD BEST + FINAL EVALUATION
# ============================================================

best_classifier_path = CKPT_DIR / "best_p4_bleach_classifier.pt"
assert best_classifier_path.exists(), "No best P4 classifier checkpoint found."

best_ckpt = torch.load(best_classifier_path, map_location="cpu")
classifier.load_state_dict(best_ckpt["model_state_dict"], strict=True)
classifier = classifier.to(DEVICE).eval()

print("Reloaded best classifier:")
print({
    "epoch": best_ckpt["epoch"],
    "best_val_macro_f1": best_ckpt["best_val_macro_f1"],
})

val_metrics, val_pred_df, val_features = evaluate_classifier(classifier, val_loader, "val", collect_features=True)
ext_metrics, ext_pred_df, ext_features = evaluate_classifier(classifier, external_loader, "external", collect_features=True)

val_pred_df.to_csv(PRED_DIR / "p4_val_predictions.csv", index=False)
ext_pred_df.to_csv(PRED_DIR / "p4_external_predictions.csv", index=False)

write_json(REPORT_DIR / "p4_val_metrics.json", serializable_metrics(val_metrics))
write_json(REPORT_DIR / "p4_external_metrics.json", serializable_metrics(ext_metrics))

val_per_dialect = report_to_df(
    val_metrics["val_13way"]["classification_report"],
    "val_13way",
    DIALECTS,
)
ext13_per_dialect = report_to_df(
    ext_metrics["external_13way"]["classification_report"],
    "external_13way",
    DIALECTS,
)
ext_present_per_dialect = report_to_df(
    ext_metrics["external_restricted"]["classification_report"],
    "external_present",
    external_present_dialects,
)

per_dialect_df = pd.concat(
    [val_per_dialect, ext13_per_dialect, ext_present_per_dialect],
    axis=0,
)
per_dialect_df.to_csv(REPORT_DIR / "p4_per_dialect_metrics.csv", index=False)

bleach_row = {
    "model": "BLEACH_Dense_CE_First_Classifier",
    "val_13_acc": val_metrics["val_13way"]["accuracy"],
    "val_13_macro_f1": val_metrics["val_13way"]["macro_f1"],
    "val_13_weighted_f1": val_metrics["val_13way"]["weighted_f1"],
    "external_13_acc": ext_metrics["external_13way"]["accuracy"],
    "external_13_macro_f1": ext_metrics["external_13way"]["macro_f1"],
    "external_present_acc": ext_metrics["external_restricted"]["accuracy"],
    "external_present_macro_f1": ext_metrics["external_restricted"]["macro_f1"],
    "external_present_weighted_f1": ext_metrics["external_restricted"]["weighted_f1"],
    "external_present_ece": ext_metrics["external_restricted"]["ece"],
}

comparison_df = pd.concat([baseline_df, pd.DataFrame([bleach_row])], axis=0)
comparison_df.to_csv(REPORT_DIR / "p4_baseline_vs_bleach_results.csv", index=False)

print("=" * 100)
print("FINAL P4 CLASSIFICATION RESULT")
print("=" * 100)
print(pd.DataFrame([bleach_row]).to_string(index=False))
print("\nBaseline comparison:")
print(comparison_df.to_string(index=False))

Reloaded best classifier:
{'epoch': 24, 'best_val_macro_f1': 0.5930017737404829}
FINAL P4 CLASSIFICATION RESULT
                           model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece
BLEACH_Dense_CE_First_Classifier    0.589846         0.593002            0.593002         0.189217              0.044822              0.209391                   0.125309                      0.133461              0.409847

Baseline comparison:
                                    model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece  train_sec  val_present_acc  val_present_macro_f1
                     majority_train_13way    0.076923         0.010989            0.010989         0.217391              0.027473            

In [35]:
# ============================================================
# PORTION 4.11 — VISUALIZATIONS
# ============================================================

def plot_confusion(cm: np.ndarray, labels: List[str], title: str, out_path: Path, normalize: bool = False) -> None:
    cm_plot = cm.astype(np.float64)

    if normalize:
        cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1.0)

    plt.figure(figsize=(11, 9))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45)
    plt.yticks(ticks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("Gold")

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            val = cm_plot[i, j]
            if normalize:
                txt = f"{val:.2f}" if val > 0.01 else ""
            else:
                txt = str(int(val)) if val > 0 else ""
            if txt:
                plt.text(j, i, txt, ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=260)
    plt.close()


plot_confusion(
    val_metrics["val_13way"]["confusion_matrix"],
    DIALECTS,
    "BLEACH Dense Classifier — Validation 13-way Confusion Matrix",
    PLOT_DIR / "p4_val_13way_confusion_counts.png",
    normalize=False,
)

plot_confusion(
    val_metrics["val_13way"]["confusion_matrix"],
    DIALECTS,
    "BLEACH Dense Classifier — Validation 13-way Normalized Confusion Matrix",
    PLOT_DIR / "p4_val_13way_confusion_normalized.png",
    normalize=True,
)

plot_confusion(
    ext_metrics["external_13way"]["confusion_matrix"],
    DIALECTS,
    "BLEACH Dense Classifier — External 13-way Confusion Matrix",
    PLOT_DIR / "p4_external_13way_confusion_counts.png",
    normalize=False,
)

plot_confusion(
    ext_metrics["external_restricted"]["confusion_matrix"],
    external_present_dialects,
    "BLEACH Dense Classifier — External Present-Dialect Confusion Matrix",
    PLOT_DIR / "p4_external_present_confusion_counts.png",
    normalize=False,
)

plot_confusion(
    ext_metrics["external_restricted"]["confusion_matrix"],
    external_present_dialects,
    "BLEACH Dense Classifier — External Present-Dialect Normalized Confusion Matrix",
    PLOT_DIR / "p4_external_present_confusion_normalized.png",
    normalize=True,
)

# Training curves.
train_log = pd.read_csv(REPORT_DIR / "p4_neural_train_log.csv")

plt.figure(figsize=(10, 6))
plt.plot(train_log["epoch"], train_log["train_loss"], label="train_loss")
plt.plot(train_log["epoch"], train_log["train_ce"], label="train_ce")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("P4 BLEACH Classifier Training Loss")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_training_loss_curves.png", dpi=260)
plt.close()

plt.figure(figsize=(10, 6))
plt.plot(train_log["epoch"], train_log["val_13_acc"], label="val_13_acc")
plt.plot(train_log["epoch"], train_log["val_13_macro_f1"], label="val_13_macro_f1")
plt.plot(train_log["epoch"], train_log["external_present_acc"], label="external_present_acc")
plt.plot(train_log["epoch"], train_log["external_present_macro_f1"], label="external_present_macro_f1")
plt.axhline(CFG.target_val_acc_min, linestyle="--", label="target_val_acc_min")
plt.axhline(CFG.target_external5_acc_min, linestyle="--", label="target_external_present_acc_min")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("P4 Classification Accuracy / Macro-F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_score_curves.png", dpi=260)
plt.close()

# Baseline comparison.
plot_df = comparison_df.copy()
plot_df = plot_df.sort_values("val_13_macro_f1", ascending=False)

plt.figure(figsize=(14, 7))
plt.bar(plot_df["model"], plot_df["val_13_macro_f1"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Validation 13-way Macro-F1")
plt.title("P4 Baseline vs BLEACH — Validation Macro-F1")
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_baseline_val_macro_f1.png", dpi=260)
plt.close()

plt.figure(figsize=(14, 7))
plt.bar(plot_df["model"], plot_df["external_present_macro_f1"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("External Present-Dialect Macro-F1")
plt.title("P4 Baseline vs BLEACH — External Present-Dialect Macro-F1")
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_baseline_external_present_macro_f1.png", dpi=260)
plt.close()

# Per-dialect PRF.
val_plot = val_per_dialect.set_index("dialect").reindex(DIALECTS).reset_index()
x = np.arange(len(DIALECTS))
w = 0.25

plt.figure(figsize=(14, 6))
plt.bar(x - w, val_plot["precision"], width=w, label="precision")
plt.bar(x, val_plot["recall"], width=w, label="recall")
plt.bar(x + w, val_plot["f1"], width=w, label="f1")
plt.xticks(x, DIALECTS, rotation=45)
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("Validation 13-way Per-Dialect Precision / Recall / F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_val_per_dialect_prf.png", dpi=260)
plt.close()

ext_plot = ext_present_per_dialect.set_index("dialect").reindex(external_present_dialects).reset_index()
x2 = np.arange(len(external_present_dialects))

plt.figure(figsize=(10, 6))
plt.bar(x2 - w, ext_plot["precision"], width=w, label="precision")
plt.bar(x2, ext_plot["recall"], width=w, label="recall")
plt.bar(x2 + w, ext_plot["f1"], width=w, label="f1")
plt.xticks(x2, external_present_dialects, rotation=45)
plt.ylim(0, 1.0)
plt.ylabel("Score")
plt.title("External Present-Dialect Precision / Recall / F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_external_present_prf.png", dpi=260)
plt.close()

# Confidence histograms.
plt.figure(figsize=(10, 6))
plt.hist(val_pred_df[val_pred_df["correct_13"] == 1]["confidence_13"], bins=30, alpha=0.65, label="correct")
plt.hist(val_pred_df[val_pred_df["correct_13"] == 0]["confidence_13"], bins=30, alpha=0.65, label="incorrect")
plt.xlabel("Confidence")
plt.ylabel("Count")
plt.title("Validation Confidence Distribution")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_val_confidence_distribution.png", dpi=260)
plt.close()

plt.figure(figsize=(10, 6))
plt.hist(ext_pred_df[ext_pred_df["correct_restricted"] == 1]["confidence_restricted"], bins=30, alpha=0.65, label="correct")
plt.hist(ext_pred_df[ext_pred_df["correct_restricted"] == 0]["confidence_restricted"], bins=30, alpha=0.65, label="incorrect")
plt.xlabel("Restricted Confidence")
plt.ylabel("Count")
plt.title("External Present-Dialect Confidence Distribution")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / "p4_external_present_confidence_distribution.png", dpi=260)
plt.close()

print("[OK] Plots saved to:", PLOT_DIR)

[OK] Plots saved to: /kaggle/working/artifacts/bleach_p4_final_classification/plots


In [36]:
# ============================================================
# PORTION 4.12 — ERROR AUDIT + FEATURE PCA
# ============================================================

val_errors = val_pred_df[val_pred_df["correct_13"] == 0].copy()
val_errors = val_errors.sort_values("confidence_13", ascending=False)
val_errors.head(500).to_csv(PRED_DIR / "p4_val_high_confidence_errors.csv", index=False)

ext_errors = ext_pred_df[ext_pred_df["correct_restricted"] == 0].copy()
ext_errors = ext_errors.sort_values("confidence_restricted", ascending=False)
ext_errors.head(500).to_csv(PRED_DIR / "p4_external_present_high_confidence_errors.csv", index=False)

val_error_pairs = (
    val_errors.groupby(["gold_dialect", "pred_13_dialect"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
val_error_pairs.to_csv(REPORT_DIR / "p4_val_error_pairs.csv", index=False)

ext_error_pairs = (
    ext_errors.groupby(["gold_dialect", "pred_restricted_dialect"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
ext_error_pairs.to_csv(REPORT_DIR / "p4_external_present_error_pairs.csv", index=False)

# PCA feature visualization.
if val_features is not None and len(val_features) == len(val_pred_df):
    n_plot = min(3000, len(val_features))
    rng = np.random.default_rng(SEED)
    idx = rng.choice(np.arange(len(val_features)), size=n_plot, replace=False)

    pca = PCA(n_components=2, random_state=SEED)
    xy = pca.fit_transform(val_features[idx])

    pca_df = pd.DataFrame({
        "pc1": xy[:, 0],
        "pc2": xy[:, 1],
        "dialect": val_pred_df.iloc[idx]["gold_dialect"].values,
        "correct": val_pred_df.iloc[idx]["correct_13"].values,
    })
    pca_df.to_csv(REPORT_DIR / "p4_val_feature_pca.csv", index=False)

    plt.figure(figsize=(10, 8))
    for d in DIALECTS:
        sub = pca_df[pca_df["dialect"] == d]
        if len(sub) == 0:
            continue
        plt.scatter(sub["pc1"], sub["pc2"], s=8, alpha=0.65, label=d)

    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.title("Validation Classifier Feature PCA by Dialect")
    plt.legend(markerscale=2, fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "p4_val_feature_pca_by_dialect.png", dpi=260)
    plt.close()

error_audit = {
    "val_high_confidence_errors": str(PRED_DIR / "p4_val_high_confidence_errors.csv"),
    "external_present_high_confidence_errors": str(PRED_DIR / "p4_external_present_high_confidence_errors.csv"),
    "val_error_pairs": str(REPORT_DIR / "p4_val_error_pairs.csv"),
    "external_present_error_pairs": str(REPORT_DIR / "p4_external_present_error_pairs.csv"),
    "top_val_error_pairs": val_error_pairs.head(20).to_dict(orient="records"),
    "top_external_present_error_pairs": ext_error_pairs.head(20).to_dict(orient="records"),
}

write_json(REPORT_DIR / "p4_error_audit_summary.json", error_audit)

print(json.dumps(error_audit, indent=2, ensure_ascii=False))

{
  "val_high_confidence_errors": "/kaggle/working/artifacts/bleach_p4_final_classification/predictions/p4_val_high_confidence_errors.csv",
  "external_present_high_confidence_errors": "/kaggle/working/artifacts/bleach_p4_final_classification/predictions/p4_external_present_high_confidence_errors.csv",
  "val_error_pairs": "/kaggle/working/artifacts/bleach_p4_final_classification/reports/p4_val_error_pairs.csv",
  "external_present_error_pairs": "/kaggle/working/artifacts/bleach_p4_final_classification/reports/p4_external_present_error_pairs.csv",
  "top_val_error_pairs": [
    {
      "gold_dialect": "STD",
      "pred_13_dialect": "MYM",
      "count": 163
    },
    {
      "gold_dialect": "NOA",
      "pred_13_dialect": "MYM",
      "count": 161
    },
    {
      "gold_dialect": "NSD",
      "pred_13_dialect": "KIS",
      "count": 133
    },
    {
      "gold_dialect": "SYL",
      "pred_13_dialect": "MYM",
      "count": 115
    },
    {
      "gold_dialect": "BAR",
      "pred_

In [37]:
# ============================================================
# PORTION 4.13 — FINAL PAPER-READY REPORT
# ============================================================

best_baseline_val_macro = float(
    comparison_df[comparison_df["model"] != "BLEACH_Dense_CE_First_Classifier"]["val_13_macro_f1"].max()
)
best_baseline_ext_macro = float(
    comparison_df[comparison_df["model"] != "BLEACH_Dense_CE_First_Classifier"]["external_present_macro_f1"].max()
)

final_metrics = {
    "val_13_acc": float(bleach_row["val_13_acc"]),
    "val_13_macro_f1": float(bleach_row["val_13_macro_f1"]),
    "val_13_weighted_f1": float(bleach_row["val_13_weighted_f1"]),
    "external_13_acc": float(bleach_row["external_13_acc"]),
    "external_13_macro_f1": float(bleach_row["external_13_macro_f1"]),
    "external_present_acc": float(bleach_row["external_present_acc"]),
    "external_present_macro_f1": float(bleach_row["external_present_macro_f1"]),
    "external_present_weighted_f1": float(bleach_row["external_present_weighted_f1"]),
    "external_present_ece": float(bleach_row["external_present_ece"]),
}

acceptance = {
    "val_acc_min_met": bool(final_metrics["val_13_acc"] >= CFG.target_val_acc_min),
    "val_macro_f1_min_met": bool(final_metrics["val_13_macro_f1"] >= CFG.target_val_macro_f1_min),
    "external_present_acc_min_met": bool(final_metrics["external_present_acc"] >= CFG.target_external5_acc_min),
    "external_present_macro_f1_min_met": bool(final_metrics["external_present_macro_f1"] >= CFG.target_external5_macro_f1_min),
    "beats_best_baseline_val_macro_f1": bool(final_metrics["val_13_macro_f1"] >= best_baseline_val_macro),
    "beats_best_baseline_external_present_macro_f1": bool(final_metrics["external_present_macro_f1"] >= best_baseline_ext_macro),
}

final_report = {
    "portion": "BLEACH Portion 4",
    "task": "classification_only",
    "status": "complete",
    "selected_dense_checkpoint": SELECTED_DENSE,
    "protocol": {
        "validation": "13-way dialect classification",
        "external": "13-way plus restricted present-dialect evaluation",
        "external_present_dialects": external_present_dialects,
        "no_translation_in_this_portion": True,
        "no_previous_classifier_checkpoint_used": True,
    },
    "final_bleach_metrics": final_metrics,
    "best_baseline_val_macro_f1": best_baseline_val_macro,
    "best_baseline_external_present_macro_f1": best_baseline_ext_macro,
    "acceptance": acceptance,
    "outputs": {
        "baseline_results": str(REPORT_DIR / "p4_baseline_results.csv"),
        "baseline_vs_bleach": str(REPORT_DIR / "p4_baseline_vs_bleach_results.csv"),
        "train_log": str(REPORT_DIR / "p4_neural_train_log.csv"),
        "per_dialect_metrics": str(REPORT_DIR / "p4_per_dialect_metrics.csv"),
        "val_predictions": str(PRED_DIR / "p4_val_predictions.csv"),
        "external_predictions": str(PRED_DIR / "p4_external_predictions.csv"),
        "error_audit": str(REPORT_DIR / "p4_error_audit_summary.json"),
        "best_classifier": str(CKPT_DIR / "best_p4_bleach_classifier.pt"),
    },
    "plots": {
        "val_13_confusion": str(PLOT_DIR / "p4_val_13way_confusion_normalized.png"),
        "external_present_confusion": str(PLOT_DIR / "p4_external_present_confusion_normalized.png"),
        "score_curves": str(PLOT_DIR / "p4_score_curves.png"),
        "baseline_val_macro_f1": str(PLOT_DIR / "p4_baseline_val_macro_f1.png"),
        "baseline_external_present_macro_f1": str(PLOT_DIR / "p4_baseline_external_present_macro_f1.png"),
        "val_prf": str(PLOT_DIR / "p4_val_per_dialect_prf.png"),
        "external_present_prf": str(PLOT_DIR / "p4_external_present_prf.png"),
        "feature_pca": str(PLOT_DIR / "p4_val_feature_pca_by_dialect.png"),
    },
}

write_json(REPORT_DIR / "p4_final_classification_report.json", final_report)

print("=" * 100)
print("PORTION 4 FINAL CHECKPOINT")
print("=" * 100)
print(f"Validation 13-way accuracy       : {final_metrics['val_13_acc']:.4f}")
print(f"Validation 13-way macro-F1       : {final_metrics['val_13_macro_f1']:.4f}")
print(f"External 13-way accuracy         : {final_metrics['external_13_acc']:.4f}")
print(f"External present accuracy        : {final_metrics['external_present_acc']:.4f}")
print(f"External present macro-F1        : {final_metrics['external_present_macro_f1']:.4f}")
print(f"Best baseline val macro-F1       : {best_baseline_val_macro:.4f}")
print(f"Best baseline external macro-F1  : {best_baseline_ext_macro:.4f}")
print("Acceptance:")
print(json.dumps(acceptance, indent=2))
print("Final report:", REPORT_DIR / "p4_final_classification_report.json")
print("=" * 100)

print("\nPaste these after running Portion 4:")
print("1. reports/p4_final_classification_report.json")
print("2. reports/p4_baseline_vs_bleach_results.csv")
print("3. reports/p4_per_dialect_metrics.csv")
print("4. reports/p4_val_error_pairs.csv")
print("5. reports/p4_external_present_error_pairs.csv")

PORTION 4 FINAL CHECKPOINT
Validation 13-way accuracy       : 0.5898
Validation 13-way macro-F1       : 0.5930
External 13-way accuracy         : 0.1892
External present accuracy        : 0.2094
External present macro-F1        : 0.1253
Best baseline val macro-F1       : 0.7100
Best baseline external macro-F1  : 0.1615
Acceptance:
{
  "val_acc_min_met": false,
  "val_macro_f1_min_met": false,
  "external_present_acc_min_met": false,
  "external_present_macro_f1_min_met": false,
  "beats_best_baseline_val_macro_f1": false,
  "beats_best_baseline_external_present_macro_f1": false
}
Final report: /kaggle/working/artifacts/bleach_p4_final_classification/reports/p4_final_classification_report.json

Paste these after running Portion 4:
1. reports/p4_final_classification_report.json
2. reports/p4_baseline_vs_bleach_results.csv
3. reports/p4_per_dialect_metrics.csv
4. reports/p4_val_error_pairs.csv
5. reports/p4_external_present_error_pairs.csv


## final Portion 4B: advanced lexical dialect classification, ensemble, 5-way external classifier, interpretability, and paper-ready exports.

In [38]:
# ============================================================
# PORTION 4B.1 — ADVANCED LEXICAL CLASSIFIER SETUP
# Final classification-only module after P4/P4A-v3
# ============================================================

from __future__ import annotations

import os
import re
import gc
import json
import time
import math
import random
import warnings
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, asdict
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.special import softmax

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
)
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.base import clone
import joblib

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DIALECTS = ["BAR", "CHI", "KHU", "KIS", "MYM", "NAR", "NOA", "NSD", "RAJ", "RAN", "STD", "SYL", "TAN"]
DIALECT_TO_ID = {d: i for i, d in enumerate(DIALECTS)}
ID_TO_DIALECT = {i: d for d, i in DIALECT_TO_ID.items()}

@dataclass(frozen=True)
class P4BConfig:
    output_dir: str = "/kaggle/working/artifacts/bleach_p4b_advanced_lexical"

    # Search strength.
    run_mode: str = "strong"  # "fast" or "strong"

    # Feature caps.
    max_features_char_small: int = 200_000
    max_features_char_mid: int = 300_000
    max_features_char_large: int = 500_000
    max_features_word: int = 150_000

    # Logistic regression.
    logreg_max_iter: int = 3000
    logreg_solver: str = "saga"

    # Ensemble.
    svm_temperature_grid: Tuple[float, ...] = (0.50, 0.75, 1.00, 1.25, 1.50, 2.00)
    ensemble_weight_step: float = 0.10

    # Reporting.
    top_k_ngrams: int = 80
    high_conf_error_k: int = 500

    # Acceptance.
    current_best_val_macro_f1: float = 0.710001
    target_val_macro_f1_min: float = 0.72
    target_val_macro_f1_strong: float = 0.74
    current_best_external_macro_f1: float = 0.161504


CFG4B = P4BConfig()

OUT_DIR_4B = Path(CFG4B.output_dir)
CONFIG_DIR_4B = OUT_DIR_4B / "config"
REPORT_DIR_4B = OUT_DIR_4B / "reports"
PLOT_DIR_4B = OUT_DIR_4B / "plots"
PRED_DIR_4B = OUT_DIR_4B / "predictions"
MODEL_DIR_4B = OUT_DIR_4B / "models"

for p in [OUT_DIR_4B, CONFIG_DIR_4B, REPORT_DIR_4B, PLOT_DIR_4B, PRED_DIR_4B, MODEL_DIR_4B]:
    p.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


# ---------- recovery helpers if notebook variables are missing ----------
DIALECT_TAG_RE = re.compile(r"<dial:[A-Za-z0-9_\-]+>")
BENGALI_RE = re.compile(r"[\u0980-\u09FF]")


def normalize_text_for_cls(x: Any) -> str:
    if pd.isna(x):
        return ""
    s = str(x)
    s = unicodedata.normalize("NFC", s)
    s = DIALECT_TAG_RE.sub(" ", s)
    s = re.sub(r"[\x00-\x1f\x7f-\x9f]", "", s)
    s = re.sub(r"[\u200b\u200c\u200d\u200e\u200f\ufeff]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def has_bangla(s: str) -> bool:
    return bool(BENGALI_RE.search(str(s)))


def select_text_column(df: pd.DataFrame) -> str:
    preferred = ["p4_text", "text", "body_text", "clean_text", "normalized_text", "sentence", "utterance", "content"]
    for c in preferred:
        if c in df.columns:
            return c
    obj_cols = [c for c in df.columns if df[c].dtype == "object"]
    if not obj_cols:
        raise ValueError("No text-like column found.")
    return max(obj_cols, key=lambda c: df[c].dropna().astype(str).map(len).mean())


def resolve_slm_p1_root() -> Path:
    candidates = [
        Path("/kaggle/working/artifacts/slm_p1_data"),
        Path("artifacts/slm_p1_data"),
    ]

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for m in sorted(input_root.rglob("slm_p1_data/splits/model_train_rows.parquet")):
            candidates.append(m.parents[1])
        for m in sorted(input_root.rglob("artifacts/slm_p1_data/splits/model_train_rows.parquet")):
            candidates.append(m.parents[2])

    for c in candidates:
        if (c / "splits" / "model_train_rows.parquet").exists():
            return c

    raise FileNotFoundError("Could not resolve SLM Portion-1 root.")


def load_split_rows_if_needed(path: Path, split_name: str) -> pd.DataFrame:
    df = pd.read_parquet(path)
    assert "dialect" in df.columns, f"{split_name} missing dialect column."

    text_col = select_text_column(df)

    df = df.copy()
    if "p4_text" not in df.columns:
        df["p4_text"] = df[text_col].map(normalize_text_for_cls)
    else:
        df["p4_text"] = df["p4_text"].map(normalize_text_for_cls)

    df["p4_len"] = df["p4_text"].str.len()
    df["p4_has_bangla"] = df["p4_text"].map(has_bangla)
    df["p4_split"] = split_name

    df = df[df["dialect"].isin(DIALECTS)].copy()
    df = df[df["p4_len"] > 0].copy()
    df = df[df["p4_has_bangla"]].copy()
    df = df.drop_duplicates(subset=["p4_text", "dialect"]).reset_index(drop=True)
    return df


# If current notebook already has train_df/val_df/external_df, use them.
missing_dfs = not all(name in globals() for name in ["train_df", "val_df", "external_df"])

if missing_dfs:
    SLM_P1_ROOT = resolve_slm_p1_root()
    train_df = load_split_rows_if_needed(SLM_P1_ROOT / "splits" / "model_train_rows.parquet", "train")
    val_df = load_split_rows_if_needed(SLM_P1_ROOT / "splits" / "model_val_balanced_rows.parquet", "val")
    external_df = load_split_rows_if_needed(SLM_P1_ROOT / "splits" / "model_eval_holdout_rows.parquet", "external")
else:
    for df_name in ["train_df", "val_df", "external_df"]:
        df = globals()[df_name]
        if "p4_text" not in df.columns:
            text_col = select_text_column(df)
            df["p4_text"] = df[text_col].map(normalize_text_for_cls)
        else:
            df["p4_text"] = df["p4_text"].map(normalize_text_for_cls)
        globals()[df_name] = df[df["dialect"].isin(DIALECTS)].copy().reset_index(drop=True)

external_present_dialects = sorted(external_df["dialect"].unique().tolist())
external_present_ids = [DIALECT_TO_ID[d] for d in external_present_dialects]

X_train = train_df["p4_text"].tolist()
y_train = train_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

X_val = val_df["p4_text"].tolist()
y_val = val_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

X_ext = external_df["p4_text"].tolist()
y_ext = external_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

write_json(CONFIG_DIR_4B / "p4b_config.json", asdict(CFG4B))
write_json(CONFIG_DIR_4B / "p4b_data_protocol.json", {
    "train_rows": int(len(train_df)),
    "val_rows": int(len(val_df)),
    "external_rows": int(len(external_df)),
    "train_dialects": sorted(train_df["dialect"].unique().tolist()),
    "val_dialects": sorted(val_df["dialect"].unique().tolist()),
    "external_present_dialects": external_present_dialects,
    "external_present_ids": external_present_ids,
})

print("=" * 100)
print("BLEACH PORTION 4B — ADVANCED LEXICAL DIALECT CLASSIFIER")
print("=" * 100)
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("External rows:", len(external_df))
print("External present dialects:", external_present_dialects)
print("Output:", OUT_DIR_4B)
print("=" * 100)

BLEACH PORTION 4B — ADVANCED LEXICAL DIALECT CLASSIFIER
Train rows: 113594
Val rows: 6500
External rows: 2875
External present dialects: ['BAR', 'CHI', 'MYM', 'NOA', 'SYL']
Output: /kaggle/working/artifacts/bleach_p4b_advanced_lexical


In [39]:
# ============================================================
# PORTION 4B.2 — METRICS + PREDICTION UTILITIES
# ============================================================

def expected_calibration_error(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = pred == y_true

    ece = 0.0
    edges = np.linspace(0.0, 1.0, n_bins + 1)

    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        mask = (conf > lo) & (conf <= hi)
        if not np.any(mask):
            continue
        ece += abs(correct[mask].mean() - conf[mask].mean()) * mask.mean()

    return float(ece)


def compute_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    labels: List[int],
    label_names: List[str],
    probs: Optional[np.ndarray] = None,
) -> Dict[str, Any]:
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, labels=labels, average="weighted", zero_division=0)),
        "mcc": float(matthews_corrcoef(y_true, y_pred)) if len(np.unique(y_true)) > 1 else 0.0,
        "ece": expected_calibration_error(probs, y_true) if probs is not None else None,
        "classification_report": classification_report(
            y_true,
            y_pred,
            labels=labels,
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels),
    }


def restrict_probs(probs_13: np.ndarray, allowed_ids: List[int]) -> Tuple[np.ndarray, np.ndarray]:
    p = probs_13[:, allowed_ids].copy()
    p = p / np.maximum(p.sum(axis=1, keepdims=True), 1e-12)
    pred_local = p.argmax(axis=1)
    pred_global = np.asarray([allowed_ids[i] for i in pred_local], dtype=np.int64)
    return p, pred_global


def evaluate_probs(
    y_true: np.ndarray,
    probs_13: np.ndarray,
    split: str,
    restricted_ids: Optional[List[int]] = None,
) -> Dict[str, Any]:
    pred_13 = probs_13.argmax(axis=1)

    out = {
        f"{split}_13way": compute_metrics(
            y_true=y_true,
            y_pred=pred_13,
            labels=list(range(len(DIALECTS))),
            label_names=DIALECTS,
            probs=probs_13,
        )
    }

    if restricted_ids is not None:
        restricted_p, restricted_pred = restrict_probs(probs_13, restricted_ids)
        restricted_names = [ID_TO_DIALECT[i] for i in restricted_ids]
        out[f"{split}_restricted"] = compute_metrics(
            y_true=y_true,
            y_pred=restricted_pred,
            labels=restricted_ids,
            label_names=restricted_names,
            probs=restricted_p,
        )

    return out


def metric_dump_to_json(metrics: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in metrics.items():
        out[k] = {}
        for kk, vv in v.items():
            if kk == "confusion_matrix":
                out[k][kk] = vv.tolist()
            else:
                out[k][kk] = vv
    return out


def decision_scores_to_probs(scores: np.ndarray, temperature: float = 1.0) -> np.ndarray:
    scores = np.asarray(scores, dtype=np.float64)

    if scores.ndim == 1:
        scores = np.stack([-scores, scores], axis=1)

    return softmax(scores / max(temperature, 1e-8), axis=1).astype(np.float32)


def align_local_probs(
    probs_local: np.ndarray,
    classes: np.ndarray,
    n_classes: int = len(DIALECTS),
) -> np.ndarray:
    probs_full = np.zeros((probs_local.shape[0], n_classes), dtype=np.float32)
    for j, c in enumerate(classes):
        probs_full[:, int(c)] = probs_local[:, j]
    return probs_full


def predict_proba_full(model, X: List[str], temperature: float = 1.0, n_classes: int = len(DIALECTS)) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        probs_local = model.predict_proba(X)
        classes = getattr(model, "classes_", None)

        if classes is None and hasattr(model, "named_steps"):
            classes = model.named_steps["clf"].classes_

        return align_local_probs(probs_local, np.asarray(classes), n_classes=n_classes)

    if hasattr(model, "decision_function"):
        scores = model.decision_function(X)
        probs_local = decision_scores_to_probs(scores, temperature=temperature)

        classes = getattr(model, "classes_", None)
        if classes is None and hasattr(model, "named_steps"):
            classes = model.named_steps["clf"].classes_

        return align_local_probs(probs_local, np.asarray(classes), n_classes=n_classes)

    pred = model.predict(X)
    probs = np.full((len(pred), n_classes), 1e-6, dtype=np.float32)
    probs[np.arange(len(pred)), pred] = 1.0
    probs = probs / probs.sum(axis=1, keepdims=True)
    return probs


def summarize_model_result(
    name: str,
    val_probs: np.ndarray,
    ext_probs: np.ndarray,
    extra: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    val_metrics = evaluate_probs(y_val, val_probs, "val", restricted_ids=None)
    ext_metrics = evaluate_probs(y_ext, ext_probs, "external", restricted_ids=external_present_ids)

    row = {
        "model": name,
        "val_13_acc": val_metrics["val_13way"]["accuracy"],
        "val_13_macro_f1": val_metrics["val_13way"]["macro_f1"],
        "val_13_weighted_f1": val_metrics["val_13way"]["weighted_f1"],
        "val_13_mcc": val_metrics["val_13way"]["mcc"],
        "external_13_acc": ext_metrics["external_13way"]["accuracy"],
        "external_13_macro_f1": ext_metrics["external_13way"]["macro_f1"],
        "external_present_acc": ext_metrics["external_restricted"]["accuracy"],
        "external_present_macro_f1": ext_metrics["external_restricted"]["macro_f1"],
        "external_present_weighted_f1": ext_metrics["external_restricted"]["weighted_f1"],
        "external_present_mcc": ext_metrics["external_restricted"]["mcc"],
        "external_present_ece": ext_metrics["external_restricted"]["ece"],
    }

    if extra:
        row.update(extra)

    return {
        "row": row,
        "val_metrics": val_metrics,
        "external_metrics": ext_metrics,
    }


def make_prediction_df(split: str, texts: List[str], y_true: np.ndarray, probs: np.ndarray, restricted_ids: Optional[List[int]] = None) -> pd.DataFrame:
    pred13 = probs.argmax(axis=1)

    df = pd.DataFrame({
        "split": split,
        "gold_id": y_true,
        "gold_dialect": [ID_TO_DIALECT[int(i)] for i in y_true],
        "pred_13_id": pred13,
        "pred_13_dialect": [ID_TO_DIALECT[int(i)] for i in pred13],
        "correct_13": (pred13 == y_true).astype(int),
        "confidence_13": probs.max(axis=1),
        "text": texts,
    })

    if restricted_ids is not None:
        p_res, pred_res = restrict_probs(probs, restricted_ids)
        df["pred_restricted_id"] = pred_res
        df["pred_restricted_dialect"] = [ID_TO_DIALECT[int(i)] for i in pred_res]
        df["correct_restricted"] = (pred_res == y_true).astype(int)
        df["confidence_restricted"] = p_res.max(axis=1)

    for i, d in enumerate(DIALECTS):
        df[f"prob_{d}"] = probs[:, i]

    return df


print("[OK] P4B metric utilities ready.")

[OK] P4B metric utilities ready.


In [40]:
# ============================================================
# PORTION 4B.3 — ADVANCED TF-IDF HYPERPARAMETER SEARCH
# Focus: improve char TF-IDF SVM / LogReg beyond 0.710 macro-F1
# ============================================================

def make_char_svm_pipeline(
    analyzer: str,
    ngram_range: Tuple[int, int],
    max_features: int,
    min_df: int,
    sublinear_tf: bool,
    C: float,
) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            sublinear_tf=sublinear_tf,
            lowercase=False,
            dtype=np.float32,
        )),
        ("clf", LinearSVC(
            C=C,
            class_weight="balanced",
            random_state=SEED,
            max_iter=5000,
        )),
    ])


def make_char_logreg_pipeline(
    analyzer: str,
    ngram_range: Tuple[int, int],
    max_features: int,
    min_df: int,
    sublinear_tf: bool,
    C: float,
) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            sublinear_tf=sublinear_tf,
            lowercase=False,
            dtype=np.float32,
        )),
        ("clf", LogisticRegression(
            C=C,
            max_iter=CFG4B.logreg_max_iter,
            class_weight="balanced",
            solver=CFG4B.logreg_solver,
            multi_class="auto",
            n_jobs=-1,
            random_state=SEED,
            verbose=0,
        )),
    ])


def make_word_logreg_pipeline(
    ngram_range: Tuple[int, int],
    max_features: int,
    min_df: int,
    sublinear_tf: bool,
    C: float,
) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer="word",
            token_pattern=r"(?u)\b\w+\b",
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            sublinear_tf=sublinear_tf,
            lowercase=False,
            dtype=np.float32,
        )),
        ("clf", LogisticRegression(
            C=C,
            max_iter=CFG4B.logreg_max_iter,
            class_weight="balanced",
            solver=CFG4B.logreg_solver,
            multi_class="auto",
            n_jobs=-1,
            random_state=SEED,
            verbose=0,
        )),
    ])


def make_combined_logreg_pipeline(
    char_ngram: Tuple[int, int],
    word_ngram: Tuple[int, int],
    char_max_features: int,
    word_max_features: int,
    min_df: int,
    sublinear_tf: bool,
    C: float,
) -> Pipeline:
    return Pipeline([
        ("features", FeatureUnion([
            ("char", TfidfVectorizer(
                analyzer="char",
                ngram_range=char_ngram,
                max_features=char_max_features,
                min_df=min_df,
                sublinear_tf=sublinear_tf,
                lowercase=False,
                dtype=np.float32,
            )),
            ("word", TfidfVectorizer(
                analyzer="word",
                token_pattern=r"(?u)\b\w+\b",
                ngram_range=word_ngram,
                max_features=word_max_features,
                min_df=min_df,
                sublinear_tf=sublinear_tf,
                lowercase=False,
                dtype=np.float32,
            )),
        ])),
        ("clf", LogisticRegression(
            C=C,
            max_iter=CFG4B.logreg_max_iter,
            class_weight="balanced",
            solver=CFG4B.logreg_solver,
            multi_class="auto",
            n_jobs=-1,
            random_state=SEED,
            verbose=0,
        )),
    ])


def make_complement_nb_pipeline(
    analyzer: str,
    ngram_range: Tuple[int, int],
    max_features: int,
    min_df: int,
    alpha: float,
) -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            analyzer=analyzer,
            ngram_range=ngram_range,
            max_features=max_features,
            min_df=min_df,
            sublinear_tf=True,
            lowercase=False,
            dtype=np.float32,
        )),
        ("clf", ComplementNB(alpha=alpha)),
    ])


candidate_specs = []

# Strong SVM candidates.
svm_grid = [
    ("char", (2, 5), CFG4B.max_features_char_mid, 2, True, 0.5),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 0.5),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_large, 2, True, 1.0),
    ("char", (2, 7), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char", (3, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char", (3, 7), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 1, True, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_large, 1, True, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 3, True, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, False, 1.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 2.0),
    ("char", (2, 7), CFG4B.max_features_char_large, 2, True, 2.0),
    ("char_wb", (3, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char_wb", (3, 7), CFG4B.max_features_char_mid, 2, True, 1.0),
    ("char_wb", (2, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
]

if CFG4B.run_mode == "fast":
    svm_grid = svm_grid[:8]

for i, spec in enumerate(svm_grid):
    analyzer, ngr, maxf, mindf, subtf, C = spec
    candidate_specs.append({
        "name": f"svm_{i:02d}_{analyzer}_{ngr[0]}{ngr[1]}_mf{maxf//1000}k_df{mindf}_sub{subtf}_C{C}",
        "family": "char_svm",
        "build": lambda analyzer=analyzer, ngr=ngr, maxf=maxf, mindf=mindf, subtf=subtf, C=C: make_char_svm_pipeline(
            analyzer, ngr, maxf, mindf, subtf, C
        ),
    })


# Logistic candidates.
logreg_grid = [
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 2.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 4.0),
    ("char", (2, 6), CFG4B.max_features_char_large, 2, True, 4.0),
    ("char", (2, 7), CFG4B.max_features_char_mid, 2, True, 4.0),
    ("char", (3, 7), CFG4B.max_features_char_mid, 2, True, 4.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 1, True, 4.0),
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 8.0),
    ("char_wb", (3, 6), CFG4B.max_features_char_mid, 2, True, 4.0),
    ("char_wb", (3, 7), CFG4B.max_features_char_mid, 2, True, 4.0),
]

if CFG4B.run_mode == "fast":
    logreg_grid = logreg_grid[:4]

for i, spec in enumerate(logreg_grid):
    analyzer, ngr, maxf, mindf, subtf, C = spec
    candidate_specs.append({
        "name": f"logreg_{i:02d}_{analyzer}_{ngr[0]}{ngr[1]}_mf{maxf//1000}k_df{mindf}_sub{subtf}_C{C}",
        "family": "char_logreg",
        "build": lambda analyzer=analyzer, ngr=ngr, maxf=maxf, mindf=mindf, subtf=subtf, C=C: make_char_logreg_pipeline(
            analyzer, ngr, maxf, mindf, subtf, C
        ),
    })


# Word and combined candidates for external robustness.
word_grid = [
    ((1, 2), CFG4B.max_features_word, 2, True, 2.0),
    ((1, 3), CFG4B.max_features_word, 2, True, 4.0),
    ((1, 4), CFG4B.max_features_word, 2, True, 4.0),
    ((1, 3), CFG4B.max_features_word, 1, True, 4.0),
]

for i, spec in enumerate(word_grid):
    ngr, maxf, mindf, subtf, C = spec
    candidate_specs.append({
        "name": f"word_logreg_{i:02d}_{ngr[0]}{ngr[1]}_mf{maxf//1000}k_df{mindf}_C{C}",
        "family": "word_logreg",
        "build": lambda ngr=ngr, maxf=maxf, mindf=mindf, subtf=subtf, C=C: make_word_logreg_pipeline(
            ngr, maxf, mindf, subtf, C
        ),
    })


combined_grid = [
    ((2, 6), (1, 2), 250_000, 80_000, 2, True, 4.0),
    ((2, 6), (1, 3), 250_000, 100_000, 2, True, 4.0),
    ((2, 7), (1, 3), 300_000, 100_000, 2, True, 4.0),
]

if CFG4B.run_mode == "fast":
    combined_grid = combined_grid[:1]

for i, spec in enumerate(combined_grid):
    cngr, wngr, cmax, wmax, mindf, subtf, C = spec
    candidate_specs.append({
        "name": f"combined_logreg_{i:02d}_c{cngr[0]}{cngr[1]}_w{wngr[0]}{wngr[1]}_C{C}",
        "family": "combined_logreg",
        "build": lambda cngr=cngr, wngr=wngr, cmax=cmax, wmax=wmax, mindf=mindf, subtf=subtf, C=C: make_combined_logreg_pipeline(
            cngr, wngr, cmax, wmax, mindf, subtf, C
        ),
    })


# ComplementNB: often useful as a cheap lexical sanity baseline.
nb_grid = [
    ("char", (2, 6), CFG4B.max_features_char_mid, 2, 0.1),
    ("char", (3, 7), CFG4B.max_features_char_mid, 2, 0.1),
    ("char_wb", (3, 6), CFG4B.max_features_char_mid, 2, 0.1),
]

if CFG4B.run_mode == "fast":
    nb_grid = nb_grid[:1]

for i, spec in enumerate(nb_grid):
    analyzer, ngr, maxf, mindf, alpha = spec
    candidate_specs.append({
        "name": f"cnb_{i:02d}_{analyzer}_{ngr[0]}{ngr[1]}_alpha{alpha}",
        "family": "complement_nb",
        "build": lambda analyzer=analyzer, ngr=ngr, maxf=maxf, mindf=mindf, alpha=alpha: make_complement_nb_pipeline(
            analyzer, ngr, maxf, mindf, alpha
        ),
    })


print("Total P4B candidates:", len(candidate_specs))
print("Run mode:", CFG4B.run_mode)

Total P4B candidates: 35
Run mode: strong


In [41]:
# ============================================================
# PORTION 4B.4 — TRAIN/EVALUATE LEXICAL CANDIDATES
# ============================================================

candidate_rows = []
candidate_metrics = {}
candidate_models = {}
candidate_val_probs = {}
candidate_ext_probs = {}

start_all = time.time()

for idx, spec in enumerate(candidate_specs, start=1):
    name = spec["name"]
    family = spec["family"]

    print("\n" + "=" * 100)
    print(f"[{idx}/{len(candidate_specs)}] Training candidate: {name}")
    print("=" * 100)

    start = time.time()
    model = spec["build"]()
    model.fit(X_train, y_train)

    # For SVM, tune softmax temperature on validation.
    best_temp = 1.0
    best_temp_score = -1.0

    if family == "char_svm":
        for temp in CFG4B.svm_temperature_grid:
            val_probs_tmp = predict_proba_full(model, X_val, temperature=temp)
            val_pred_tmp = val_probs_tmp.argmax(axis=1)
            score = f1_score(y_val, val_pred_tmp, labels=list(range(len(DIALECTS))), average="macro", zero_division=0)

            if score > best_temp_score:
                best_temp_score = score
                best_temp = temp

    val_probs = predict_proba_full(model, X_val, temperature=best_temp)
    ext_probs = predict_proba_full(model, X_ext, temperature=best_temp)

    result = summarize_model_result(
        name=name,
        val_probs=val_probs,
        ext_probs=ext_probs,
        extra={
            "family": family,
            "temperature": best_temp,
            "train_sec": time.time() - start,
        },
    )

    candidate_rows.append(result["row"])
    candidate_metrics[name] = {
        "val": metric_dump_to_json(result["val_metrics"]),
        "external": metric_dump_to_json(result["external_metrics"]),
    }
    candidate_models[name] = model
    candidate_val_probs[name] = val_probs
    candidate_ext_probs[name] = ext_probs

    # Save only promising models to disk to control space.
    if result["row"]["val_13_macro_f1"] >= 0.68 or result["row"]["external_present_macro_f1"] >= 0.14:
        joblib.dump(model, MODEL_DIR_4B / f"{name}.joblib")

    candidate_df = pd.DataFrame(candidate_rows).sort_values("val_13_macro_f1", ascending=False)
    candidate_df.to_csv(REPORT_DIR_4B / "p4b_candidate_results_live.csv", index=False)

    print(pd.DataFrame([result["row"]]).to_string(index=False))


candidate_df = pd.DataFrame(candidate_rows)
candidate_df = candidate_df.sort_values(
    ["val_13_macro_f1", "external_present_macro_f1"],
    ascending=[False, False],
).reset_index(drop=True)

candidate_df.to_csv(REPORT_DIR_4B / "p4b_candidate_results.csv", index=False)
write_json(REPORT_DIR_4B / "p4b_candidate_metric_dump.json", candidate_metrics)

print("=" * 100)
print("P4B CANDIDATE SEARCH COMPLETE")
print("Total search sec:", time.time() - start_all)
print("=" * 100)
print(candidate_df.head(20).to_string(index=False))


[1/35] Training candidate: svm_00_char_25_mf300k_df2_subTrue_C0.5
                                 model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  val_13_mcc  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_mcc  external_present_ece   family  temperature  train_sec
svm_00_char_25_mf300k_df2_subTrue_C0.5    0.704923         0.709912            0.709912    0.681759            0.176              0.047123              0.210783                   0.140459                      0.149863             -0.004399              0.404961 char_svm          0.5  14.963083

[2/35] Training candidate: svm_01_char_26_mf300k_df2_subTrue_C0.5
                                 model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  val_13_mcc  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_mcc  external_present_ece   family  temp

In [42]:
# ============================================================
# PORTION 4B.5 — SELECT BEST MODELS
# ============================================================

best_val_name = candidate_df.iloc[0]["model"]
best_val_model = candidate_models[best_val_name]
best_val_val_probs = candidate_val_probs[best_val_name]
best_val_ext_probs = candidate_ext_probs[best_val_name]

best_ext_row = candidate_df.sort_values(
    ["external_present_macro_f1", "val_13_macro_f1"],
    ascending=[False, False],
).iloc[0]
best_ext_name = best_ext_row["model"]
best_ext_model = candidate_models[best_ext_name]
best_ext_val_probs = candidate_val_probs[best_ext_name]
best_ext_ext_probs = candidate_ext_probs[best_ext_name]

selection_summary = {
    "best_validation_model": best_val_name,
    "best_validation_val_macro_f1": float(candidate_df.iloc[0]["val_13_macro_f1"]),
    "best_validation_val_acc": float(candidate_df.iloc[0]["val_13_acc"]),
    "best_external_present_model": best_ext_name,
    "best_external_present_macro_f1": float(best_ext_row["external_present_macro_f1"]),
    "best_external_present_acc": float(best_ext_row["external_present_acc"]),
    "previous_best_val_macro_f1": CFG4B.current_best_val_macro_f1,
    "previous_best_external_macro_f1": CFG4B.current_best_external_macro_f1,
    "val_improved_over_previous": bool(candidate_df.iloc[0]["val_13_macro_f1"] > CFG4B.current_best_val_macro_f1),
    "external_improved_over_previous": bool(best_ext_row["external_present_macro_f1"] > CFG4B.current_best_external_macro_f1),
}

write_json(REPORT_DIR_4B / "p4b_model_selection_summary.json", selection_summary)

joblib.dump(best_val_model, MODEL_DIR_4B / "best_val_13way_lexical_model.joblib")
joblib.dump(best_ext_model, MODEL_DIR_4B / "best_external_present_lexical_model.joblib")

print(json.dumps(selection_summary, indent=2, ensure_ascii=False))

{
  "best_validation_model": "svm_15_char_wb_26_mf300k_df2_subTrue_C1.0",
  "best_validation_val_macro_f1": 0.7219447674172714,
  "best_validation_val_acc": 0.7173846153846154,
  "best_external_present_model": "word_logreg_00_12_mf150k_df2_C2.0",
  "best_external_present_macro_f1": 0.15168832538019736,
  "best_external_present_acc": 0.20973913043478262,
  "previous_best_val_macro_f1": 0.710001,
  "previous_best_external_macro_f1": 0.161504,
  "val_improved_over_previous": true,
  "external_improved_over_previous": false
}


In [43]:
# ============================================================
# PORTION 4B.6 — SEPARATE EXTERNAL-PRESENT 5-WAY CLASSIFIER
# Valid because it trains only on train subset for external-present dialects.
# ============================================================

train_present_df = train_df[train_df["dialect"].isin(external_present_dialects)].copy().reset_index(drop=True)
val_present_df = val_df[val_df["dialect"].isin(external_present_dialects)].copy().reset_index(drop=True)

X_train_p = train_present_df["p4_text"].tolist()
y_train_p = train_present_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

X_val_p = val_present_df["p4_text"].tolist()
y_val_p = val_present_df["dialect"].map(DIALECT_TO_ID).values.astype(np.int64)

present_specs = [
    {
        "name": "present5_char_svm_26_C1",
        "family": "present5_svm",
        "build": lambda: make_char_svm_pipeline("char", (2, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
    },
    {
        "name": "present5_char_svm_27_C2",
        "family": "present5_svm",
        "build": lambda: make_char_svm_pipeline("char", (2, 7), CFG4B.max_features_char_mid, 2, True, 2.0),
    },
    {
        "name": "present5_char_wb_svm_36_C1",
        "family": "present5_svm",
        "build": lambda: make_char_svm_pipeline("char_wb", (3, 6), CFG4B.max_features_char_mid, 2, True, 1.0),
    },
    {
        "name": "present5_word_logreg_13_C4",
        "family": "present5_logreg",
        "build": lambda: make_word_logreg_pipeline((1, 3), CFG4B.max_features_word, 2, True, 4.0),
    },
    {
        "name": "present5_combined_logreg",
        "family": "present5_combined",
        "build": lambda: make_combined_logreg_pipeline((2, 6), (1, 3), 250_000, 100_000, 2, True, 4.0),
    },
]

present_rows = []
present_models = {}
present_ext_probs = {}
present_val_probs = {}

for idx, spec in enumerate(present_specs, start=1):
    name = spec["name"]
    family = spec["family"]

    print("\n" + "=" * 100)
    print(f"[PRESENT {idx}/{len(present_specs)}] Training: {name}")
    print("=" * 100)

    start = time.time()
    model = spec["build"]()
    model.fit(X_train_p, y_train_p)

    best_temp = 1.0
    if "svm" in family:
        best_score = -1.0
        for temp in CFG4B.svm_temperature_grid:
            val_probs_tmp = predict_proba_full(model, X_val_p, temperature=temp)
            _, val_pred_tmp = restrict_probs(val_probs_tmp, external_present_ids)
            score = f1_score(y_val_p, val_pred_tmp, labels=external_present_ids, average="macro", zero_division=0)
            if score > best_score:
                best_score = score
                best_temp = temp

    val_probs = predict_proba_full(model, X_val_p, temperature=best_temp)
    ext_probs = predict_proba_full(model, X_ext, temperature=best_temp)

    val_metrics = evaluate_probs(y_val_p, val_probs, "val_present", restricted_ids=external_present_ids)
    ext_metrics = evaluate_probs(y_ext, ext_probs, "external", restricted_ids=external_present_ids)

    row = {
        "model": name,
        "family": family,
        "temperature": best_temp,
        "val_present_acc": val_metrics["val_present_restricted"]["accuracy"],
        "val_present_macro_f1": val_metrics["val_present_restricted"]["macro_f1"],
        "external_present_acc": ext_metrics["external_restricted"]["accuracy"],
        "external_present_macro_f1": ext_metrics["external_restricted"]["macro_f1"],
        "external_present_weighted_f1": ext_metrics["external_restricted"]["weighted_f1"],
        "external_present_ece": ext_metrics["external_restricted"]["ece"],
        "train_sec": time.time() - start,
    }

    present_rows.append(row)
    present_models[name] = model
    present_ext_probs[name] = ext_probs
    present_val_probs[name] = val_probs

    joblib.dump(model, MODEL_DIR_4B / f"{name}.joblib")
    print(pd.DataFrame([row]).to_string(index=False))


present_df = pd.DataFrame(present_rows).sort_values(
    ["external_present_macro_f1", "val_present_macro_f1"],
    ascending=[False, False],
).reset_index(drop=True)

present_df.to_csv(REPORT_DIR_4B / "p4b_present5_results.csv", index=False)

best_present5_name = present_df.iloc[0]["model"]
best_present5_model = present_models[best_present5_name]
best_present5_ext_probs = present_ext_probs[best_present5_name]

joblib.dump(best_present5_model, MODEL_DIR_4B / "best_present5_lexical_model.joblib")

print("=" * 100)
print("BEST PRESENT-5 MODEL")
print("=" * 100)
print(present_df.head(10).to_string(index=False))


[PRESENT 1/5] Training: present5_char_svm_26_C1
                  model       family  temperature  val_present_acc  val_present_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece  train_sec
present5_char_svm_26_C1 present5_svm          0.5           0.6492              0.660885              0.209391                   0.140625                      0.150109              0.475776   7.062774

[PRESENT 2/5] Training: present5_char_svm_27_C2
                  model       family  temperature  val_present_acc  val_present_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_ece  train_sec
present5_char_svm_27_C2 present5_svm          0.5           0.6412              0.653137              0.208696                   0.140489                      0.150639               0.52316  11.131684

[PRESENT 3/5] Training: present5_char_wb_svm_36_C1
                     model       family  tempe

In [44]:
# ============================================================
# PORTION 4B.7 — ENSEMBLE SEARCH
# Ensemble over strongest diverse lexical systems.
# ============================================================

# Pick top candidates from each family for diversity.
top_pool = []

for fam in ["char_svm", "char_logreg", "word_logreg", "combined_logreg", "complement_nb"]:
    sub = candidate_df[candidate_df["family"] == fam].copy()
    if len(sub):
        top_pool.extend(sub.head(2)["model"].tolist())

# Ensure best val and best external are included.
top_pool = list(dict.fromkeys([best_val_name, best_ext_name] + top_pool))

print("Ensemble candidate pool:")
for x in top_pool:
    print(" -", x)

ensemble_inputs = {
    name: {
        "val_probs": candidate_val_probs[name],
        "ext_probs": candidate_ext_probs[name],
        "row": candidate_df[candidate_df["model"] == name].iloc[0].to_dict(),
    }
    for name in top_pool
}

# Generate sparse simplex weights.
def generate_weight_grid(n: int, step: float = 0.10) -> List[np.ndarray]:
    values = np.arange(0.0, 1.0 + 1e-9, step)
    weights = []

    def rec(prefix, remaining, k):
        if k == 1:
            w = np.asarray(prefix + [remaining], dtype=np.float32)
            if abs(w.sum() - 1.0) < 1e-5:
                weights.append(w)
            return
        for v in values:
            if v <= remaining + 1e-9:
                rec(prefix + [float(v)], remaining - float(v), k - 1)

    rec([], 1.0, n)
    return weights


# To avoid combinatorial explosion, ensemble top 3-5 models.
ensemble_model_names = top_pool[:5]
weight_grid = generate_weight_grid(len(ensemble_model_names), CFG4B.ensemble_weight_step)

ensemble_rows = []
best_ensemble = None
best_ensemble_score = -1.0

for w in weight_grid:
    if np.count_nonzero(w > 0) < 2:
        continue

    val_probs = np.zeros_like(candidate_val_probs[ensemble_model_names[0]])
    ext_probs = np.zeros_like(candidate_ext_probs[ensemble_model_names[0]])

    for wi, name in zip(w, ensemble_model_names):
        val_probs += wi * candidate_val_probs[name]
        ext_probs += wi * candidate_ext_probs[name]

    result = summarize_model_result(
        name="ensemble_tmp",
        val_probs=val_probs,
        ext_probs=ext_probs,
        extra=None,
    )

    row = result["row"]
    row["weights"] = json.dumps({name: float(wi) for name, wi in zip(ensemble_model_names, w) if wi > 0})
    row["model"] = "ensemble"

    ensemble_rows.append(row)

    score = row["val_13_macro_f1"]
    if score > best_ensemble_score:
        best_ensemble_score = score
        best_ensemble = {
            "weights": w.copy(),
            "model_names": ensemble_model_names.copy(),
            "val_probs": val_probs.copy(),
            "ext_probs": ext_probs.copy(),
            "row": row.copy(),
            "val_metrics": result["val_metrics"],
            "external_metrics": result["external_metrics"],
        }


ensemble_df = pd.DataFrame(ensemble_rows).sort_values(
    ["val_13_macro_f1", "external_present_macro_f1"],
    ascending=[False, False],
).reset_index(drop=True)

ensemble_df.to_csv(REPORT_DIR_4B / "p4b_ensemble_search_results.csv", index=False)

best_ensemble_name = "P4B_lexical_weighted_ensemble"
best_ensemble_row = best_ensemble["row"].copy()
best_ensemble_row["model"] = best_ensemble_name

write_json(REPORT_DIR_4B / "p4b_best_ensemble.json", {
    "model": best_ensemble_name,
    "model_names": best_ensemble["model_names"],
    "weights": {
        name: float(w)
        for name, w in zip(best_ensemble["model_names"], best_ensemble["weights"])
        if w > 0
    },
    "row": best_ensemble_row,
})

print("=" * 100)
print("BEST ENSEMBLE")
print("=" * 100)
print(pd.DataFrame([best_ensemble_row]).to_string(index=False))
print("Weights:", best_ensemble_row["weights"])

Ensemble candidate pool:
 - svm_15_char_wb_26_mf300k_df2_subTrue_C1.0
 - word_logreg_00_12_mf150k_df2_C2.0
 - svm_13_char_wb_36_mf300k_df2_subTrue_C1.0
 - logreg_07_char_wb_36_mf300k_df2_subTrue_C4.0
 - logreg_08_char_wb_37_mf300k_df2_subTrue_C4.0
 - word_logreg_02_14_mf150k_df2_C4.0
 - word_logreg_03_13_mf150k_df1_C4.0
 - combined_logreg_01_c26_w13_C4.0
 - combined_logreg_00_c26_w12_C4.0
 - cnb_02_char_wb_36_alpha0.1
 - cnb_00_char_26_alpha0.1
BEST ENSEMBLE
                        model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  val_13_mcc  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_mcc  external_present_ece                                                                                                                     weights
P4B_lexical_weighted_ensemble    0.719385         0.723859            0.723859     0.69775         0.186783              0.046885              0.212174         

In [45]:
# ============================================================
# PORTION 4B.8 — FINAL MODEL SELECTION
# ============================================================

final_rows = []

# Candidate rows.
final_rows.extend(candidate_df.to_dict(orient="records"))

# Present5 rows adapted to common columns.
for _, r in present_df.iterrows():
    final_rows.append({
        "model": r["model"],
        "family": r["family"],
        "val_13_acc": np.nan,
        "val_13_macro_f1": np.nan,
        "val_13_weighted_f1": np.nan,
        "external_13_acc": np.nan,
        "external_13_macro_f1": np.nan,
        "external_present_acc": r["external_present_acc"],
        "external_present_macro_f1": r["external_present_macro_f1"],
        "external_present_weighted_f1": r["external_present_weighted_f1"],
        "external_present_ece": r["external_present_ece"],
    })

# Ensemble.
final_rows.append(best_ensemble_row)

final_selection_df = pd.DataFrame(final_rows)

# Best final by validation 13-way.
valid_val_df = final_selection_df.dropna(subset=["val_13_macro_f1"]).sort_values(
    ["val_13_macro_f1", "external_present_macro_f1"],
    ascending=[False, False],
)
final_best_val_row = valid_val_df.iloc[0].to_dict()

# Best final by external present macro-F1.
valid_ext_df = final_selection_df.dropna(subset=["external_present_macro_f1"]).sort_values(
    ["external_present_macro_f1", "val_13_macro_f1"],
    ascending=[False, False],
)
final_best_ext_row = valid_ext_df.iloc[0].to_dict()

final_selection_df.to_csv(REPORT_DIR_4B / "p4b_final_model_selection_table.csv", index=False)

final_decision = {
    "selected_internal_validation_model": final_best_val_row["model"],
    "selected_internal_validation_val_macro_f1": float(final_best_val_row["val_13_macro_f1"]),
    "selected_internal_validation_val_acc": float(final_best_val_row["val_13_acc"]),
    "selected_external_present_model": final_best_ext_row["model"],
    "selected_external_present_macro_f1": float(final_best_ext_row["external_present_macro_f1"]),
    "selected_external_present_acc": float(final_best_ext_row["external_present_acc"]),
    "improved_over_current_best_val_macro_f1": bool(float(final_best_val_row["val_13_macro_f1"]) > CFG4B.current_best_val_macro_f1),
    "improved_over_current_best_external_macro_f1": bool(float(final_best_ext_row["external_present_macro_f1"]) > CFG4B.current_best_external_macro_f1),
}

write_json(REPORT_DIR_4B / "p4b_final_selection_decision.json", final_decision)

print("=" * 100)
print("P4B FINAL SELECTION")
print("=" * 100)
print(json.dumps(final_decision, indent=2, ensure_ascii=False))
print("\nTop validation models:")
print(valid_val_df.head(10).to_string(index=False))
print("\nTop external-present models:")
print(valid_ext_df.head(10).to_string(index=False))

P4B FINAL SELECTION
{
  "selected_internal_validation_model": "P4B_lexical_weighted_ensemble",
  "selected_internal_validation_val_macro_f1": 0.7238589186071713,
  "selected_internal_validation_val_acc": 0.7193846153846154,
  "selected_external_present_model": "word_logreg_00_12_mf150k_df2_C2.0",
  "selected_external_present_macro_f1": 0.15168832538019736,
  "selected_external_present_acc": 0.20973913043478262,
  "improved_over_current_best_val_macro_f1": true,
  "improved_over_current_best_external_macro_f1": false
}

Top validation models:
                                       model  val_13_acc  val_13_macro_f1  val_13_weighted_f1  val_13_mcc  external_13_acc  external_13_macro_f1  external_present_acc  external_present_macro_f1  external_present_weighted_f1  external_present_mcc  external_present_ece      family  temperature  train_sec                                                                                                                     weights
               P4B_lexic

In [46]:
# ============================================================
# PORTION 4B.9 — SAVE FINAL PREDICTIONS
# ============================================================

# Internal selected model probabilities.
if final_best_val_row["model"] == best_ensemble_name:
    selected_val_probs = best_ensemble["val_probs"]
    selected_ext_probs = best_ensemble["ext_probs"]
    selected_model_type = "ensemble"
else:
    selected_model_name = final_best_val_row["model"]
    selected_val_probs = candidate_val_probs[selected_model_name]
    selected_ext_probs = candidate_ext_probs[selected_model_name]
    selected_model_type = "single"

# External selected model probabilities.
if final_best_ext_row["model"] == best_ensemble_name:
    selected_ext_focus_probs = best_ensemble["ext_probs"]
    selected_ext_focus_type = "ensemble"
elif final_best_ext_row["model"] in present_ext_probs:
    selected_ext_focus_probs = present_ext_probs[final_best_ext_row["model"]]
    selected_ext_focus_type = "present5"
else:
    selected_ext_focus_probs = candidate_ext_probs[final_best_ext_row["model"]]
    selected_ext_focus_type = "single"

val_pred_df_4b = make_prediction_df(
    split="val",
    texts=X_val,
    y_true=y_val,
    probs=selected_val_probs,
    restricted_ids=None,
)

ext_pred_df_4b = make_prediction_df(
    split="external",
    texts=X_ext,
    y_true=y_ext,
    probs=selected_ext_probs,
    restricted_ids=external_present_ids,
)

ext_focus_pred_df_4b = make_prediction_df(
    split="external",
    texts=X_ext,
    y_true=y_ext,
    probs=selected_ext_focus_probs,
    restricted_ids=external_present_ids,
)

val_pred_df_4b.to_csv(PRED_DIR_4B / "p4b_selected_val_predictions.csv", index=False)
ext_pred_df_4b.to_csv(PRED_DIR_4B / "p4b_selected_external_predictions.csv", index=False)
ext_focus_pred_df_4b.to_csv(PRED_DIR_4B / "p4b_selected_external_focus_predictions.csv", index=False)

print("Saved:")
print(PRED_DIR_4B / "p4b_selected_val_predictions.csv")
print(PRED_DIR_4B / "p4b_selected_external_predictions.csv")
print(PRED_DIR_4B / "p4b_selected_external_focus_predictions.csv")

Saved:
/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_selected_val_predictions.csv
/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_selected_external_predictions.csv
/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_selected_external_focus_predictions.csv


In [47]:
# ============================================================
# PORTION 4B.10 — TOP DIALECT-DISCRIMINATIVE N-GRAMS
# Works best for LinearSVC / LogisticRegression with a single TfidfVectorizer.
# ============================================================

def extract_top_ngrams_from_pipeline(model, model_name: str, top_k: int = 80) -> pd.DataFrame:
    rows = []

    if not hasattr(model, "named_steps"):
        return pd.DataFrame()

    # Single-vectorizer case.
    if "tfidf" not in model.named_steps or "clf" not in model.named_steps:
        return pd.DataFrame()

    vectorizer = model.named_steps["tfidf"]
    clf = model.named_steps["clf"]

    if not hasattr(vectorizer, "get_feature_names_out"):
        return pd.DataFrame()

    feature_names = np.asarray(vectorizer.get_feature_names_out())

    if not hasattr(clf, "coef_"):
        return pd.DataFrame()

    coef = clf.coef_

    # Binary can produce shape [1, F], but here normally 13-way.
    if coef.shape[0] == 1 and len(getattr(clf, "classes_", [])) == 2:
        class_ids = list(clf.classes_)
        coef_to_use = np.vstack([-coef[0], coef[0]])
    else:
        class_ids = list(clf.classes_)
        coef_to_use = coef

    for row_idx, class_id in enumerate(class_ids):
        if int(class_id) not in ID_TO_DIALECT:
            continue

        dialect = ID_TO_DIALECT[int(class_id)]
        weights = coef_to_use[row_idx]
        top_idx = np.argsort(weights)[-top_k:][::-1]

        for rank, fid in enumerate(top_idx, start=1):
            rows.append({
                "model": model_name,
                "dialect": dialect,
                "rank": rank,
                "ngram": feature_names[fid],
                "weight": float(weights[fid]),
            })

    return pd.DataFrame(rows)


# Prefer the best validation single model for interpretability.
if final_best_val_row["model"] in candidate_models:
    interp_model_name = final_best_val_row["model"]
    interp_model = candidate_models[interp_model_name]
else:
    # fallback: best char_svm candidate
    interp_model_name = candidate_df[candidate_df["family"] == "char_svm"].iloc[0]["model"]
    interp_model = candidate_models[interp_model_name]

top_ngram_df = extract_top_ngrams_from_pipeline(
    interp_model,
    model_name=interp_model_name,
    top_k=CFG4B.top_k_ngrams,
)

if len(top_ngram_df) == 0:
    print("[WARN] Could not extract top n-grams from selected model. Trying best char_svm fallback.")
    fallback_name = candidate_df[candidate_df["family"] == "char_svm"].iloc[0]["model"]
    fallback_model = candidate_models[fallback_name]
    top_ngram_df = extract_top_ngrams_from_pipeline(
        fallback_model,
        model_name=fallback_name,
        top_k=CFG4B.top_k_ngrams,
    )

top_ngram_df.to_csv(REPORT_DIR_4B / "p4b_top_ngrams_by_dialect.csv", index=False)

print("=" * 100)
print("TOP N-GRAMS BY DIALECT")
print("=" * 100)
print(top_ngram_df.groupby("dialect").head(10).to_string(index=False))

TOP N-GRAMS BY DIALECT
                                    model dialect  rank  ngram   weight
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     1    ইরর 1.882314
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     2    লও  1.881800
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     3     হর 1.859452
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     4  মানু  1.797592
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     5  মানু  1.797592
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     6   কবা  1.757676
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     7     লও 1.730652
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     8    মু  1.717358
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     9  আম্মে 1.662235
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR    10  আম্মে 1.662235
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     CHI     1     ন  5.555971
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     CHI     2     ত  2.833235
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0

In [48]:
# ============================================================
# PORTION 4B.11 — VISUALIZATION
# ============================================================

def plot_confusion(cm: np.ndarray, labels: List[str], title: str, out_path: Path, normalize: bool = False):
    cm_plot = cm.astype(np.float64)
    if normalize:
        cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1.0)

    plt.figure(figsize=(11, 9))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title)
    plt.colorbar()

    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=45)
    plt.yticks(ticks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("Gold")

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            v = cm_plot[i, j]
            txt = f"{v:.2f}" if normalize else str(int(v))
            if v > (0.01 if normalize else 0):
                plt.text(j, i, txt, ha="center", va="center", fontsize=8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=260)
    plt.close()


# Selected internal validation metrics.
selected_val_metrics = evaluate_probs(y_val, selected_val_probs, "val", restricted_ids=None)
selected_ext_metrics = evaluate_probs(y_ext, selected_ext_probs, "external", restricted_ids=external_present_ids)
selected_ext_focus_metrics = evaluate_probs(y_ext, selected_ext_focus_probs, "external_focus", restricted_ids=external_present_ids)

write_json(REPORT_DIR_4B / "p4b_selected_val_metrics.json", metric_dump_to_json(selected_val_metrics))
write_json(REPORT_DIR_4B / "p4b_selected_external_metrics.json", metric_dump_to_json(selected_ext_metrics))
write_json(REPORT_DIR_4B / "p4b_selected_external_focus_metrics.json", metric_dump_to_json(selected_ext_focus_metrics))

# Confusion matrices.
plot_confusion(
    selected_val_metrics["val_13way"]["confusion_matrix"],
    DIALECTS,
    "P4B Selected Lexical Classifier — Validation 13-way Confusion Matrix",
    PLOT_DIR_4B / "p4b_val_13way_confusion_counts.png",
    normalize=False,
)

plot_confusion(
    selected_val_metrics["val_13way"]["confusion_matrix"],
    DIALECTS,
    "P4B Selected Lexical Classifier — Validation 13-way Normalized Confusion Matrix",
    PLOT_DIR_4B / "p4b_val_13way_confusion_normalized.png",
    normalize=True,
)

plot_confusion(
    selected_ext_metrics["external_restricted"]["confusion_matrix"],
    external_present_dialects,
    "P4B Selected Lexical Classifier — External Present Confusion Matrix",
    PLOT_DIR_4B / "p4b_external_present_confusion_normalized.png",
    normalize=True,
)

plot_confusion(
    selected_ext_focus_metrics["external_focus_restricted"]["confusion_matrix"],
    external_present_dialects,
    "P4B External-Focused Classifier — External Present Confusion Matrix",
    PLOT_DIR_4B / "p4b_external_focus_present_confusion_normalized.png",
    normalize=True,
)

# Candidate comparison plots.
plot_df = final_selection_df.copy()
plot_df_val = plot_df.dropna(subset=["val_13_macro_f1"]).sort_values("val_13_macro_f1", ascending=False).head(15)

plt.figure(figsize=(14, 7))
plt.bar(plot_df_val["model"], plot_df_val["val_13_macro_f1"])
plt.axhline(CFG4B.current_best_val_macro_f1, linestyle="--", label="previous best 0.7100")
plt.axhline(CFG4B.target_val_macro_f1_min, linestyle="--", label="target 0.72")
plt.xticks(rotation=40, ha="right")
plt.ylabel("Validation 13-way Macro-F1")
plt.title("P4B Lexical Candidate Comparison — Validation Macro-F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR_4B / "p4b_candidate_val_macro_f1.png", dpi=260)
plt.close()

plot_df_ext = plot_df.dropna(subset=["external_present_macro_f1"]).sort_values("external_present_macro_f1", ascending=False).head(15)

plt.figure(figsize=(14, 7))
plt.bar(plot_df_ext["model"], plot_df_ext["external_present_macro_f1"])
plt.axhline(CFG4B.current_best_external_macro_f1, linestyle="--", label="previous best 0.1615")
plt.xticks(rotation=40, ha="right")
plt.ylabel("External Present Macro-F1")
plt.title("P4B Lexical Candidate Comparison — External Present Macro-F1")
plt.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR_4B / "p4b_candidate_external_present_macro_f1.png", dpi=260)
plt.close()

# Top n-gram plot for top few dialects.
if len(top_ngram_df):
    for dialect in DIALECTS:
        sub = top_ngram_df[top_ngram_df["dialect"] == dialect].head(20).iloc[::-1]
        if len(sub) == 0:
            continue

        plt.figure(figsize=(9, 7))
        plt.barh(sub["ngram"], sub["weight"])
        plt.xlabel("Classifier weight")
        plt.title(f"Top Character N-grams for {dialect}")
        plt.tight_layout()
        plt.savefig(PLOT_DIR_4B / f"p4b_top_ngrams_{dialect}.png", dpi=260)
        plt.close()

print("[OK] P4B plots saved to:", PLOT_DIR_4B)

[OK] P4B plots saved to: /kaggle/working/artifacts/bleach_p4b_advanced_lexical/plots


In [49]:
# ============================================================
# PORTION 4B.12 — HIGH-CONFIDENCE ERROR AUDIT
# ============================================================

val_errors = val_pred_df_4b[val_pred_df_4b["correct_13"] == 0].copy()
val_errors = val_errors.sort_values("confidence_13", ascending=False)
val_errors.head(CFG4B.high_conf_error_k).to_csv(PRED_DIR_4B / "p4b_val_high_confidence_errors.csv", index=False)

external_errors = ext_pred_df_4b[ext_pred_df_4b["correct_restricted"] == 0].copy()
external_errors = external_errors.sort_values("confidence_restricted", ascending=False)
external_errors.head(CFG4B.high_conf_error_k).to_csv(PRED_DIR_4B / "p4b_external_present_high_confidence_errors.csv", index=False)

external_focus_errors = ext_focus_pred_df_4b[ext_focus_pred_df_4b["correct_restricted"] == 0].copy()
external_focus_errors = external_focus_errors.sort_values("confidence_restricted", ascending=False)
external_focus_errors.head(CFG4B.high_conf_error_k).to_csv(
    PRED_DIR_4B / "p4b_external_focus_high_confidence_errors.csv",
    index=False,
)

val_error_pairs = (
    val_errors.groupby(["gold_dialect", "pred_13_dialect"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
val_error_pairs.to_csv(REPORT_DIR_4B / "p4b_val_error_pairs.csv", index=False)

ext_error_pairs = (
    external_errors.groupby(["gold_dialect", "pred_restricted_dialect"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
ext_error_pairs.to_csv(REPORT_DIR_4B / "p4b_external_present_error_pairs.csv", index=False)

ext_focus_error_pairs = (
    external_focus_errors.groupby(["gold_dialect", "pred_restricted_dialect"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)
ext_focus_error_pairs.to_csv(REPORT_DIR_4B / "p4b_external_focus_error_pairs.csv", index=False)

error_audit = {
    "val_high_confidence_errors": str(PRED_DIR_4B / "p4b_val_high_confidence_errors.csv"),
    "external_present_high_confidence_errors": str(PRED_DIR_4B / "p4b_external_present_high_confidence_errors.csv"),
    "external_focus_high_confidence_errors": str(PRED_DIR_4B / "p4b_external_focus_high_confidence_errors.csv"),
    "val_error_pairs": str(REPORT_DIR_4B / "p4b_val_error_pairs.csv"),
    "external_present_error_pairs": str(REPORT_DIR_4B / "p4b_external_present_error_pairs.csv"),
    "external_focus_error_pairs": str(REPORT_DIR_4B / "p4b_external_focus_error_pairs.csv"),
    "top_val_error_pairs": val_error_pairs.head(20).to_dict(orient="records"),
    "top_external_present_error_pairs": ext_error_pairs.head(20).to_dict(orient="records"),
    "top_external_focus_error_pairs": ext_focus_error_pairs.head(20).to_dict(orient="records"),
}

write_json(REPORT_DIR_4B / "p4b_error_audit_summary.json", error_audit)

print(json.dumps(error_audit, indent=2, ensure_ascii=False))

{
  "val_high_confidence_errors": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_val_high_confidence_errors.csv",
  "external_present_high_confidence_errors": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_external_present_high_confidence_errors.csv",
  "external_focus_high_confidence_errors": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/predictions/p4b_external_focus_high_confidence_errors.csv",
  "val_error_pairs": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/reports/p4b_val_error_pairs.csv",
  "external_present_error_pairs": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/reports/p4b_external_present_error_pairs.csv",
  "external_focus_error_pairs": "/kaggle/working/artifacts/bleach_p4b_advanced_lexical/reports/p4b_external_focus_error_pairs.csv",
  "top_val_error_pairs": [
    {
      "gold_dialect": "STD",
      "pred_13_dialect": "NOA",
      "count": 137
    },
    {
      "gold_dialect": "NOA",
      "pr

In [50]:
# ============================================================
# P4B-FIX — CLEAN TOP N-GRAM INTERPRETABILITY TABLE
# Removes duplicate, punctuation-only, and too-short artifacts.
# ============================================================

import re
import pandas as pd
from pathlib import Path

TOP_NGRAM_PATH = REPORT_DIR_4B / "p4b_top_ngrams_by_dialect.csv"
clean_ngram_df = pd.read_csv(TOP_NGRAM_PATH)

BANGLA_RE = re.compile(r"[\u0980-\u09FF]")
PUNCT_ONLY_RE = re.compile(r"^[\s\W_।!?.,;:()\[\]{}'\"`~\-–—]+$")

def keep_ngram(x):
    s = str(x)
    if len(s.strip()) < 2:
        return False
    if PUNCT_ONLY_RE.match(s):
        return False
    if not BANGLA_RE.search(s):
        return False
    return True

clean_ngram_df = clean_ngram_df.copy()
clean_ngram_df["ngram"] = clean_ngram_df["ngram"].astype(str)

clean_ngram_df = clean_ngram_df[clean_ngram_df["ngram"].map(keep_ngram)].copy()

# Remove duplicates per dialect.
clean_ngram_df = clean_ngram_df.sort_values(["dialect", "weight"], ascending=[True, False])
clean_ngram_df = clean_ngram_df.drop_duplicates(subset=["dialect", "ngram"], keep="first")

# Re-rank after cleanup.
clean_ngram_df["rank_clean"] = clean_ngram_df.groupby("dialect")["weight"].rank(
    method="first",
    ascending=False
).astype(int)

clean_ngram_df = clean_ngram_df[clean_ngram_df["rank_clean"] <= 30].copy()
clean_ngram_df = clean_ngram_df.sort_values(["dialect", "rank_clean"])

clean_path = REPORT_DIR_4B / "p4b_top_ngrams_by_dialect_cleaned.csv"
clean_ngram_df.to_csv(clean_path, index=False)

print("Saved cleaned interpretability table:", clean_path)
print(clean_ngram_df.groupby("dialect").head(10).to_string(index=False))

Saved cleaned interpretability table: /kaggle/working/artifacts/bleach_p4b_advanced_lexical/reports/p4b_top_ngrams_by_dialect_cleaned.csv
                                    model dialect  rank  ngram   weight  rank_clean
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     1    ইরর 1.882314           1
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     2    লও  1.881800           2
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     3     হর 1.859452           3
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     4  মানু  1.797592           4
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     5  মানু  1.797592           5
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     6   কবা  1.757676           6
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     7     লও 1.730652           7
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     8    মু  1.717358           8
svm_15_char_wb_26_mf300k_df2_subTrue_C1.0     BAR     9  আম্মে 1.662235           9
svm_15_char_wb_26_mf30

In [51]:
# ============================================================
# PORTION 4B.13 — FINAL PAPER-READY REPORT
# ============================================================

selected_val_metrics_summary = {
    "val_13_acc": float(selected_val_metrics["val_13way"]["accuracy"]),
    "val_13_macro_f1": float(selected_val_metrics["val_13way"]["macro_f1"]),
    "val_13_weighted_f1": float(selected_val_metrics["val_13way"]["weighted_f1"]),
    "val_13_mcc": float(selected_val_metrics["val_13way"]["mcc"]),
}

selected_external_metrics_summary = {
    "external_13_acc": float(selected_ext_metrics["external_13way"]["accuracy"]),
    "external_13_macro_f1": float(selected_ext_metrics["external_13way"]["macro_f1"]),
    "external_present_acc": float(selected_ext_metrics["external_restricted"]["accuracy"]),
    "external_present_macro_f1": float(selected_ext_metrics["external_restricted"]["macro_f1"]),
    "external_present_weighted_f1": float(selected_ext_metrics["external_restricted"]["weighted_f1"]),
    "external_present_mcc": float(selected_ext_metrics["external_restricted"]["mcc"]),
    "external_present_ece": float(selected_ext_metrics["external_restricted"]["ece"]),
}

selected_external_focus_metrics_summary = {
    "external_focus_present_acc": float(selected_ext_focus_metrics["external_focus_restricted"]["accuracy"]),
    "external_focus_present_macro_f1": float(selected_ext_focus_metrics["external_focus_restricted"]["macro_f1"]),
    "external_focus_present_weighted_f1": float(selected_ext_focus_metrics["external_focus_restricted"]["weighted_f1"]),
    "external_focus_present_mcc": float(selected_ext_focus_metrics["external_focus_restricted"]["mcc"]),
    "external_focus_present_ece": float(selected_ext_focus_metrics["external_focus_restricted"]["ece"]),
}

acceptance = {
    "improved_over_previous_val_macro_f1_0_7100": bool(
        selected_val_metrics_summary["val_13_macro_f1"] > CFG4B.current_best_val_macro_f1
    ),
    "val_macro_f1_reaches_0_72": bool(
        selected_val_metrics_summary["val_13_macro_f1"] >= CFG4B.target_val_macro_f1_min
    ),
    "val_macro_f1_reaches_0_74": bool(
        selected_val_metrics_summary["val_13_macro_f1"] >= CFG4B.target_val_macro_f1_strong
    ),
    "external_focus_improved_over_previous_0_1615": bool(
        selected_external_focus_metrics_summary["external_focus_present_macro_f1"] > CFG4B.current_best_external_macro_f1
    ),
}

paper_claim = {
    "main_classification_result": (
        "Advanced lexical classifiers remain the strongest downstream dialect-identification systems. "
        "This supports the interpretation that the benchmark is dominated by orthographic and subword-level dialect cues."
    ),
    "external_result_interpretation": (
        "If external-present scores remain near 0.16-0.20 macro-F1, the external benchmark should be framed as severe "
        "source/domain shift rather than ordinary in-distribution dialect classification."
    ),
    "recommended_paper_usage": (
        "Report P4B as the final classification module. Report BLEACH decoder fine-tuning only as a diagnostic baseline, "
        "not as the selected classifier."
    ),
}

final_report = {
    "portion": "P4B",
    "task": "advanced_lexical_dialect_classification",
    "status": "complete",
    "external_present_dialects": external_present_dialects,
    "selected_internal_validation_model": final_decision["selected_internal_validation_model"],
    "selected_external_present_model": final_decision["selected_external_present_model"],
    "selected_internal_validation_metrics": selected_val_metrics_summary,
    "selected_external_metrics": selected_external_metrics_summary,
    "selected_external_focus_metrics": selected_external_focus_metrics_summary,
    "acceptance": acceptance,
    "paper_claim": paper_claim,
    "outputs": {
        "candidate_results": str(REPORT_DIR_4B / "p4b_candidate_results.csv"),
        "present5_results": str(REPORT_DIR_4B / "p4b_present5_results.csv"),
        "ensemble_results": str(REPORT_DIR_4B / "p4b_ensemble_search_results.csv"),
        "final_selection_table": str(REPORT_DIR_4B / "p4b_final_model_selection_table.csv"),
        "top_ngrams": str(REPORT_DIR_4B / "p4b_top_ngrams_by_dialect.csv"),
        "error_audit": str(REPORT_DIR_4B / "p4b_error_audit_summary.json"),
        "selected_val_predictions": str(PRED_DIR_4B / "p4b_selected_val_predictions.csv"),
        "selected_external_predictions": str(PRED_DIR_4B / "p4b_selected_external_predictions.csv"),
        "selected_external_focus_predictions": str(PRED_DIR_4B / "p4b_selected_external_focus_predictions.csv"),
    },
    "plots": {
        "val_confusion": str(PLOT_DIR_4B / "p4b_val_13way_confusion_normalized.png"),
        "external_present_confusion": str(PLOT_DIR_4B / "p4b_external_present_confusion_normalized.png"),
        "external_focus_confusion": str(PLOT_DIR_4B / "p4b_external_focus_present_confusion_normalized.png"),
        "candidate_val_macro_f1": str(PLOT_DIR_4B / "p4b_candidate_val_macro_f1.png"),
        "candidate_external_present_macro_f1": str(PLOT_DIR_4B / "p4b_candidate_external_present_macro_f1.png"),
    },
}

write_json(REPORT_DIR_4B / "p4b_final_report.json", final_report)

print("=" * 100)
print("P4B FINAL CHECKPOINT")
print("=" * 100)
print("Selected internal model:", final_report["selected_internal_validation_model"])
print("Val 13-way accuracy    :", f"{selected_val_metrics_summary['val_13_acc']:.4f}")
print("Val 13-way macro-F1    :", f"{selected_val_metrics_summary['val_13_macro_f1']:.4f}")
print("Selected external model:", final_report["selected_external_present_model"])
print("Ext present accuracy   :", f"{selected_external_focus_metrics_summary['external_focus_present_acc']:.4f}")
print("Ext present macro-F1   :", f"{selected_external_focus_metrics_summary['external_focus_present_macro_f1']:.4f}")
print("Acceptance:")
print(json.dumps(acceptance, indent=2))
print("Final report:", REPORT_DIR_4B / "p4b_final_report.json")
print("=" * 100)

print("\nPaste these after running P4B:")
print("1. reports/p4b_final_report.json")
print("2. reports/p4b_candidate_results.csv")
print("3. reports/p4b_present5_results.csv")
print("4. reports/p4b_final_model_selection_table.csv")
print("5. reports/p4b_top_ngrams_by_dialect.csv")
print("6. reports/p4b_val_error_pairs.csv")
print("7. reports/p4b_external_present_error_pairs.csv")

P4B FINAL CHECKPOINT
Selected internal model: P4B_lexical_weighted_ensemble
Val 13-way accuracy    : 0.7194
Val 13-way macro-F1    : 0.7239
Selected external model: word_logreg_00_12_mf150k_df2_C2.0
Ext present accuracy   : 0.2097
Ext present macro-F1   : 0.1517
Acceptance:
{
  "improved_over_previous_val_macro_f1_0_7100": true,
  "val_macro_f1_reaches_0_72": true,
  "val_macro_f1_reaches_0_74": false,
  "external_focus_improved_over_previous_0_1615": false
}
Final report: /kaggle/working/artifacts/bleach_p4b_advanced_lexical/reports/p4b_final_report.json

Paste these after running P4B:
1. reports/p4b_final_report.json
2. reports/p4b_candidate_results.csv
3. reports/p4b_present5_results.csv
4. reports/p4b_final_model_selection_table.csv
5. reports/p4b_top_ngrams_by_dialect.csv
6. reports/p4b_val_error_pairs.csv
7. reports/p4b_external_present_error_pairs.csv
